In [ ]:
!fusermount -u /content/drive 2>/dev/null || true
!rm -rf /content/drive
!mkdir -p /content/drive
!ls -lah /content

In [ ]:
!ls /content/drive/MyDrive/dsresearch/

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!rm -rf /content/deepfashion_data
!mkdir -p /content/deepfashion_data
!unzip -q "/content/drive/MyDrive/dsresearch/img_highres.zip" -d "/content/deepfashion_data/"
!find "/content/deepfashion_data/img_highres" -iname "*.jpg" | wc -l

In [ ]:
from pathlib import Path
import pandas as pd

PORTABLE_CSV = "/content/drive/MyDrive/dsresearch/outputs/deepfashion_sleeve_portable.csv"
LOCAL_OUT = "/content/deepfashion_sleeve_with_style.csv"

IMAGE_ROOT = Path("/content/deepfashion_data/img_highres")

df_portable = pd.read_csv(PORTABLE_CSV)

def resolve_from_portable(rel_path):
    rel_path = str(rel_path).strip()
    name = Path(rel_path).name

    if "-img_" not in name:
        return None

    folder_name, img_tail = name.split("-img_", 1)
    img_name = "img_" + img_tail

    candidate = IMAGE_ROOT / folder_name / img_name
    if candidate.exists():
        return str(candidate)
    return None

df_portable["image_path"] = df_portable["original_image_path"].apply(resolve_from_portable)

print("rows before resolve:", len(df_portable))
print("resolved:", df_portable["image_path"].notna().sum())
print("failed:", df_portable["image_path"].isna().sum())

df_ready = df_portable[df_portable["image_path"].notna()].copy()
df_ready = df_ready[["image_path", "label", "style_split"]].reset_index(drop=True)

df_ready.to_csv(LOCAL_OUT, index=False)

print("saved:", LOCAL_OUT)
print("rows:", len(df_ready))
print(df_ready.head())
print(df_ready["label"].value_counts())
print(df_ready["style_split"].value_counts())

In [ ]:
import torch
from transformers import CLIPModel, CLIPProcessor
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from tqdm import tqdm
from PIL import Image
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

CSV_PATH = "/content/deepfashion_sleeve_with_style.csv"
OUTPUT_DIR = "/content/drive/MyDrive/dsresearch/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

df_test = pd.read_csv(CSV_PATH)
print("Total rows in CSV:", len(df_test))
print(df_test["label"].value_counts())
print(df_test["style_split"].value_counts())

df_test = df_test.sample(n=min(500, len(df_test)), random_state=42).reset_index(drop=True)
print("Rows used in experiment:", len(df_test))

class_names = ["long sleeve", "short sleeve", "sleeveless"]
prompts = [
    "a photo of a long sleeve top",
    "a photo of a short sleeve top",
    "a photo of a sleeveless top",
]

model_id = "openai/clip-vit-base-patch32"
model = CLIPModel.from_pretrained(model_id).to(device)
processor = CLIPProcessor.from_pretrained(model_id)
print("CLIP完成")

text_inputs = processor(
    text=prompts,
    return_tensors="pt",
    padding=True,
    truncation=True
)
text_inputs = {k: v.to(device) for k, v in text_inputs.items()}

with torch.no_grad():
    text_outputs = model.text_model(
        input_ids=text_inputs["input_ids"],
        attention_mask=text_inputs["attention_mask"]
    )
    text_features = text_outputs.pooler_output
    text_features = model.text_projection(text_features)
    text_features = text_features / text_features.norm(p=2, dim=-1, keepdim=True)

y_true, y_pred, rows = [], [], []

for _, row in tqdm(df_test.iterrows(), total=len(df_test)):
    try:
        image = Image.open(row["image_path"]).convert("RGB")
        image_inputs = processor(images=image, return_tensors="pt")
        image_inputs = {k: v.to(device) for k, v in image_inputs.items()}

        with torch.no_grad():
            image_outputs = model.vision_model(
                pixel_values=image_inputs["pixel_values"]
            )
            image_features = image_outputs.pooler_output
            image_features = model.visual_projection(image_features)
            image_features = image_features / image_features.norm(p=2, dim=-1, keepdim=True)

            logits = image_features @ text_features.T
            pred_idx = int(logits.argmax(dim=1).cpu().item())

        pred = class_names[pred_idx]
        gt = row["label"]

        y_true.append(gt)
        y_pred.append(pred)

        rows.append({
            "image_path": row["image_path"],
            "style_split": row["style_split"],
            "ground_truth": gt,
            "prediction": pred,
            "correct": int(gt == pred),
        })

    except Exception as e:
        print("Skipping due to error:", row["image_path"], e)

pred_df = pd.DataFrame(rows)

LOCAL_PRED_PATH = "/content/deepfashion_sleeve_clip_predictions_exp1.csv"
DRIVE_PRED_PATH = os.path.join(OUTPUT_DIR, "deepfashion_sleeve_clip_predictions_exp1.csv")

pred_df.to_csv(LOCAL_PRED_PATH, index=False)
pred_df.to_csv(DRIVE_PRED_PATH, index=False)

if len(pred_df) == 0:
    print("No valid predictions were generated.")
else:
    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    cm = confusion_matrix(y_true, y_pred, labels=class_names)

    print("\Overall Results ")
    print("Accuracy:", acc)
    print("Macro F1:", macro_f1)
    print("Confusion Matrix:\n", cm)
    print(classification_report(y_true, y_pred, labels=class_names, target_names=class_names))

    print("By Style Split ")
    for split_name in ["catalog_like", "less_curated"]:
        sub = pred_df[pred_df["style_split"] == split_name]
        if len(sub) == 0:
            continue

        sub_acc = accuracy_score(sub["ground_truth"], sub["prediction"])
        sub_macro_f1 = f1_score(sub["ground_truth"], sub["prediction"], average="macro")

        print(f"\n{split_name}")
        print("n =", len(sub))
        print("accuracy =", sub_acc)
        print("macro_f1 =", sub_macro_f1)

    print("Saved local predictions:", LOCAL_PRED_PATH)
    print("Saved drive predictions:", DRIVE_PRED_PATH)

    plt.figure(figsize=(6, 5))
    plt.imshow(cm, interpolation="nearest")
    plt.title("CLIP Confusion Matrix on Sleeve Type")
    plt.colorbar()

    tick_marks = np.arange(len(class_names))
    plt.xticks(tick_marks, class_names, rotation=45)
    plt.yticks(tick_marks, class_names)

    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    plt.show()

In [ ]:
import torch
from transformers import CLIPModel, CLIPProcessor
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from tqdm import tqdm
from PIL import Image
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
import textwrap


device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

CSV_PATH = "/content/deepfashion_sleeve_with_style.csv"
OUTPUT_DIR = "/content/drive/MyDrive/dsresearch/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)


LOCAL_PRED_PATH = "/content/deepfashion_sleeve_clip_predictions_exp1_full.csv"
DRIVE_PRED_PATH = os.path.join(OUTPUT_DIR, "deepfashion_sleeve_clip_predictions_exp1_full.csv")
CM_FIG_PATH = os.path.join(OUTPUT_DIR, "deepfashion_sleeve_clip_confusion_matrix_exp1_full.png")
SUMMARY_TXT_PATH = os.path.join(OUTPUT_DIR, "deepfashion_sleeve_clip_exp1_full_summary.txt")


df_test = pd.read_csv(CSV_PATH)

print("Total rows in CSV:", len(df_test))
print("\nLabel distribution:")
print(df_test["label"].value_counts())
print("\nStyle split distribution:")
print(df_test["style_split"].value_counts())

df_test = df_test.reset_index(drop=True)
print("\nRows used in experiment:", len(df_test))

class_names = ["long sleeve", "short sleeve", "sleeveless"]
prompts = [
    "a photo of a long sleeve top",
    "a photo of a short sleeve top",
    "a photo of a sleeveless top",
]


print("\nLoading CLIP model...")
model_id = "openai/clip-vit-base-patch32"
model = CLIPModel.from_pretrained(model_id).to(device)
processor = CLIPProcessor.from_pretrained(model_id)
model.eval()
print("CLIP model loaded.")


text_inputs = processor(
    text=prompts,
    return_tensors="pt",
    padding=True,
    truncation=True
)
text_inputs = {k: v.to(device) for k, v in text_inputs.items()}

with torch.no_grad():
    text_outputs = model.text_model(
        input_ids=text_inputs["input_ids"],
        attention_mask=text_inputs["attention_mask"]
    )
    text_features = text_outputs.pooler_output
    text_features = model.text_projection(text_features)
    text_features = text_features / text_features.norm(p=2, dim=-1, keepdim=True)


y_true, y_pred, rows = [], [], []

for _, row in tqdm(df_test.iterrows(), total=len(df_test)):
    try:
        image = Image.open(row["image_path"]).convert("RGB")

        image_inputs = processor(images=image, return_tensors="pt")
        image_inputs = {k: v.to(device) for k, v in image_inputs.items()}

        with torch.no_grad():
            image_outputs = model.vision_model(
                pixel_values=image_inputs["pixel_values"]
            )
            image_features = image_outputs.pooler_output
            image_features = model.visual_projection(image_features)
            image_features = image_features / image_features.norm(p=2, dim=-1, keepdim=True)

            logits = image_features @ text_features.T
            pred_idx = int(logits.argmax(dim=1).cpu().item())

        pred = class_names[pred_idx]
        gt = row["label"]

        y_true.append(gt)
        y_pred.append(pred)

        rows.append({
            "image_path": row["image_path"],
            "style_split": row["style_split"],
            "ground_truth": gt,
            "prediction": pred,
            "correct": int(gt == pred),
        })

    except Exception as e:
        print("Skipping due to error:", row["image_path"], e)

pred_df = pd.DataFrame(rows)
pred_df.to_csv(LOCAL_PRED_PATH, index=False)
pred_df.to_csv(DRIVE_PRED_PATH, index=False)


if len(pred_df) == 0:
    print("No valid predictions were generated.")
else:
    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    cm = confusion_matrix(y_true, y_pred, labels=class_names)
    clf_report = classification_report(
        y_true, y_pred,
        labels=class_names,
        target_names=class_names
    )

    print("\n===== Overall Results =====")
    print("Accuracy:", acc)
    print("Macro F1:", macro_f1)
    print("Confusion Matrix:\n", cm)
    print(clf_report)


    style_lines = []
    print("\n===== By Style Split =====")
    for split_name in ["catalog_like", "less_curated"]:
        sub = pred_df[pred_df["style_split"] == split_name]
        if len(sub) == 0:
            continue

        sub_acc = accuracy_score(sub["ground_truth"], sub["prediction"])
        sub_macro_f1 = f1_score(sub["ground_truth"], sub["prediction"], average="macro")

        print(f"\n{split_name}")
        print("n =", len(sub))
        print("accuracy =", sub_acc)
        print("macro_f1 =", sub_macro_f1)

        style_lines.append(
            f"{split_name}\n"
            f"n = {len(sub)}\n"
            f"accuracy = {sub_acc}\n"
            f"macro_f1 = {sub_macro_f1}\n"
        )


    plt.figure(figsize=(6, 5))
    plt.imshow(cm, interpolation="nearest")
    plt.title("CLIP Confusion Matrix on Sleeve Type (Full)")
    plt.colorbar()

    tick_marks = np.arange(len(class_names))
    plt.xticks(tick_marks, class_names, rotation=45)
    plt.yticks(tick_marks, class_names)

    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    plt.savefig(CM_FIG_PATH, dpi=200, bbox_inches="tight")
    plt.show()


    summary_text = f"""
Experiment 1: Zero-shot CLIP baseline on full sleeve-type subset

Model:
{model_id}

Prompts:
- {prompts[0]}
- {prompts[1]}
- {prompts[2]}

Dataset size:
{len(df_test)}

Label distribution:
{df_test["label"].value_counts().to_string()}

Style split distribution:
{df_test["style_split"].value_counts().to_string()}

Overall Results:
Accuracy: {acc}
Macro F1: {macro_f1}

Confusion Matrix:
{cm}

Classification Report:
{clf_report}

By Style Split:
{''.join(style_lines)}

Saved prediction CSV:
{DRIVE_PRED_PATH}

Saved confusion matrix figure:
{CM_FIG_PATH}
""".strip()

    with open(SUMMARY_TXT_PATH, "w", encoding="utf-8") as f:
        f.write(summary_text)

    print("\nSaved local predictions:", LOCAL_PRED_PATH)
    print("Saved drive predictions:", DRIVE_PRED_PATH)
    print("Saved confusion matrix figure:", CM_FIG_PATH)
    print("Saved summary text:", SUMMARY_TXT_PATH)

experiment2

In [ ]:
import torch
from transformers import CLIPModel, CLIPProcessor
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from tqdm import tqdm
from PIL import Image, ImageFile
import pandas as pd
import os

ImageFile.LOAD_TRUNCATED_IMAGES = True
Image.MAX_IMAGE_PIXELS = None

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

CSV_PATH = "/content/deepfashion_sleeve_with_style.csv"
OUTPUT_DIR = "/content/drive/MyDrive/dsresearch/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

df_test = pd.read_csv(CSV_PATH).reset_index(drop=True)
print("Total rows in CSV:", len(df_test))
print(df_test["label"].value_counts())
print(df_test["style_split"].value_counts())

class_names = ["long sleeve", "short sleeve", "sleeveless"]

prompt_sets = {
    "A_basic": [
        "a photo of a long sleeve top",
        "a photo of a short sleeve top",
        "a photo of a sleeveless top",
    ],
    "B_fashion_product": [
        "a fashion product photo of a long sleeve top",
        "a fashion product photo of a short sleeve top",
        "a fashion product photo of a sleeveless top",
    ],
    "C_garment_description": [
        "a clothing item with long sleeves",
        "a clothing item with short sleeves",
        "a sleeveless clothing item",
    ],
}

print("Loading CLIP model...")
model_id = "openai/clip-vit-base-patch32"
model = CLIPModel.from_pretrained(model_id).to(device)
processor = CLIPProcessor.from_pretrained(model_id)
model.eval()
print("CLIP model loaded.")

summary_rows = []

for prompt_name, prompts in prompt_sets.items():
    print("\n" + "=" * 80)
    print("Running prompt set:", prompt_name)
    print("Prompts:")
    for p in prompts:
        print("-", p)

    text_inputs = processor(
        text=prompts,
        return_tensors="pt",
        padding=True,
        truncation=True
    )
    text_inputs = {k: v.to(device) for k, v in text_inputs.items()}

    with torch.no_grad():
        text_outputs = model.text_model(
            input_ids=text_inputs["input_ids"],
            attention_mask=text_inputs["attention_mask"]
        )
        text_features = text_outputs.pooler_output
        text_features = model.text_projection(text_features)
        text_features = text_features / text_features.norm(p=2, dim=-1, keepdim=True)

    y_true, y_pred, rows = [], [], []

    for _, row in tqdm(df_test.iterrows(), total=len(df_test), desc=prompt_name):
        try:
            image = Image.open(row["image_path"]).convert("RGB")
            image_inputs = processor(images=image, return_tensors="pt")
            image_inputs = {k: v.to(device) for k, v in image_inputs.items()}

            with torch.no_grad():
                image_outputs = model.vision_model(
                    pixel_values=image_inputs["pixel_values"]
                )
                image_features = image_outputs.pooler_output
                image_features = model.visual_projection(image_features)
                image_features = image_features / image_features.norm(p=2, dim=-1, keepdim=True)

                logits = image_features @ text_features.T
                pred_idx = int(logits.argmax(dim=1).cpu().item())

            pred = class_names[pred_idx]
            gt = row["label"]

            y_true.append(gt)
            y_pred.append(pred)

            rows.append({
                "image_path": row["image_path"],
                "style_split": row["style_split"],
                "ground_truth": gt,
                "prediction": pred,
                "correct": int(gt == pred),
                "prompt_set": prompt_name,
            })

        except Exception as e:
            print("Skipping due to error:", row["image_path"], e)

    pred_df = pd.DataFrame(rows)

    pred_path = os.path.join(OUTPUT_DIR, f"deepfashion_clip_{prompt_name}_predictions.csv")
    pred_df.to_csv(pred_path, index=False)

    if len(pred_df) == 0:
        print("No valid predictions for", prompt_name)
        continue

    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    cm = confusion_matrix(y_true, y_pred, labels=class_names)
    clf_report = classification_report(
        y_true, y_pred,
        labels=class_names,
        target_names=class_names
    )

    print("\n===== Overall Results:", prompt_name, "=====")
    print("Accuracy:", acc)
    print("Macro F1:", macro_f1)
    print("Confusion Matrix:\n", cm)
    print(clf_report)

    summary_rows.append({
        "prompt_set": prompt_name,
        "subset": "overall",
        "n": len(pred_df),
        "accuracy": acc,
        "macro_f1": macro_f1,
    })

    for split_name in ["catalog_like", "less_curated"]:
        sub = pred_df[pred_df["style_split"] == split_name]
        if len(sub) == 0:
            continue

        sub_acc = accuracy_score(sub["ground_truth"], sub["prediction"])
        sub_macro_f1 = f1_score(sub["ground_truth"], sub["prediction"], average="macro")

        print(f"\n{prompt_name} | {split_name}")
        print("n =", len(sub))
        print("accuracy =", sub_acc)
        print("macro_f1 =", sub_macro_f1)

        summary_rows.append({
            "prompt_set": prompt_name,
            "subset": split_name,
            "n": len(sub),
            "accuracy": sub_acc,
            "macro_f1": sub_macro_f1,
        })

summary_df = pd.DataFrame(summary_rows)
summary_csv = os.path.join(OUTPUT_DIR, "deepfashion_clip_prompt_comparison_summary.csv")
summary_df.to_csv(summary_csv, index=False)

print("\n" + "=" * 80)
print("Prompt comparison finished.")
print(summary_df)
print("Saved summary to:", summary_csv)

experiment 3

In [ ]:
import torch
import numpy as np
from transformers import BlipProcessor, BlipForImageTextRetrieval
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "cpu"


processor = BlipProcessor.from_pretrained("Salesforce/blip-itm-base-coco")
model = BlipForImageTextRetrieval.from_pretrained(
    "Salesforce/blip-itm-base-coco"
).to(device)
model.eval()


def blip_predict(image, prompts):
    inputs = processor(
        images=image,
        text=prompts,
        return_tensors="pt",
        padding=True
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)

        scores = outputs.itm_score[:, 1]

    return scores.cpu().numpy()

In [ ]:
def blip_predict(image, prompts):
    images = [image] * len(prompts)  

    inputs = processor(
        images=images,
        text=prompts,
        return_tensors="pt",
        padding=True
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)

        scores = outputs.itm_score[:, 1] 

    return scores.cpu().numpy()

In [ ]:
import pandas as pd

df = pd.read_csv("/content/deepfashion_sleeve_with_style.csv")

print("Loaded:", len(df))

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from tqdm import tqdm
from PIL import Image
import numpy as np
import os

all_preds = []
all_labels = []
all_styles = []

prompts = [
    "a fashion product photo of a long sleeve top",
    "a fashion product photo of a short sleeve top",
    "a fashion product photo of a sleeveless top"
]

label_map = {
    "long sleeve": 0,
    "short sleeve": 1,
    "sleeveless": 2
}

for idx, row in tqdm(df.iterrows(), total=len(df)):
    image = Image.open(row["image_path"]).convert("RGB")

    scores = blip_predict(image, prompts)
    pred = int(np.argmax(scores))

    label = label_map[row["label"]]
    all_preds.append(pred)
    all_labels.append(label)
    all_styles.append(row["style_split"])

In [ ]:
acc = accuracy_score(all_labels, all_preds)
macro_f1 = f1_score(all_labels, all_preds, average="macro")

print("\n===== BLIP Results =====")
print("Accuracy:", acc)
print("Macro F1:", macro_f1)

In [ ]:
import pandas as pd

results = pd.DataFrame({
    "label": all_labels,
    "pred": all_preds,
    "style": all_styles
})

for style in ["catalog_like", "less_curated"]:
    subset = results[results["style"] == style]

    acc = accuracy_score(subset["label"], subset["pred"])
    f1 = f1_score(subset["label"], subset["pred"], average="macro")

    print(f"\nBLIP | {style}")
    print("n =", len(subset))
    print("accuracy =", acc)
    print("macro_f1 =", f1)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report,
    precision_recall_fscore_support
)

OUTPUT_DIR = "/content/drive/MyDrive/dsresearch/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

MODEL_NAME = "BLIP"
class_names = ["long sleeve", "short sleeve", "sleeveless"]

acc = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro")
cm = confusion_matrix(y_true, y_pred, labels=class_names)
clf_report_dict = classification_report(
    y_true, y_pred,
    labels=class_names,
    target_names=class_names,
    output_dict=True
)
clf_report_text = classification_report(
    y_true, y_pred,
    labels=class_names,
    target_names=class_names
)

print("===== BLIP Detailed Results =====")
print("Accuracy:", acc)
print("Macro F1:", macro_f1)
print("Confusion Matrix:\n", cm)
print(clf_report_text)

pred_csv_path = os.path.join(OUTPUT_DIR, "deepfashion_blip_predictions_full.csv")
pred_df.to_csv(pred_csv_path, index=False)

report_df = pd.DataFrame(clf_report_dict).T
report_csv_path = os.path.join(OUTPUT_DIR, "deepfashion_blip_classification_report_full.csv")
report_df.to_csv(report_csv_path, index=True)

summary_txt_path = os.path.join(OUTPUT_DIR, "deepfashion_blip_summary_full.txt")
with open(summary_txt_path, "w", encoding="utf-8") as f:
    f.write("BLIP Full Experiment Summary\n")
    f.write(f"Accuracy: {acc}\n")
    f.write(f"Macro F1: {macro_f1}\n\n")
    f.write("Confusion Matrix:\n")
    f.write(str(cm))
    f.write("\n\nClassification Report:\n")
    f.write(clf_report_text)

plt.figure(figsize=(6, 5))
plt.imshow(cm, interpolation="nearest")
plt.title("BLIP Confusion Matrix on Sleeve Type")
plt.colorbar()

tick_marks = np.arange(len(class_names))
plt.xticks(tick_marks, class_names, rotation=45)
plt.yticks(tick_marks, class_names)

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, str(cm[i, j]), ha="center", va="center")

plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()

cm_png_path = os.path.join(OUTPUT_DIR, "deepfashion_blip_confusion_matrix_full.png")
plt.savefig(cm_png_path, dpi=200, bbox_inches="tight")
plt.show()

class_f1 = [clf_report_dict[c]["f1-score"] for c in class_names]

plt.figure(figsize=(6, 4))
plt.bar(class_names, class_f1)
plt.title("BLIP Class-wise F1")
plt.ylabel("F1 Score")
plt.ylim(0, 1)
plt.tight_layout()

class_f1_png_path = os.path.join(OUTPUT_DIR, "deepfashion_blip_classwise_f1_full.png")
plt.savefig(class_f1_png_path, dpi=200, bbox_inches="tight")
plt.show()

style_rows = []
for split_name in ["catalog_like", "less_curated"]:
    sub = pred_df[pred_df["style_split"] == split_name]
    if len(sub) == 0:
        continue

    sub_acc = accuracy_score(sub["ground_truth"], sub["prediction"])
    sub_macro_f1 = f1_score(sub["ground_truth"], sub["prediction"], average="macro")

    style_rows.append({
        "model": MODEL_NAME,
        "subset": split_name,
        "n": len(sub),
        "accuracy": sub_acc,
        "macro_f1": sub_macro_f1,
    })

style_df = pd.DataFrame(style_rows)
style_csv_path = os.path.join(OUTPUT_DIR, "deepfashion_blip_style_metrics_full.csv")
style_df.to_csv(style_csv_path, index=False)

print("\nSaved files:")
print(pred_csv_path)
print(report_csv_path)
print(summary_txt_path)
print(cm_png_path)
print(class_f1_png_path)
print(style_csv_path)

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

OUTPUT_DIR = "/content/drive/MyDrive/dsresearch/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

compare_df = pd.DataFrame([
    {"model": "CLIP", "subset": "overall", "accuracy": 0.6751837591879594, "macro_f1": 0.6617174714283183},
    {"model": "CLIP", "subset": "catalog_like", "accuracy": 0.6853633831349709, "macro_f1": 0.6556128683466492},
    {"model": "CLIP", "subset": "less_curated", "accuracy": 0.6685397900999918, "macro_f1": 0.6601983420997463},

    {"model": "BLIP", "subset": "overall", "accuracy": 0.7802, "macro_f1": 0.7695386211799374},
    {"model": "BLIP", "subset": "catalog_like", "accuracy": 0.7824765763484427, "macro_f1": 0.7661542265078746},
    {"model": "BLIP", "subset": "less_curated", "accuracy": 0.7787142621054371, "macro_f1": 0.768387682753616},
])

compare_csv_path = os.path.join(OUTPUT_DIR, "clip_blip_comparison_metrics.csv")
compare_df.to_csv(compare_csv_path, index=False)

print(compare_df)

overall_df = compare_df[compare_df["subset"] == "overall"]

plt.figure(figsize=(6, 4))
x = range(len(overall_df))
plt.bar([i - 0.15 for i in x], overall_df["accuracy"], width=0.3, label="Accuracy")
plt.bar([i + 0.15 for i in x], overall_df["macro_f1"], width=0.3, label="Macro F1")
plt.xticks(list(x), overall_df["model"])
plt.ylim(0, 1)
plt.title("Overall Comparison: CLIP vs BLIP")
plt.legend()
plt.tight_layout()

overall_png_path = os.path.join(OUTPUT_DIR, "clip_blip_overall_comparison.png")
plt.savefig(overall_png_path, dpi=200, bbox_inches="tight")
plt.show()

style_df = compare_df[compare_df["subset"] != "overall"].copy()

for metric in ["accuracy", "macro_f1"]:
    pivot_df = style_df.pivot(index="subset", columns="model", values=metric)

    plt.figure(figsize=(6, 4))
    pivot_df.plot(kind="bar", ax=plt.gca())
    plt.ylim(0, 1)
    plt.title(f"CLIP vs BLIP by Style Split ({metric})")
    plt.ylabel(metric)
    plt.xticks(rotation=0)
    plt.tight_layout()

    out_path = os.path.join(OUTPUT_DIR, f"clip_blip_style_comparison_{metric}.png")
    plt.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.show()

print("Saved comparison csv:", compare_csv_path)
print("Saved overall figure:", overall_png_path)

In [ ]:
import os

OUTPUT_DIR = "/content/drive/MyDrive/dsresearch/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

blip_pred_path = os.path.join(OUTPUT_DIR, "deepfashion_blip_predictions_full.csv")
pred_df.to_csv(blip_pred_path, index=False)

print("Saved BLIP predictions to:", blip_pred_path)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report,
    precision_recall_fscore_support
)

OUTPUT_DIR = "/content/drive/MyDrive/dsresearch/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

CLIP_PATH = os.path.join(OUTPUT_DIR, "deepfashion_sleeve_clip_predictions_exp1_full.csv")
BLIP_PATH = os.path.join(OUTPUT_DIR, "deepfashion_blip_predictions_full.csv")

class_names = ["long sleeve", "short sleeve", "sleeveless"]

clip_df = pd.read_csv(CLIP_PATH)
blip_df = pd.read_csv(BLIP_PATH)

print("CLIP rows:", len(clip_df))
print("BLIP rows:", len(blip_df))


def compute_overall_metrics(df, model_name):
    y_true = df["ground_truth"]
    y_pred = df["prediction"]

    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")

    return {
        "model": model_name,
        "subset": "overall",
        "n": len(df),
        "accuracy": acc,
        "macro_f1": macro_f1,
    }

def compute_style_metrics(df, model_name):
    rows = []
    for split_name in ["catalog_like", "less_curated"]:
        sub = df[df["style_split"] == split_name]
        if len(sub) == 0:
            continue

        y_true = sub["ground_truth"]
        y_pred = sub["prediction"]

        rows.append({
            "model": model_name,
            "subset": split_name,
            "n": len(sub),
            "accuracy": accuracy_score(y_true, y_pred),
            "macro_f1": f1_score(y_true, y_pred, average="macro"),
        })
    return rows

def compute_class_metrics(df, model_name):
    y_true = df["ground_truth"]
    y_pred = df["prediction"]

    p, r, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=class_names, zero_division=0
    )

    rows = []
    for i, c in enumerate(class_names):
        rows.append({
            "model": model_name,
            "class": c,
            "precision": p[i],
            "recall": r[i],
            "f1": f1[i],
            "support": support[i],
        })
    return rows

def compute_confusion(df):
    return confusion_matrix(df["ground_truth"], df["prediction"], labels=class_names)


overall_rows = [
    compute_overall_metrics(clip_df, "CLIP"),
    compute_overall_metrics(blip_df, "BLIP"),
]
overall_df = pd.DataFrame(overall_rows)

style_rows = compute_style_metrics(clip_df, "CLIP") + compute_style_metrics(blip_df, "BLIP")
style_df = pd.DataFrame(style_rows)

class_rows = compute_class_metrics(clip_df, "CLIP") + compute_class_metrics(blip_df, "BLIP")
class_df = pd.DataFrame(class_rows)

clip_cm = compute_confusion(clip_df)
blip_cm = compute_confusion(blip_df)


gap_rows = []
for model_name in ["CLIP", "BLIP"]:
    sub = style_df[style_df["model"] == model_name].set_index("subset")
    if "catalog_like" in sub.index and "less_curated" in sub.index:
        gap_rows.append({
            "model": model_name,
            "accuracy_gap_catalog_minus_less": sub.loc["catalog_like", "accuracy"] - sub.loc["less_curated", "accuracy"],
            "macro_f1_gap_catalog_minus_less": sub.loc["catalog_like", "macro_f1"] - sub.loc["less_curated", "macro_f1"],
        })

gap_df = pd.DataFrame(gap_rows)


clip_overall = overall_df[overall_df["model"] == "CLIP"].iloc[0]
blip_overall = overall_df[overall_df["model"] == "BLIP"].iloc[0]

improvement_df = pd.DataFrame([{
    "metric": "accuracy",
    "clip": clip_overall["accuracy"],
    "blip": blip_overall["accuracy"],
    "absolute_improvement": blip_overall["accuracy"] - clip_overall["accuracy"],
    "relative_improvement_percent": 100 * (blip_overall["accuracy"] - clip_overall["accuracy"]) / clip_overall["accuracy"],
}, {
    "metric": "macro_f1",
    "clip": clip_overall["macro_f1"],
    "blip": blip_overall["macro_f1"],
    "absolute_improvement": blip_overall["macro_f1"] - clip_overall["macro_f1"],
    "relative_improvement_percent": 100 * (blip_overall["macro_f1"] - clip_overall["macro_f1"]) / clip_overall["macro_f1"],
}])


overall_csv = os.path.join(OUTPUT_DIR, "comparison_overall_metrics.csv")
style_csv = os.path.join(OUTPUT_DIR, "comparison_style_metrics.csv")
class_csv = os.path.join(OUTPUT_DIR, "comparison_class_metrics.csv")
gap_csv = os.path.join(OUTPUT_DIR, "comparison_robustness_gap.csv")
improvement_csv = os.path.join(OUTPUT_DIR, "comparison_relative_improvement.csv")

overall_df.to_csv(overall_csv, index=False)
style_df.to_csv(style_csv, index=False)
class_df.to_csv(class_csv, index=False)
gap_df.to_csv(gap_csv, index=False)
improvement_df.to_csv(improvement_csv, index=False)

print("\n=== Overall Metrics ===")
print(overall_df)

print("\n=== Style Metrics ===")
print(style_df)

print("\n=== Class Metrics ===")
print(class_df)

print("\n=== Robustness Gap ===")
print(gap_df)

print("\n=== Relative Improvement ===")
print(improvement_df)


plt.figure(figsize=(6, 4))
x = np.arange(len(overall_df))
width = 0.35
plt.bar(x - width/2, overall_df["accuracy"], width, label="Accuracy")
plt.bar(x + width/2, overall_df["macro_f1"], width, label="Macro F1")
plt.xticks(x, overall_df["model"])
plt.ylim(0, 1)
plt.title("Overall Performance: CLIP vs BLIP")
plt.legend()
plt.tight_layout()
overall_png = os.path.join(OUTPUT_DIR, "fig_overall_clip_vs_blip.png")
plt.savefig(overall_png, dpi=200, bbox_inches="tight")
plt.show()


for metric in ["accuracy", "macro_f1"]:
    pivot_df = style_df.pivot(index="subset", columns="model", values=metric)

    plt.figure(figsize=(6, 4))
    pivot_df.plot(kind="bar", ax=plt.gca())
    plt.ylim(0, 1)
    plt.title(f"Style Split Comparison ({metric})")
    plt.ylabel(metric)
    plt.xticks(rotation=0)
    plt.tight_layout()

    out_path = os.path.join(OUTPUT_DIR, f"fig_style_comparison_{metric}.png")
    plt.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.show()


pivot_f1 = class_df.pivot(index="class", columns="model", values="f1")

plt.figure(figsize=(7, 4))
pivot_f1.plot(kind="bar", ax=plt.gca())
plt.ylim(0, 1)
plt.title("Per-class F1: CLIP vs BLIP")
plt.ylabel("F1")
plt.xticks(rotation=0)
plt.tight_layout()
class_f1_png = os.path.join(OUTPUT_DIR, "fig_per_class_f1_clip_vs_blip.png")
plt.savefig(class_f1_png, dpi=200, bbox_inches="tight")
plt.show()


for metric in ["precision", "recall"]:
    pivot_metric = class_df.pivot(index="class", columns="model", values=metric)

    plt.figure(figsize=(7, 4))
    pivot_metric.plot(kind="bar", ax=plt.gca())
    plt.ylim(0, 1)
    plt.title(f"Per-class {metric.capitalize()}: CLIP vs BLIP")
    plt.ylabel(metric)
    plt.xticks(rotation=0)
    plt.tight_layout()

    out_path = os.path.join(OUTPUT_DIR, f"fig_per_class_{metric}_clip_vs_blip.png")
    plt.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.show()


if len(gap_df) > 0:
    plt.figure(figsize=(6, 4))
    x = np.arange(len(gap_df))
    width = 0.35
    plt.bar(x - width/2, gap_df["accuracy_gap_catalog_minus_less"], width, label="Accuracy gap")
    plt.bar(x + width/2, gap_df["macro_f1_gap_catalog_minus_less"], width, label="Macro F1 gap")
    plt.xticks(x, gap_df["model"])
    plt.axhline(0, linestyle="--")
    plt.title("Robustness Gap: catalog_like minus less_curated")
    plt.legend()
    plt.tight_layout()
    gap_png = os.path.join(OUTPUT_DIR, "fig_robustness_gap_clip_vs_blip.png")
    plt.savefig(gap_png, dpi=200, bbox_inches="tight")
    plt.show()


fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, cm, title in zip(axes, [clip_cm, blip_cm], ["CLIP Confusion Matrix", "BLIP Confusion Matrix"]):
    im = ax.imshow(cm, interpolation="nearest")
    ax.set_title(title)
    ax.set_xticks(np.arange(len(class_names)))
    ax.set_yticks(np.arange(len(class_names)))
    ax.set_xticklabels(class_names, rotation=45)
    ax.set_yticklabels(class_names)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center")

fig.tight_layout()
cm_compare_png = os.path.join(OUTPUT_DIR, "fig_confusion_matrices_clip_vs_blip.png")
plt.savefig(cm_compare_png, dpi=200, bbox_inches="tight")
plt.show()


summary_txt = os.path.join(OUTPUT_DIR, "comparison_clip_vs_blip_summary.txt")
with open(summary_txt, "w", encoding="utf-8") as f:
    f.write("CLIP vs BLIP Comparison Summary\n\n")
    f.write("Overall Metrics\n")
    f.write(overall_df.to_string(index=False))
    f.write("\n\nStyle Metrics\n")
    f.write(style_df.to_string(index=False))
    f.write("\n\nClass Metrics\n")
    f.write(class_df.to_string(index=False))
    f.write("\n\nRobustness Gap\n")
    f.write(gap_df.to_string(index=False))
    f.write("\n\nRelative Improvement\n")
    f.write(improvement_df.to_string(index=False))

print("\nSaved files to Google Drive:")
print(overall_csv)
print(style_csv)
print(class_csv)
print(gap_csv)
print(improvement_csv)
print(overall_png)
print(class_f1_png)
print(cm_compare_png)
print(summary_txt)

构造 pattern / material 子集，并存到 Google Drive

In [ ]:
from pathlib import Path
import pandas as pd

DRIVE_BASE = Path("/content/drive/MyDrive/dsresearch")
BENCHMARK_DIR = DRIVE_BASE / "Category and Attribute Prediction Benchmark"

ATTR_CLOTH_FILE = BENCHMARK_DIR / "Anno_fine" / "list_attr_cloth.txt"
ATTR_IMG_FILE = BENCHMARK_DIR / "Anno_fine" / "list_attr_img.txt"

print("ATTR_CLOTH_FILE exists:", ATTR_CLOTH_FILE.exists())
print("ATTR_IMG_FILE exists:", ATTR_IMG_FILE.exists())

def parse_attr_cloth(filepath):
    rows = []
    with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
        lines = [line.strip() for line in f if line.strip()]

    for line in lines[2:]:
        parts = line.split()
        if len(parts) < 2:
            continue
        attr_type = int(parts[-1])
        attr_name = " ".join(parts[:-1]).strip()
        rows.append((attr_name, attr_type))

    df_meta = pd.DataFrame(rows, columns=["attr_name", "attr_type"])
    df_meta["attr_index"] = range(len(df_meta))
    return df_meta[["attr_index", "attr_name", "attr_type"]]

def parse_attr_img(filepath, num_attrs):
    rows = []
    with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
        lines = [line.rstrip("\n") for line in f if line.strip()]

    for line in lines[2:]:
        parts = line.split()
        if len(parts) < 1 + num_attrs:
            continue

        image_path = parts[0]
        labels = parts[1:1+num_attrs]

        if len(labels) != num_attrs:
            continue

        row = {"image_path": image_path}
        for i, v in enumerate(labels):
            row[f"attr_{i}"] = int(v)
        rows.append(row)

    return pd.DataFrame(rows)

df_meta = parse_attr_cloth(ATTR_CLOTH_FILE)
df_img = parse_attr_img(ATTR_IMG_FILE, len(df_meta))

rename_map = {
    f"attr_{i}": name for i, name in zip(df_meta["attr_index"], df_meta["attr_name"])
}

df_full = df_img.rename(columns=rename_map)

print("df_full shape:", df_full.shape)
print(df_full.columns.tolist())
print(df_full.head())

In [ ]:
from pathlib import Path

IMAGE_ROOT = Path("/content/deepfashion_data/img_highres")

def resolve_image_path(rel_path):
    rel_path = str(rel_path).strip()
    name = Path(rel_path).name

    if "-img_" not in name:
        return None

    folder_name, img_tail = name.split("-img_", 1)
    img_name = "img_" + img_tail

    candidate = IMAGE_ROOT / folder_name / img_name
    if candidate.exists():
        return str(candidate)
    return None

test_paths = df_full["image_path"].head(5).tolist()
for p in test_paths:
    print("rel_path :", p)
    print("resolved :", resolve_image_path(p))
    print("-" * 60)

In [ ]:
import os
import numpy as np
import pandas as pd
from PIL import Image

OUTPUT_DIR = "/content/drive/MyDrive/dsresearch/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def compute_style_split(image):
    arr = np.array(image).astype(np.float32) / 255.0
    h, w, c = arr.shape

    border = np.concatenate([
        arr[:max(1, h//10), :, :].reshape(-1, 3),
        arr[-max(1, h//10):, :, :].reshape(-1, 3),
        arr[:, :max(1, w//10), :].reshape(-1, 3),
        arr[:, -max(1, w//10):, :].reshape(-1, 3),
    ], axis=0)

    brightness = border.mean()
    color_std = border.std()

    if brightness > 0.82 and color_std < 0.18:
        return "catalog_like"
    return "less_curated"

def build_single_label_subset(df_source, attr_cols, task_name, prompt_label_map=None):
    sub = df_source[["image_path"] + attr_cols].copy()
    sub["positive_count"] = (sub[attr_cols] == 1).sum(axis=1)

    print(f"\n{task_name} positive_count distribution:")
    print(sub["positive_count"].value_counts().sort_index())

    sub = sub[sub["positive_count"] == 1].copy()

    def get_label(row):
        for c in attr_cols:
            if row[c] == 1:
                return prompt_label_map[c] if prompt_label_map else c
        return None

    sub["label"] = sub.apply(get_label, axis=1)

    sub["resolved_path"] = sub["image_path"].apply(resolve_image_path)
    sub = sub[sub["resolved_path"].notna()].copy()

    style_splits = []
    for img_path in sub["resolved_path"]:
        try:
            img = Image.open(img_path).convert("RGB")
            style_splits.append(compute_style_split(img))
        except Exception:
            style_splits.append("less_curated")

    sub["style_split"] = style_splits

    final_df = sub[["resolved_path", "label", "style_split"]].rename(
        columns={"resolved_path": "image_path"}
    ).reset_index(drop=True)

    csv_path = os.path.join(OUTPUT_DIR, f"deepfashion_{task_name}_with_style.csv")
    final_df.to_csv(csv_path, index=False)

    print(f"\n===== {task_name} subset =====")
    print("Rows:", len(final_df))
    print(final_df["label"].value_counts())
    print(final_df["style_split"].value_counts())
    print("Saved to:", csv_path)

    return final_df, csv_path


pattern_cols = ["floral", "graphic", "striped", "solid"]
pattern_label_map = {
    "floral": "floral",
    "graphic": "graphic",
    "striped": "striped",
    "solid": "solid",
}

pattern_df, pattern_csv = build_single_label_subset(
    df_source=df_full,
    attr_cols=pattern_cols,
    task_name="pattern",
    prompt_label_map=pattern_label_map
)


material_cols = ["denim", "chiffon", "cotton", "leather", "knit"]
material_label_map = {
    "denim": "denim",
    "chiffon": "chiffon",
    "cotton": "cotton",
    "leather": "leather",
    "knit": "knit",
}

material_df, material_csv = build_single_label_subset(
    df_source=df_full,
    attr_cols=material_cols,
    task_name="material",
    prompt_label_map=material_label_map
)

In [ ]:
import pandas as pd

pattern_check = pd.read_csv("/content/drive/MyDrive/dsresearch/outputs/deepfashion_pattern_with_style.csv")
material_check = pd.read_csv("/content/drive/MyDrive/dsresearch/outputs/deepfashion_material_with_style.csv")

print("Pattern rows:", len(pattern_check))
print(pattern_check["label"].value_counts())
print(pattern_check["style_split"].value_counts())

print("\nMaterial rows:", len(material_check))
print(material_check["label"].value_counts())
print(material_check["style_split"].value_counts())

Pattern / Material 的 CLIP full experiment

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from PIL import Image, ImageFile
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import torch
from transformers import CLIPModel, CLIPProcessor

ImageFile.LOAD_TRUNCATED_IMAGES = True
Image.MAX_IMAGE_PIXELS = None

OUTPUT_DIR = "/content/drive/MyDrive/dsresearch/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

def run_clip_task(task_name, csv_path, class_names, prompts):
    print(f"\n===== Running CLIP on {task_name} =====")
    df_test = pd.read_csv(csv_path).reset_index(drop=True)

    print("Rows:", len(df_test))
    print("\nLabel distribution:")
    print(df_test["label"].value_counts())
    print("\nStyle split distribution:")
    print(df_test["style_split"].value_counts())

    model_id = "openai/clip-vit-base-patch32"
    model = CLIPModel.from_pretrained(model_id).to(device)
    processor = CLIPProcessor.from_pretrained(model_id)
    model.eval()

    text_inputs = processor(
        text=prompts,
        return_tensors="pt",
        padding=True,
        truncation=True
    )
    text_inputs = {k: v.to(device) for k, v in text_inputs.items()}

    with torch.no_grad():
        text_outputs = model.text_model(
            input_ids=text_inputs["input_ids"],
            attention_mask=text_inputs["attention_mask"]
        )
        text_features = text_outputs.pooler_output
        text_features = model.text_projection(text_features)
        text_features = text_features / text_features.norm(p=2, dim=-1, keepdim=True)

    y_true, y_pred, rows = [], [], []

    for _, row in tqdm(df_test.iterrows(), total=len(df_test), desc=f"CLIP-{task_name}"):
        try:
            image = Image.open(row["image_path"]).convert("RGB")
            image_inputs = processor(images=image, return_tensors="pt")
            image_inputs = {k: v.to(device) for k, v in image_inputs.items()}

            with torch.no_grad():
                image_outputs = model.vision_model(pixel_values=image_inputs["pixel_values"])
                image_features = image_outputs.pooler_output
                image_features = model.visual_projection(image_features)
                image_features = image_features / image_features.norm(p=2, dim=-1, keepdim=True)

                logits = image_features @ text_features.T
                pred_idx = int(logits.argmax(dim=1).cpu().item())

            pred = class_names[pred_idx]
            gt = row["label"]

            y_true.append(gt)
            y_pred.append(pred)

            rows.append({
                "image_path": row["image_path"],
                "style_split": row["style_split"],
                "ground_truth": gt,
                "prediction": pred,
                "correct": int(gt == pred),
                "model": "CLIP",
                "task": task_name,
            })
        except Exception as e:
            print("Skipping due to error:", row["image_path"], e)

    pred_df = pd.DataFrame(rows)

    pred_path = os.path.join(OUTPUT_DIR, f"{task_name}_clip_predictions_full.csv")
    pred_df.to_csv(pred_path, index=False)

    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    cm = confusion_matrix(y_true, y_pred, labels=class_names)
    clf_report = classification_report(
        y_true, y_pred,
        labels=class_names,
        target_names=class_names
    )

    print("\n===== Overall Results =====")
    print("Accuracy:", acc)
    print("Macro F1:", macro_f1)
    print("Confusion Matrix:\n", cm)
    print(clf_report)

    plt.figure(figsize=(6, 5))
    plt.imshow(cm, interpolation="nearest")
    plt.title(f"CLIP Confusion Matrix: {task_name}")
    plt.colorbar()
    tick_marks = np.arange(len(class_names))
    plt.xticks(tick_marks, class_names, rotation=45)
    plt.yticks(tick_marks, class_names)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    cm_path = os.path.join(OUTPUT_DIR, f"{task_name}_clip_confusion_matrix_full.png")
    plt.savefig(cm_path, dpi=200, bbox_inches="tight")
    plt.show()

    summary_rows = [{
        "model": "CLIP",
        "task": task_name,
        "subset": "overall",
        "n": len(pred_df),
        "accuracy": acc,
        "macro_f1": macro_f1,
    }]

    for split_name in ["catalog_like", "less_curated"]:
        sub = pred_df[pred_df["style_split"] == split_name]
        if len(sub) == 0:
            continue

        sub_acc = accuracy_score(sub["ground_truth"], sub["prediction"])
        sub_macro_f1 = f1_score(sub["ground_truth"], sub["prediction"], average="macro")

        print(f"\n{split_name}")
        print("n =", len(sub))
        print("accuracy =", sub_acc)
        print("macro_f1 =", sub_macro_f1)

        summary_rows.append({
            "model": "CLIP",
            "task": task_name,
            "subset": split_name,
            "n": len(sub),
            "accuracy": sub_acc,
            "macro_f1": sub_macro_f1,
        })

    summary_df = pd.DataFrame(summary_rows)
    summary_path = os.path.join(OUTPUT_DIR, f"{task_name}_clip_summary_metrics_full.csv")
    summary_df.to_csv(summary_path, index=False)

    print("\nSaved predictions:", pred_path)
    print("Saved summary:", summary_path)
    print("Saved confusion matrix:", cm_path)

    return pred_df, summary_df

跑 pattern / material 的 CLIP

In [ ]:
pattern_classes = ["floral", "graphic", "striped", "solid"]
pattern_prompts = [
    "a floral garment",
    "a graphic garment",
    "a striped garment",
    "a solid garment",
]

material_classes = ["denim", "chiffon", "cotton", "leather", "knit"]
material_prompts = [
    "a denim garment",
    "a chiffon garment",
    "a cotton garment",
    "a leather garment",
    "a knit garment",
]

pattern_clip_pred, pattern_clip_summary = run_clip_task(
    task_name="pattern",
    csv_path="/content/drive/MyDrive/dsresearch/outputs/deepfashion_pattern_with_style.csv",
    class_names=pattern_classes,
    prompts=pattern_prompts
)

material_clip_pred, material_clip_summary = run_clip_task(
    task_name="material",
    csv_path="/content/drive/MyDrive/dsresearch/outputs/deepfashion_material_with_style.csv",
    class_names=material_classes,
    prompts=material_prompts
)

BLIP 的 pattern/material

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from PIL import Image, ImageFile
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

import torch
from transformers import BlipProcessor, BlipForImageTextRetrieval

ImageFile.LOAD_TRUNCATED_IMAGES = True
Image.MAX_IMAGE_PIXELS = None

OUTPUT_DIR = "/content/drive/MyDrive/dsresearch/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

def run_blip_task(task_name, csv_path, class_names, prompts):
    print(f"\n===== Running BLIP on {task_name} =====")
    df_test = pd.read_csv(csv_path).reset_index(drop=True)

    print("Rows:", len(df_test))
    print("\nLabel distribution:")
    print(df_test["label"].value_counts())
    print("\nStyle split distribution:")
    print(df_test["style_split"].value_counts())

    print("Loading BLIP model...")
    processor = BlipProcessor.from_pretrained("Salesforce/blip-itm-base-coco")
    model = BlipForImageTextRetrieval.from_pretrained("Salesforce/blip-itm-base-coco").to(device)
    model.eval()
    print("BLIP model loaded.")

    y_true, y_pred, rows = [], [], []

    for _, row in tqdm(df_test.iterrows(), total=len(df_test), desc=f"BLIP-{task_name}"):
        try:
            image = Image.open(row["image_path"]).convert("RGB")

            scores = []
            for prompt in prompts:
                inputs = processor(
                    images=image,
                    text=prompt,
                    return_tensors="pt",
                    padding=True,
                    truncation=True
                )
                inputs = {k: v.to(device) for k, v in inputs.items()}

                with torch.no_grad():
                    outputs = model(**inputs, use_itm_head=True)
                    score = outputs.itm_score[:, 1].item()
                scores.append(score)

            pred_idx = int(np.argmax(scores))
            pred = class_names[pred_idx]
            gt = row["label"]

            y_true.append(gt)
            y_pred.append(pred)

            rows.append({
                "image_path": row["image_path"],
                "style_split": row["style_split"],
                "ground_truth": gt,
                "prediction": pred,
                "correct": int(gt == pred),
                "model": "BLIP",
                "task": task_name,
            })

        except Exception as e:
            print("Skipping due to error:", row["image_path"], e)

    pred_df = pd.DataFrame(rows)

    pred_path = os.path.join(OUTPUT_DIR, f"{task_name}_blip_predictions_full.csv")
    pred_df.to_csv(pred_path, index=False)

    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    cm = confusion_matrix(y_true, y_pred, labels=class_names)
    clf_report = classification_report(
        y_true, y_pred,
        labels=class_names,
        target_names=class_names
    )

    print("\n===== Overall Results =====")
    print("Accuracy:", acc)
    print("Macro F1:", macro_f1)
    print("Confusion Matrix:\n", cm)
    print(clf_report)

    plt.figure(figsize=(6, 5))
    plt.imshow(cm, interpolation="nearest")
    plt.title(f"BLIP Confusion Matrix: {task_name}")
    plt.colorbar()
    tick_marks = np.arange(len(class_names))
    plt.xticks(tick_marks, class_names, rotation=45)
    plt.yticks(tick_marks, class_names)

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, str(cm[i, j]), ha="center", va="center")

    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()

    cm_path = os.path.join(OUTPUT_DIR, f"{task_name}_blip_confusion_matrix_full.png")
    plt.savefig(cm_path, dpi=200, bbox_inches="tight")
    plt.show()

    summary_rows = [{
        "model": "BLIP",
        "task": task_name,
        "subset": "overall",
        "n": len(pred_df),
        "accuracy": acc,
        "macro_f1": macro_f1,
    }]

    print("\n===== By Style Split =====")
    for split_name in ["catalog_like", "less_curated"]:
        sub = pred_df[pred_df["style_split"] == split_name]
        if len(sub) == 0:
            continue

        sub_acc = accuracy_score(sub["ground_truth"], sub["prediction"])
        sub_macro_f1 = f1_score(sub["ground_truth"], sub["prediction"], average="macro")

        print(f"\nBLIP | {split_name}")
        print("n =", len(sub))
        print("accuracy =", sub_acc)
        print("macro_f1 =", sub_macro_f1)

        summary_rows.append({
            "model": "BLIP",
            "task": task_name,
            "subset": split_name,
            "n": len(sub),
            "accuracy": sub_acc,
            "macro_f1": sub_macro_f1,
        })

    summary_df = pd.DataFrame(summary_rows)
    summary_path = os.path.join(OUTPUT_DIR, f"{task_name}_blip_summary_metrics_full.csv")
    summary_df.to_csv(summary_path, index=False)

    print("\nSaved predictions:", pred_path)
    print("Saved summary:", summary_path)
    print("Saved confusion matrix:", cm_path)

    return pred_df, summary_df

In [ ]:
pattern_classes = ["floral", "graphic", "striped", "solid"]
pattern_prompts = [
    "a floral garment",
    "a graphic garment",
    "a striped garment",
    "a solid garment",
]

material_classes = ["denim", "chiffon", "cotton", "leather", "knit"]
material_prompts = [
    "a denim garment",
    "a chiffon garment",
    "a cotton garment",
    "a leather garment",
    "a knit garment",
]

pattern_blip_pred, pattern_blip_summary = run_blip_task(
    task_name="pattern",
    csv_path="/content/drive/MyDrive/dsresearch/outputs/deepfashion_pattern_with_style.csv",
    class_names=pattern_classes,
    prompts=pattern_prompts
)

material_blip_pred, material_blip_summary = run_blip_task(
    task_name="material",
    csv_path="/content/drive/MyDrive/dsresearch/outputs/deepfashion_material_with_style.csv",
    class_names=material_classes,
    prompts=material_prompts
)

CLIP vs BLIP × sleeve/pattern/material

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    confusion_matrix
)


OUTPUT_DIR = "/content/drive/MyDrive/dsresearch/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

FILE_MAP = {
    ("CLIP", "sleeve"): "/content/drive/MyDrive/dsresearch/outputs/deepfashion_sleeve_clip_predictions_exp1_full.csv",
    ("CLIP", "pattern"): "/content/drive/MyDrive/dsresearch/outputs/pattern_clip_predictions_full.csv",
    ("CLIP", "material"): "/content/drive/MyDrive/dsresearch/outputs/material_clip_predictions_full.csv",

    ("BLIP", "sleeve"): "/content/drive/MyDrive/dsresearch/outputs/deepfashion_blip_predictions_full.csv",
    ("BLIP", "pattern"): "/content/drive/MyDrive/dsresearch/outputs/pattern_blip_predictions_full.csv",
    ("BLIP", "material"): "/content/drive/MyDrive/dsresearch/outputs/material_blip_predictions_full.csv",
}

TASK_CLASSES = {
    "sleeve": ["long sleeve", "short sleeve", "sleeveless"],
    "pattern": ["floral", "graphic", "striped", "solid"],
    "material": ["denim", "chiffon", "cotton", "leather", "knit"],
}

TASK_ORDER = ["sleeve", "pattern", "material"]
MODEL_ORDER = ["CLIP", "BLIP"]
SPLIT_ORDER = ["overall", "catalog_like", "less_curated"]


data = {}
for key, path in FILE_MAP.items():
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing file: {path}")
    df = pd.read_csv(path)
    data[key] = df
    print(f"Loaded {key}: {len(df)} rows")


overall_rows = []
style_rows = []
class_rows = []
gap_rows = []
improvement_rows = []
conf_mats = {}

def compute_metrics(df, classes):
    y_true = df["ground_truth"]
    y_pred = df["prediction"]

    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    p, r, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=classes, zero_division=0
    )
    cm = confusion_matrix(y_true, y_pred, labels=classes)

    return acc, macro_f1, p, r, f1, support, cm

for model in MODEL_ORDER:
    for task in TASK_ORDER:
        df = data[(model, task)]
        classes = TASK_CLASSES[task]

        acc, macro_f1, p, r, f1, support, cm = compute_metrics(df, classes)
        conf_mats[(model, task)] = cm

        overall_rows.append({
            "model": model,
            "task": task,
            "subset": "overall",
            "n": len(df),
            "accuracy": acc,
            "macro_f1": macro_f1,
        })

        for i, cls in enumerate(classes):
            class_rows.append({
                "model": model,
                "task": task,
                "class": cls,
                "precision": p[i],
                "recall": r[i],
                "f1": f1[i],
                "support": support[i],
            })

        for split_name in ["catalog_like", "less_curated"]:
            sub = df[df["style_split"] == split_name]
            sub_acc, sub_macro_f1, _, _, _, _, _ = compute_metrics(sub, classes)

            style_rows.append({
                "model": model,
                "task": task,
                "subset": split_name,
                "n": len(sub),
                "accuracy": sub_acc,
                "macro_f1": sub_macro_f1,
            })

overall_df = pd.DataFrame(overall_rows)
style_df = pd.DataFrame(style_rows)
class_df = pd.DataFrame(class_rows)

# robustness gap
for model in MODEL_ORDER:
    for task in TASK_ORDER:
        sub = style_df[(style_df["model"] == model) & (style_df["task"] == task)].set_index("subset")
        gap_rows.append({
            "model": model,
            "task": task,
            "accuracy_gap_catalog_minus_less": sub.loc["catalog_like", "accuracy"] - sub.loc["less_curated", "accuracy"],
            "macro_f1_gap_catalog_minus_less": sub.loc["catalog_like", "macro_f1"] - sub.loc["less_curated", "macro_f1"],
        })

gap_df = pd.DataFrame(gap_rows)

for task in TASK_ORDER:
    clip_row = overall_df[(overall_df["model"] == "CLIP") & (overall_df["task"] == task)].iloc[0]
    blip_row = overall_df[(overall_df["model"] == "BLIP") & (overall_df["task"] == task)].iloc[0]

    for metric in ["accuracy", "macro_f1"]:
        clip_val = clip_row[metric]
        blip_val = blip_row[metric]
        improvement_rows.append({
            "task": task,
            "metric": metric,
            "clip": clip_val,
            "blip": blip_val,
            "absolute_improvement": blip_val - clip_val,
            "relative_improvement_percent": 100 * (blip_val - clip_val) / clip_val if clip_val != 0 else np.nan,
        })

improvement_df = pd.DataFrame(improvement_rows)

error_rows = []
for model in MODEL_ORDER:
    for task in TASK_ORDER:
        cm = conf_mats[(model, task)]
        total = cm.sum()
        correct = np.trace(cm)
        error = total - correct
        error_rows.append({
            "model": model,
            "task": task,
            "total": total,
            "correct": correct,
            "error": error,
            "error_rate": error / total,
        })
error_df = pd.DataFrame(error_rows)



overall_csv = os.path.join(OUTPUT_DIR, "paper_overall_metrics.csv")
style_csv = os.path.join(OUTPUT_DIR, "paper_style_metrics.csv")
class_csv = os.path.join(OUTPUT_DIR, "paper_class_metrics.csv")
gap_csv = os.path.join(OUTPUT_DIR, "paper_style_gap_metrics.csv")
improvement_csv = os.path.join(OUTPUT_DIR, "paper_relative_improvement.csv")
error_csv = os.path.join(OUTPUT_DIR, "paper_error_metrics.csv")

overall_df.to_csv(overall_csv, index=False)
style_df.to_csv(style_csv, index=False)
class_df.to_csv(class_csv, index=False)
gap_df.to_csv(gap_csv, index=False)
improvement_df.to_csv(improvement_csv, index=False)
error_df.to_csv(error_csv, index=False)

print("Saved metric tables.")


fig = plt.figure(figsize=(16, 11))

ax1 = plt.subplot(2, 2, 1)

heat_tasks = TASK_ORDER
heat_models = MODEL_ORDER

heat_acc = np.zeros((len(heat_tasks), len(heat_models)))
heat_f1 = np.zeros((len(heat_tasks), len(heat_models)))

for i, task in enumerate(heat_tasks):
    for j, model in enumerate(heat_models):
        row = overall_df[(overall_df["task"] == task) & (overall_df["model"] == model)].iloc[0]
        heat_acc[i, j] = row["accuracy"]
        heat_f1[i, j] = row["macro_f1"]

im = ax1.imshow(heat_acc, aspect="auto")
ax1.set_xticks(np.arange(len(heat_models)))
ax1.set_xticklabels(heat_models)
ax1.set_yticks(np.arange(len(heat_tasks)))
ax1.set_yticklabels(heat_tasks)
ax1.set_title("A. Overall Performance (cell shows Acc / Macro-F1)")

for i in range(len(heat_tasks)):
    for j in range(len(heat_models)):
        txt = f"{heat_acc[i,j]:.3f}\n{heat_f1[i,j]:.3f}"
        ax1.text(j, i, txt, ha="center", va="center")

plt.colorbar(im, ax=ax1, fraction=0.046, pad=0.04)

ax2 = plt.subplot(2, 2, 2)

x_positions = []
x_labels = []
catalog_vals = []
less_vals = []
colors = []

for task in TASK_ORDER:
    for model in MODEL_ORDER:
        x_labels.append(f"{task}\n{model}")
        sub = style_df[(style_df["task"] == task) & (style_df["model"] == model)].set_index("subset")
        catalog_vals.append(sub.loc["catalog_like", "accuracy"])
        less_vals.append(sub.loc["less_curated", "accuracy"])

x = np.arange(len(x_labels))
ax2.scatter(x, catalog_vals, s=80, label="catalog_like")
ax2.scatter(x, less_vals, s=80, label="less_curated")

for i in range(len(x)):
    ax2.plot([x[i], x[i]], [catalog_vals[i], less_vals[i]])

ax2.set_xticks(x)
ax2.set_xticklabels(x_labels, rotation=0)
ax2.set_ylim(0, 1)
ax2.set_title("B. Style Robustness (Accuracy by style split)")
ax2.legend()

ax3 = plt.subplot(2, 2, 3)

row_labels = []
f1_matrix_rows = []

for task in TASK_ORDER:
    classes = TASK_CLASSES[task]
    for cls in classes:
        row_labels.append(f"{task}:{cls}")
        row_vals = []
        for model in MODEL_ORDER:
            row = class_df[
                (class_df["task"] == task) &
                (class_df["class"] == cls) &
                (class_df["model"] == model)
            ].iloc[0]
            row_vals.append(row["f1"])
        f1_matrix_rows.append(row_vals)

f1_matrix = np.array(f1_matrix_rows)
im2 = ax3.imshow(f1_matrix, aspect="auto")
ax3.set_xticks(np.arange(len(MODEL_ORDER)))
ax3.set_xticklabels(MODEL_ORDER)
ax3.set_yticks(np.arange(len(row_labels)))
ax3.set_yticklabels(row_labels)
ax3.set_title("C. Class-wise F1 Heatmap")

for i in range(f1_matrix.shape[0]):
    for j in range(f1_matrix.shape[1]):
        ax3.text(j, i, f"{f1_matrix[i,j]:.2f}", ha="center", va="center")

plt.colorbar(im2, ax=ax3, fraction=0.046, pad=0.04)

ax4 = plt.subplot(2, 2, 4)

imp_plot_df = improvement_df.copy()
tasks_for_plot = TASK_ORDER
metrics_for_plot = ["accuracy", "macro_f1"]

bar_width = 0.35
base_x = np.arange(len(tasks_for_plot))

acc_imps = []
f1_imps = []
for task in tasks_for_plot:
    acc_imps.append(
        imp_plot_df[(imp_plot_df["task"] == task) & (imp_plot_df["metric"] == "accuracy")]["absolute_improvement"].iloc[0]
    )
    f1_imps.append(
        imp_plot_df[(imp_plot_df["task"] == task) & (imp_plot_df["metric"] == "macro_f1")]["absolute_improvement"].iloc[0]
    )

ax4.bar(base_x - bar_width/2, acc_imps, width=bar_width, label="Accuracy gain")
ax4.bar(base_x + bar_width/2, f1_imps, width=bar_width, label="Macro-F1 gain")
ax4.axhline(0, linestyle="--")
ax4.set_xticks(base_x)
ax4.set_xticklabels(tasks_for_plot)
ax4.set_title("D. BLIP Improvement over CLIP")
ax4.legend()

plt.tight_layout()

main_png = os.path.join(OUTPUT_DIR, "paper_main_multitask_comparison.png")
main_pdf = os.path.join(OUTPUT_DIR, "paper_main_multitask_comparison.pdf")
plt.savefig(main_png, dpi=300, bbox_inches="tight")
plt.savefig(main_pdf, bbox_inches="tight")
plt.show()


fig, axes = plt.subplots(3, 2, figsize=(14, 18))

for i, task in enumerate(TASK_ORDER):
    classes = TASK_CLASSES[task]
    for j, model in enumerate(MODEL_ORDER):
        ax = axes[i, j]
        cm = conf_mats[(model, task)]
        im = ax.imshow(cm, interpolation="nearest")
        ax.set_title(f"{task} | {model}")
        ax.set_xticks(np.arange(len(classes)))
        ax.set_yticks(np.arange(len(classes)))
        ax.set_xticklabels(classes, rotation=45)
        ax.set_yticklabels(classes)
        ax.set_xlabel("Predicted")
        ax.set_ylabel("True")

        for r in range(cm.shape[0]):
            for c in range(cm.shape[1]):
                ax.text(c, r, str(cm[r, c]), ha="center", va="center")

plt.tight_layout()
cm_png = os.path.join(OUTPUT_DIR, "paper_confusion_matrices_3tasks_2models.png")
cm_pdf = os.path.join(OUTPUT_DIR, "paper_confusion_matrices_3tasks_2models.pdf")
plt.savefig(cm_png, dpi=300, bbox_inches="tight")
plt.savefig(cm_pdf, bbox_inches="tight")
plt.show()


plt.figure(figsize=(10, 5))

x_labels = []
acc_gap_vals = []
f1_gap_vals = []

for task in TASK_ORDER:
    for model in MODEL_ORDER:
        row = gap_df[(gap_df["task"] == task) & (gap_df["model"] == model)].iloc[0]
        x_labels.append(f"{task}\n{model}")
        acc_gap_vals.append(row["accuracy_gap_catalog_minus_less"])
        f1_gap_vals.append(row["macro_f1_gap_catalog_minus_less"])

x = np.arange(len(x_labels))
width = 0.35

plt.bar(x - width/2, acc_gap_vals, width=width, label="Accuracy gap")
plt.bar(x + width/2, f1_gap_vals, width=width, label="Macro-F1 gap")
plt.axhline(0, linestyle="--")
plt.xticks(x, x_labels)
plt.title("Style Gap: catalog_like minus less_curated")
plt.legend()
plt.tight_layout()

gap_png = os.path.join(OUTPUT_DIR, "paper_style_gap_comparison.png")
gap_pdf = os.path.join(OUTPUT_DIR, "paper_style_gap_comparison.pdf")
plt.savefig(gap_png, dpi=300, bbox_inches="tight")
plt.savefig(gap_pdf, bbox_inches="tight")
plt.show()


fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, task in enumerate(TASK_ORDER):
    ax = axes[idx]
    sub = class_df[class_df["task"] == task]
    pivot = sub.pivot(index="class", columns="model", values="f1")
    pivot.plot(kind="bar", ax=ax)
    ax.set_title(f"{task}: Per-class F1")
    ax.set_ylim(0, 1)
    ax.set_xlabel("")
    ax.set_ylabel("F1")
    ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
classf1_png = os.path.join(OUTPUT_DIR, "paper_per_class_f1_by_task.png")
classf1_pdf = os.path.join(OUTPUT_DIR, "paper_per_class_f1_by_task.pdf")
plt.savefig(classf1_png, dpi=300, bbox_inches="tight")
plt.savefig(classf1_pdf, bbox_inches="tight")
plt.show()


summary_txt = os.path.join(OUTPUT_DIR, "paper_multitask_comparison_summary.txt")
with open(summary_txt, "w", encoding="utf-8") as f:
    f.write("Multitask Comparison Summary: CLIP vs BLIP\n\n")
    f.write("=== Overall Metrics ===\n")
    f.write(overall_df.to_string(index=False))
    f.write("\n\n=== Style Metrics ===\n")
    f.write(style_df.to_string(index=False))
    f.write("\n\n=== Class Metrics ===\n")
    f.write(class_df.to_string(index=False))
    f.write("\n\n=== Style Gap ===\n")
    f.write(gap_df.to_string(index=False))
    f.write("\n\n=== Relative Improvement ===\n")
    f.write(improvement_df.to_string(index=False))
    f.write("\n\n=== Error Metrics ===\n")
    f.write(error_df.to_string(index=False))

print("\nSaved to Google Drive:")
print(overall_csv)
print(style_csv)
print(class_csv)
print(gap_csv)
print(improvement_csv)
print(error_csv)
print(main_png)
print(main_pdf)
print(cm_png)
print(cm_pdf)
print(gap_png)
print(gap_pdf)
print(classf1_png)
print(classf1_pdf)
print(summary_txt)

Normalized confusion matrices + per-class improvement + bootstrap CI

这段代码会：
	•	读取 6 个 prediction CSV
	•	计算 overall metrics
	•	做 bootstrap 95% CI
	•	生成 row-normalized confusion matrices
	•	生成 per-class F1 improvement heatmap
	•	保存所有结果到 Google Drive

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, precision_recall_fscore_support

OUTPUT_DIR = "/content/drive/MyDrive/dsresearch/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

FILE_MAP = {
    ("CLIP", "sleeve"): "/content/drive/MyDrive/dsresearch/outputs/deepfashion_sleeve_clip_predictions_exp1_full.csv",
    ("CLIP", "pattern"): "/content/drive/MyDrive/dsresearch/outputs/pattern_clip_predictions_full.csv",
    ("CLIP", "material"): "/content/drive/MyDrive/dsresearch/outputs/material_clip_predictions_full.csv",

    ("BLIP", "sleeve"): "/content/drive/MyDrive/dsresearch/outputs/deepfashion_blip_predictions_full.csv",
    ("BLIP", "pattern"): "/content/drive/MyDrive/dsresearch/outputs/pattern_blip_predictions_full.csv",
    ("BLIP", "material"): "/content/drive/MyDrive/dsresearch/outputs/material_blip_predictions_full.csv",
}

TASK_CLASSES = {
    "sleeve": ["long sleeve", "short sleeve", "sleeveless"],
    "pattern": ["floral", "graphic", "striped", "solid"],
    "material": ["denim", "chiffon", "cotton", "leather", "knit"],
}

TASK_ORDER = ["sleeve", "pattern", "material"]
MODEL_ORDER = ["CLIP", "BLIP"]

data = {}
for key, path in FILE_MAP.items():
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing file: {path}")
    data[key] = pd.read_csv(path)
    print("Loaded", key, "rows =", len(data[key]))

def bootstrap_metric(y_true, y_pred, metric_fn, n_boot=1000, seed=42):
    rng = np.random.default_rng(seed)
    n = len(y_true)
    scores = []

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        score = metric_fn(y_true[idx], y_pred[idx])
        scores.append(score)

    scores = np.array(scores)
    return {
        "mean": scores.mean(),
        "ci_lower": np.percentile(scores, 2.5),
        "ci_upper": np.percentile(scores, 97.5),
    }

def normalize_confusion(cm):
    row_sums = cm.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1
    return cm / row_sums


ci_rows = []
class_rows = []
norm_cms = {}

for model in MODEL_ORDER:
    for task in TASK_ORDER:
        df = data[(model, task)]
        classes = TASK_CLASSES[task]

        y_true = df["ground_truth"].values
        y_pred = df["prediction"].values

        acc = accuracy_score(y_true, y_pred)
        macro_f1 = f1_score(y_true, y_pred, average="macro")
        acc_ci = bootstrap_metric(y_true, y_pred, accuracy_score, n_boot=1000, seed=42)
        f1_ci = bootstrap_metric(
            y_true, y_pred,
            lambda yt, yp: f1_score(yt, yp, average="macro"),
            n_boot=1000,
            seed=42
        )

        ci_rows.append({
            "model": model,
            "task": task,
            "n": len(df),
            "accuracy": acc,
            "accuracy_ci_lower": acc_ci["ci_lower"],
            "accuracy_ci_upper": acc_ci["ci_upper"],
            "macro_f1": macro_f1,
            "macro_f1_ci_lower": f1_ci["ci_lower"],
            "macro_f1_ci_upper": f1_ci["ci_upper"],
        })

        p, r, f1, support = precision_recall_fscore_support(
            y_true, y_pred, labels=classes, zero_division=0
        )
        for i, cls in enumerate(classes):
            class_rows.append({
                "model": model,
                "task": task,
                "class": cls,
                "precision": p[i],
                "recall": r[i],
                "f1": f1[i],
                "support": support[i],
            })

        cm = confusion_matrix(y_true, y_pred, labels=classes)
        norm_cms[(model, task)] = normalize_confusion(cm)

ci_df = pd.DataFrame(ci_rows)
class_df = pd.DataFrame(class_rows)

ci_csv = os.path.join(OUTPUT_DIR, "paper_bootstrap_ci_metrics.csv")
class_csv = os.path.join(OUTPUT_DIR, "paper_class_metrics_with_ci_context.csv")
ci_df.to_csv(ci_csv, index=False)
class_df.to_csv(class_csv, index=False)

print("\nBootstrap CI metrics:")
print(ci_df)


fig, axes = plt.subplots(3, 2, figsize=(14, 18))

for i, task in enumerate(TASK_ORDER):
    classes = TASK_CLASSES[task]
    for j, model in enumerate(MODEL_ORDER):
        ax = axes[i, j]
        ncm = norm_cms[(model, task)]
        im = ax.imshow(ncm, vmin=0, vmax=1, interpolation="nearest")
        ax.set_title(f"{task} | {model}")
        ax.set_xticks(np.arange(len(classes)))
        ax.set_yticks(np.arange(len(classes)))
        ax.set_xticklabels(classes, rotation=45)
        ax.set_yticklabels(classes)
        ax.set_xlabel("Predicted")
        ax.set_ylabel("True")

        for r in range(ncm.shape[0]):
            for c in range(ncm.shape[1]):
                ax.text(c, r, f"{ncm[r, c]:.2f}", ha="center", va="center")

fig.tight_layout()
norm_cm_png = os.path.join(OUTPUT_DIR, "paper_normalized_confusion_matrices.png")
norm_cm_pdf = os.path.join(OUTPUT_DIR, "paper_normalized_confusion_matrices.pdf")
plt.savefig(norm_cm_png, dpi=300, bbox_inches="tight")
plt.savefig(norm_cm_pdf, bbox_inches="tight")
plt.show()


fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, metric, lower_col, upper_col, title in [
    (axes[0], "accuracy", "accuracy_ci_lower", "accuracy_ci_upper", "Overall Accuracy with 95% Bootstrap CI"),
    (axes[1], "macro_f1", "macro_f1_ci_lower", "macro_f1_ci_upper", "Overall Macro-F1 with 95% Bootstrap CI"),
]:
    x_labels = []
    means = []
    lowers = []
    uppers = []

    for task in TASK_ORDER:
        for model in MODEL_ORDER:
            row = ci_df[(ci_df["task"] == task) & (ci_df["model"] == model)].iloc[0]
            x_labels.append(f"{task}\n{model}")
            means.append(row[metric])
            lowers.append(row[metric] - row[lower_col])
            uppers.append(row[upper_col] - row[metric])

    x = np.arange(len(x_labels))
    ax.bar(x, means)
    ax.errorbar(x, means, yerr=[lowers, uppers], fmt="none", capsize=4)
    ax.set_xticks(x)
    ax.set_xticklabels(x_labels)
    ax.set_ylim(0, 1)
    ax.set_title(title)

plt.tight_layout()
ci_fig_png = os.path.join(OUTPUT_DIR, "paper_overall_metrics_with_bootstrap_ci.png")
ci_fig_pdf = os.path.join(OUTPUT_DIR, "paper_overall_metrics_with_bootstrap_ci.pdf")
plt.savefig(ci_fig_png, dpi=300, bbox_inches="tight")
plt.savefig(ci_fig_pdf, bbox_inches="tight")
plt.show()


row_labels = []
improve_rows = []

for task in TASK_ORDER:
    classes = TASK_CLASSES[task]
    for cls in classes:
        clip_row = class_df[
            (class_df["task"] == task) & (class_df["class"] == cls) & (class_df["model"] == "CLIP")
        ].iloc[0]
        blip_row = class_df[
            (class_df["task"] == task) & (class_df["class"] == cls) & (class_df["model"] == "BLIP")
        ].iloc[0]

        row_labels.append(f"{task}:{cls}")
        improve_rows.append([
            blip_row["precision"] - clip_row["precision"],
            blip_row["recall"] - clip_row["recall"],
            blip_row["f1"] - clip_row["f1"],
        ])

improve_mat = np.array(improve_rows)

plt.figure(figsize=(8, 10))
im = plt.imshow(improve_mat, aspect="auto")
plt.xticks(np.arange(3), ["Precision gain", "Recall gain", "F1 gain"])
plt.yticks(np.arange(len(row_labels)), row_labels)
plt.title("BLIP minus CLIP: Per-class Metric Gains")

for i in range(improve_mat.shape[0]):
    for j in range(improve_mat.shape[1]):
        plt.text(j, i, f"{improve_mat[i, j]:.2f}", ha="center", va="center")

plt.colorbar(im)
plt.tight_layout()
improve_png = os.path.join(OUTPUT_DIR, "paper_per_class_metric_gains_blip_minus_clip.png")
improve_pdf = os.path.join(OUTPUT_DIR, "paper_per_class_metric_gains_blip_minus_clip.pdf")
plt.savefig(improve_png, dpi=300, bbox_inches="tight")
plt.savefig(improve_pdf, bbox_inches="tight")
plt.show()

print("\nSaved:")
print(ci_csv)
print(class_csv)
print(norm_cm_png)
print(norm_cm_pdf)
print(ci_fig_png)
print(ci_fig_pdf)
print(improve_png)
print(improve_pdf)

最有价值的 error-case 可视化图

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

OUTPUT_DIR = "/content/drive/MyDrive/dsresearch/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

FILE_MAP = {
    ("CLIP", "sleeve"): "/content/drive/MyDrive/dsresearch/outputs/deepfashion_sleeve_clip_predictions_exp1_full.csv",
    ("CLIP", "pattern"): "/content/drive/MyDrive/dsresearch/outputs/pattern_clip_predictions_full.csv",
    ("CLIP", "material"): "/content/drive/MyDrive/dsresearch/outputs/material_clip_predictions_full.csv",

    ("BLIP", "sleeve"): "/content/drive/MyDrive/dsresearch/outputs/deepfashion_blip_predictions_full.csv",
    ("BLIP", "pattern"): "/content/drive/MyDrive/dsresearch/outputs/pattern_blip_predictions_full.csv",
    ("BLIP", "material"): "/content/drive/MyDrive/dsresearch/outputs/material_blip_predictions_full.csv",
}

TASK_ORDER = ["sleeve", "pattern", "material"]

def merge_predictions(task):
    clip_df = pd.read_csv(FILE_MAP[("CLIP", task)]).copy()
    blip_df = pd.read_csv(FILE_MAP[("BLIP", task)]).copy()

    clip_df = clip_df.rename(columns={
        "prediction": "clip_pred",
        "correct": "clip_correct"
    })
    blip_df = blip_df.rename(columns={
        "prediction": "blip_pred",
        "correct": "blip_correct"
    })

    keep_clip = ["image_path", "style_split", "ground_truth", "clip_pred", "clip_correct"]
    keep_blip = ["image_path", "blip_pred", "blip_correct"]

    merged = clip_df[keep_clip].merge(blip_df[keep_blip], on="image_path", how="inner")
    return merged

def plot_examples(task, case_name, df_case, n_show=8):
    df_case = df_case.head(n_show).copy()
    if len(df_case) == 0:
        print(f"No examples for {task} | {case_name}")
        return

    ncols = 4
    nrows = int(np.ceil(len(df_case) / ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=(16, 4 * nrows))
    axes = np.array(axes).reshape(-1)

    for ax in axes[len(df_case):]:
        ax.axis("off")

    for i, (_, row) in enumerate(df_case.iterrows()):
        ax = axes[i]
        try:
            img = Image.open(row["image_path"]).convert("RGB")
            ax.imshow(img)
            ax.set_title(
                f"GT: {row['ground_truth']}\n"
                f"CLIP: {row['clip_pred']} ({row['clip_correct']})\n"
                f"BLIP: {row['blip_pred']} ({row['blip_correct']})\n"
                f"{row['style_split']}"
            )
            ax.axis("off")
        except Exception as e:
            ax.text(0.5, 0.5, f"Load error\n{e}", ha="center", va="center")
            ax.axis("off")

    plt.tight_layout()
    out_png = os.path.join(OUTPUT_DIR, f"paper_examples_{task}_{case_name}.png")
    out_pdf = os.path.join(OUTPUT_DIR, f"paper_examples_{task}_{case_name}.pdf")
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.savefig(out_pdf, bbox_inches="tight")
    plt.show()

for task in TASK_ORDER:
    merged = merge_predictions(task)

    case_clip_wrong_blip_right = merged[
        (merged["clip_correct"] == 0) & (merged["blip_correct"] == 1)
    ].sample(n=min(8, ((merged["clip_correct"] == 0) & (merged["blip_correct"] == 1)).sum()), random_state=42)

    case_both_wrong = merged[
        (merged["clip_correct"] == 0) & (merged["blip_correct"] == 0)
    ].sample(n=min(8, ((merged["clip_correct"] == 0) & (merged["blip_correct"] == 0)).sum()), random_state=42)

    print(f"\nTask: {task}")
    print("CLIP wrong / BLIP right:", len(case_clip_wrong_blip_right))
    print("Both wrong:", len(case_both_wrong))

    plot_examples(task, "clip_wrong_blip_right", case_clip_wrong_blip_right, n_show=8)
    plot_examples(task, "both_wrong", case_both_wrong, n_show=8)

In [ ]:
import os

OUTPUT_DIR = "/content/drive/MyDrive/dsresearch/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Saving everything to:", OUTPUT_DIR)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from PIL import Image, ImageFile
import torch
from transformers import CLIPModel, CLIPProcessor

ImageFile.LOAD_TRUNCATED_IMAGES = True
Image.MAX_IMAGE_PIXELS = None

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

def run_clip_prompt_perturbation(
    task_name,
    csv_path,
    class_names,
    prompt_groups,
    sample_n=2000,
    model_id="openai/clip-vit-base-patch32",
):
    """
    prompt_groups: dict
        {
          "correct_style": [... prompts aligned with class_names ...],
          "near_miss": [...],
          "wrong_style": [...]
        }
    每个 prompt list 的顺序必须和 class_names 对齐
    """

    print(f"\n===== Prompt Perturbation | CLIP | {task_name} =====")
    df = pd.read_csv(csv_path).reset_index(drop=True)

    if sample_n is not None:
        df = df.sample(n=min(sample_n, len(df)), random_state=42).reset_index(drop=True)

    print("Rows used:", len(df))
    print(df["label"].value_counts())
    print(df["style_split"].value_counts())

    print("Loading CLIP...")
    model = CLIPModel.from_pretrained(model_id).to(device)
    processor = CLIPProcessor.from_pretrained(model_id)
    model.eval()
    print("CLIP loaded.")

    group_text_features = {}
    for group_name, prompts in prompt_groups.items():
        assert len(prompts) == len(class_names), f"{group_name} prompt count must match class count"

        text_inputs = processor(
            text=prompts,
            return_tensors="pt",
            padding=True,
            truncation=True
        )
        text_inputs = {k: v.to(device) for k, v in text_inputs.items()}

        with torch.no_grad():
            text_outputs = model.text_model(
                input_ids=text_inputs["input_ids"],
                attention_mask=text_inputs["attention_mask"]
            )
            text_features = text_outputs.pooler_output
            text_features = model.text_projection(text_features)
            text_features = text_features / text_features.norm(p=2, dim=-1, keepdim=True)

        group_text_features[group_name] = text_features

    rows = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"PromptPerturb-{task_name}"):
        try:
            image = Image.open(row["image_path"]).convert("RGB")
            image_inputs = processor(images=image, return_tensors="pt")
            image_inputs = {k: v.to(device) for k, v in image_inputs.items()}

            with torch.no_grad():
                image_outputs = model.vision_model(pixel_values=image_inputs["pixel_values"])
                image_features = image_outputs.pooler_output
                image_features = model.visual_projection(image_features)
                image_features = image_features / image_features.norm(p=2, dim=-1, keepdim=True)

            gt = row["label"]
            gt_idx = class_names.index(gt)

            record = {
                "task": task_name,
                "image_path": row["image_path"],
                "style_split": row["style_split"],
                "ground_truth": gt,
            }

            for group_name, text_features in group_text_features.items():
                logits = (image_features @ text_features.T).cpu().numpy().flatten()
                correct_score = float(logits[gt_idx])
                wrong_scores = [float(logits[i]) for i in range(len(class_names)) if i != gt_idx]
                max_wrong_score = max(wrong_scores)
                margin = correct_score - max_wrong_score

                record[f"{group_name}_correct_score"] = correct_score
                record[f"{group_name}_max_wrong_score"] = max_wrong_score
                record[f"{group_name}_margin"] = margin

            rows.append(record)

        except Exception as e:
            print("Skipping due to error:", row["image_path"], e)

    margin_df = pd.DataFrame(rows)

    raw_csv = os.path.join(OUTPUT_DIR, f"{task_name}_clip_prompt_perturbation_raw.csv")
    margin_df.to_csv(raw_csv, index=False)

    summary_rows = []
    for group_name in prompt_groups.keys():
        for subset_name, subset_df in [("overall", margin_df)] + [
            (s, margin_df[margin_df["style_split"] == s]) for s in ["catalog_like", "less_curated"]
        ]:
            if len(subset_df) == 0:
                continue

            summary_rows.append({
                "task": task_name,
                "prompt_group": group_name,
                "subset": subset_name,
                "n": len(subset_df),
                "mean_margin": subset_df[f"{group_name}_margin"].mean(),
                "median_margin": subset_df[f"{group_name}_margin"].median(),
                "std_margin": subset_df[f"{group_name}_margin"].std(),
                "mean_correct_score": subset_df[f"{group_name}_correct_score"].mean(),
                "mean_max_wrong_score": subset_df[f"{group_name}_max_wrong_score"].mean(),
            })

    summary_df = pd.DataFrame(summary_rows)
    summary_csv = os.path.join(OUTPUT_DIR, f"{task_name}_clip_prompt_perturbation_summary.csv")
    summary_df.to_csv(summary_csv, index=False)

    print("\nSummary:")
    print(summary_df)

    plt.figure(figsize=(8, 5))
    for group_name in prompt_groups.keys():
        plt.hist(
            margin_df[f"{group_name}_margin"],
            bins=40,
            alpha=0.5,
            label=group_name
        )
    plt.title(f"CLIP Prompt Margin Distribution | {task_name}")
    plt.xlabel("correct_score - max_wrong_score")
    plt.ylabel("count")
    plt.legend()
    plt.tight_layout()
    hist_png = os.path.join(OUTPUT_DIR, f"{task_name}_clip_prompt_margin_hist.png")
    hist_pdf = os.path.join(OUTPUT_DIR, f"{task_name}_clip_prompt_margin_hist.pdf")
    plt.savefig(hist_png, dpi=300, bbox_inches="tight")
    plt.savefig(hist_pdf, bbox_inches="tight")
    plt.show()

    plt.figure(figsize=(8, 5))
    box_data = [margin_df[f"{group_name}_margin"].dropna().values for group_name in prompt_groups.keys()]
    plt.boxplot(box_data, tick_labels=list(prompt_groups.keys()))
    plt.title(f"CLIP Prompt Margin Boxplot | {task_name}")
    plt.ylabel("margin")
    plt.tight_layout()
    box_png = os.path.join(OUTPUT_DIR, f"{task_name}_clip_prompt_margin_boxplot.png")
    box_pdf = os.path.join(OUTPUT_DIR, f"{task_name}_clip_prompt_margin_boxplot.pdf")
    plt.savefig(box_png, dpi=300, bbox_inches="tight")
    plt.savefig(box_pdf, bbox_inches="tight")
    plt.show()
    txt_path = os.path.join(OUTPUT_DIR, f"{task_name}_clip_prompt_perturbation_summary.txt")
    with open(txt_path, "w", encoding="utf-8") as f:
        f.write(f"CLIP Prompt Perturbation Summary | {task_name}\n\n")
        f.write(summary_df.to_string(index=False))

    print("\nSaved:")
    print(raw_csv)
    print(summary_csv)
    print(hist_png)
    print(box_png)
    print(txt_path)

    return margin_df, summary_df

 三个任务的 prompt 设置

In [ ]:
sleeve_prompt_groups = {
    "correct_style": [
        "a long-sleeve top",
        "a short-sleeve top",
        "a sleeveless top",
    ],
    "near_miss": [
        "a top with sleeves",
        "a top with medium sleeves",
        "a top without visible sleeves",
    ],
    "wrong_style": [
        "a striped garment",
        "a floral garment",
        "a denim garment",
    ],
}

In [ ]:
sleeve_prompt_groups = {
    "correct_style": [
        "a long-sleeve top",
        "a short-sleeve top",
        "a sleeveless top",
    ],
    "near_miss": [
        "a top with sleeves",
        "a top with medium sleeves",
        "a top without visible sleeves",
    ],
    "wrong_style": [
        "a striped garment",
        "a floral garment",
        "a denim garment",
    ],
}

In [ ]:
pattern_prompt_groups = {
    "correct_style": [
        "a floral garment",
        "a graphic garment",
        "a striped garment",
        "a solid garment",
    ],
    "near_miss": [
        "a garment with flower-like pattern",
        "a garment with printed design",
        "a garment with line pattern",
        "a garment with no obvious pattern",
    ],
    "wrong_style": [
        "a denim garment",
        "a sleeveless top",
        "a leather garment",
        "a long-sleeve top",
    ],
}

In [ ]:
material_prompt_groups = {
    "correct_style": [
        "a denim garment",
        "a chiffon garment",
        "a cotton garment",
        "a leather garment",
        "a knit garment",
    ],
    "near_miss": [
        "a garment made of thick woven fabric",
        "a garment made of light flowing fabric",
        "a garment made of soft natural fabric",
        "a garment made of smooth glossy material",
        "a garment made of textured woven material",
    ],
    "wrong_style": [
        "a floral garment",
        "a striped garment",
        "a sleeveless top",
        "a solid garment",
        "a graphic garment",
    ],
}

In [ ]:
sleeve_margin_df, sleeve_margin_summary = run_clip_prompt_perturbation(
    task_name="sleeve",
    csv_path="/content/deepfashion_sleeve_with_style.csv",
    class_names=["long sleeve", "short sleeve", "sleeveless"],
    prompt_groups=sleeve_prompt_groups,
    sample_n=2000
)

pattern_margin_df, pattern_margin_summary = run_clip_prompt_perturbation(
    task_name="pattern",
    csv_path="/content/drive/MyDrive/dsresearch/outputs/deepfashion_pattern_with_style.csv",
    class_names=["floral", "graphic", "striped", "solid"],
    prompt_groups=pattern_prompt_groups,
    sample_n=2000
)

material_margin_df, material_margin_summary = run_clip_prompt_perturbation(
    task_name="material",
    csv_path="/content/drive/MyDrive/dsresearch/outputs/deepfashion_material_with_style.csv",
    class_names=["denim", "chiffon", "cotton", "leather", "knit"],
    prompt_groups=material_prompt_groups,
    sample_n=2000
)

Bootstrap CI + model-difference CI

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score

OUTPUT_DIR = "/content/drive/MyDrive/dsresearch/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

FILE_MAP = {
    ("CLIP", "sleeve"): "/content/drive/MyDrive/dsresearch/outputs/deepfashion_sleeve_clip_predictions_exp1_full.csv",
    ("CLIP", "pattern"): "/content/drive/MyDrive/dsresearch/outputs/pattern_clip_predictions_full.csv",
    ("CLIP", "material"): "/content/drive/MyDrive/dsresearch/outputs/material_clip_predictions_full.csv",

    ("BLIP", "sleeve"): "/content/drive/MyDrive/dsresearch/outputs/deepfashion_blip_predictions_full.csv",
    ("BLIP", "pattern"): "/content/drive/MyDrive/dsresearch/outputs/pattern_blip_predictions_full.csv",
    ("BLIP", "material"): "/content/drive/MyDrive/dsresearch/outputs/material_blip_predictions_full.csv",
}

TASK_ORDER = ["sleeve", "pattern", "material"]
MODEL_ORDER = ["CLIP", "BLIP"]

def bootstrap_ci(y_true, y_pred, metric_fn, n_boot=1000, seed=42):
    rng = np.random.default_rng(seed)
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    n = len(y_true)

    vals = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        vals.append(metric_fn(y_true[idx], y_pred[idx]))

    vals = np.array(vals)
    return vals.mean(), np.percentile(vals, 2.5), np.percentile(vals, 97.5)

def bootstrap_diff_ci(y_true_a, y_pred_a, y_true_b, y_pred_b, metric_fn, n_boot=1000, seed=42):
    rng = np.random.default_rng(seed)
    y_true_a = np.array(y_true_a)
    y_pred_a = np.array(y_pred_a)
    y_true_b = np.array(y_true_b)
    y_pred_b = np.array(y_pred_b)

    n = min(len(y_true_a), len(y_true_b))
    vals = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        score_a = metric_fn(y_true_a[idx], y_pred_a[idx])
        score_b = metric_fn(y_true_b[idx], y_pred_b[idx])
        vals.append(score_b - score_a)  

    vals = np.array(vals)
    return vals.mean(), np.percentile(vals, 2.5), np.percentile(vals, 97.5)

rows = []
diff_rows = []

for task in TASK_ORDER:
    clip_df = pd.read_csv(FILE_MAP[("CLIP", task)])
    blip_df = pd.read_csv(FILE_MAP[("BLIP", task)])

    for model, df in [("CLIP", clip_df), ("BLIP", blip_df)]:
        y_true = df["ground_truth"].values
        y_pred = df["prediction"].values

        acc_mean, acc_lo, acc_hi = bootstrap_ci(y_true, y_pred, accuracy_score, n_boot=1000, seed=42)
        f1_mean, f1_lo, f1_hi = bootstrap_ci(
            y_true, y_pred,
            lambda yt, yp: f1_score(yt, yp, average="macro"),
            n_boot=1000,
            seed=42
        )

        rows.append({
            "task": task,
            "model": model,
            "n": len(df),
            "accuracy_mean": acc_mean,
            "accuracy_ci_lower": acc_lo,
            "accuracy_ci_upper": acc_hi,
            "macro_f1_mean": f1_mean,
            "macro_f1_ci_lower": f1_lo,
            "macro_f1_ci_upper": f1_hi,
        })

    # diff CI
    y_true_clip = clip_df["ground_truth"].values
    y_pred_clip = clip_df["prediction"].values
    y_true_blip = blip_df["ground_truth"].values
    y_pred_blip = blip_df["prediction"].values

    d_acc_mean, d_acc_lo, d_acc_hi = bootstrap_diff_ci(
        y_true_clip, y_pred_clip, y_true_blip, y_pred_blip, accuracy_score, n_boot=1000, seed=42
    )
    d_f1_mean, d_f1_lo, d_f1_hi = bootstrap_diff_ci(
        y_true_clip, y_pred_clip, y_true_blip, y_pred_blip,
        lambda yt, yp: f1_score(yt, yp, average="macro"),
        n_boot=1000,
        seed=42
    )

    diff_rows.append({
        "task": task,
        "metric": "accuracy",
        "blip_minus_clip_mean": d_acc_mean,
        "ci_lower": d_acc_lo,
        "ci_upper": d_acc_hi,
    })
    diff_rows.append({
        "task": task,
        "metric": "macro_f1",
        "blip_minus_clip_mean": d_f1_mean,
        "ci_lower": d_f1_lo,
        "ci_upper": d_f1_hi,
    })

ci_df = pd.DataFrame(rows)
diff_df = pd.DataFrame(diff_rows)

ci_csv = os.path.join(OUTPUT_DIR, "bootstrap_ci_task_model_metrics.csv")
diff_csv = os.path.join(OUTPUT_DIR, "bootstrap_ci_model_difference_metrics.csv")

ci_df.to_csv(ci_csv, index=False)
diff_df.to_csv(diff_csv, index=False)

print(ci_df)
print(diff_df)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, metric, lo_col, hi_col, title in [
    (axes[0], "accuracy_mean", "accuracy_ci_lower", "accuracy_ci_upper", "Accuracy with 95% CI"),
    (axes[1], "macro_f1_mean", "macro_f1_ci_lower", "macro_f1_ci_upper", "Macro-F1 with 95% CI"),
]:
    x_labels = []
    means = []
    err_lo = []
    err_hi = []

    for task in TASK_ORDER:
        for model in MODEL_ORDER:
            row = ci_df[(ci_df["task"] == task) & (ci_df["model"] == model)].iloc[0]
            x_labels.append(f"{task}\n{model}")
            means.append(row[metric])
            err_lo.append(row[metric] - row[lo_col])
            err_hi.append(row[hi_col] - row[metric])

    x = np.arange(len(x_labels))
    ax.bar(x, means)
    ax.errorbar(x, means, yerr=[err_lo, err_hi], fmt="none", capsize=4)
    ax.set_xticks(x)
    ax.set_xticklabels(x_labels)
    ax.set_ylim(0, 1)
    ax.set_title(title)

plt.tight_layout()
ci_png = os.path.join(OUTPUT_DIR, "bootstrap_ci_model_task_plot.png")
ci_pdf = os.path.join(OUTPUT_DIR, "bootstrap_ci_model_task_plot.pdf")
plt.savefig(ci_png, dpi=300, bbox_inches="tight")
plt.savefig(ci_pdf, bbox_inches="tight")
plt.show()

plt.figure(figsize=(8, 5))
x_labels = []
means = []
err_lo = []
err_hi = []

for task in TASK_ORDER:
    for metric in ["accuracy", "macro_f1"]:
        row = diff_df[(diff_df["task"] == task) & (diff_df["metric"] == metric)].iloc[0]
        x_labels.append(f"{task}\n{metric}")
        means.append(row["blip_minus_clip_mean"])
        err_lo.append(row["blip_minus_clip_mean"] - row["ci_lower"])
        err_hi.append(row["ci_upper"] - row["blip_minus_clip_mean"])

x = np.arange(len(x_labels))
plt.bar(x, means)
plt.errorbar(x, means, yerr=[err_lo, err_hi], fmt="none", capsize=4)
plt.axhline(0, linestyle="--")
plt.xticks(x, x_labels)
plt.title("BLIP minus CLIP with 95% Bootstrap CI")
plt.tight_layout()

diff_png = os.path.join(OUTPUT_DIR, "bootstrap_ci_blip_minus_clip_plot.png")
diff_pdf = os.path.join(OUTPUT_DIR, "bootstrap_ci_blip_minus_clip_plot.pdf")
plt.savefig(diff_png, dpi=300, bbox_inches="tight")
plt.savefig(diff_pdf, bbox_inches="tight")
plt.show()

print("Saved:")
print(ci_csv)
print(diff_csv)
print(ci_png)
print(diff_pdf)

Error-case visualization

三类图：1.CLIP错，BLIP对 2.都错 3.catalog_like对less_curated 错的困难样本切片

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

OUTPUT_DIR = "/content/drive/MyDrive/dsresearch/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

FILE_MAP = {
    ("CLIP", "sleeve"): "/content/drive/MyDrive/dsresearch/outputs/deepfashion_sleeve_clip_predictions_exp1_full.csv",
    ("CLIP", "pattern"): "/content/drive/MyDrive/dsresearch/outputs/pattern_clip_predictions_full.csv",
    ("CLIP", "material"): "/content/drive/MyDrive/dsresearch/outputs/material_clip_predictions_full.csv",

    ("BLIP", "sleeve"): "/content/drive/MyDrive/dsresearch/outputs/deepfashion_blip_predictions_full.csv",
    ("BLIP", "pattern"): "/content/drive/MyDrive/dsresearch/outputs/pattern_blip_predictions_full.csv",
    ("BLIP", "material"): "/content/drive/MyDrive/dsresearch/outputs/material_blip_predictions_full.csv",
}

TASK_ORDER = ["sleeve", "pattern", "material"]

def merge_preds(task):
    clip_df = pd.read_csv(FILE_MAP[("CLIP", task)]).copy()
    blip_df = pd.read_csv(FILE_MAP[("BLIP", task)]).copy()

    clip_df = clip_df.rename(columns={"prediction": "clip_pred", "correct": "clip_correct"})
    blip_df = blip_df.rename(columns={"prediction": "blip_pred", "correct": "blip_correct"})

    clip_keep = ["image_path", "style_split", "ground_truth", "clip_pred", "clip_correct"]
    blip_keep = ["image_path", "style_split", "ground_truth", "blip_pred", "blip_correct"]

    merged = clip_df[clip_keep].merge(
        blip_df[blip_keep],
        on=["image_path", "style_split", "ground_truth"],
        how="inner"
    )
    return merged

def plot_case_grid(df_case, task, case_name, n_show=8):
    if len(df_case) == 0:
        print(f"No samples for {task} | {case_name}")
        return

    df_case = df_case.head(n_show).copy()
    ncols = 4
    nrows = int(np.ceil(len(df_case) / ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=(16, 4 * nrows))
    axes = np.array(axes).reshape(-1)

    for ax in axes[len(df_case):]:
        ax.axis("off")

    for i, (_, row) in enumerate(df_case.iterrows()):
        ax = axes[i]
        try:
            img = Image.open(row["image_path"]).convert("RGB")
            ax.imshow(img)
            ax.set_title(
                f"GT: {row['ground_truth']}\n"
                f"CLIP: {row['clip_pred']} ({row['clip_correct']})\n"
                f"BLIP: {row['blip_pred']} ({row['blip_correct']})\n"
                f"{row['style_split']}",
                fontsize=9
            )
            ax.axis("off")
        except Exception as e:
            ax.text(0.5, 0.5, f"Load error\n{e}", ha="center", va="center")
            ax.axis("off")

    plt.tight_layout()
    out_png = os.path.join(OUTPUT_DIR, f"qual_{task}_{case_name}.png")
    out_pdf = os.path.join(OUTPUT_DIR, f"qual_{task}_{case_name}.pdf")
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.savefig(out_pdf, bbox_inches="tight")
    plt.show()

    print("Saved:", out_png)

for task in TASK_ORDER:
    merged = merge_preds(task)

    clip_wrong_blip_right = merged[
        (merged["clip_correct"] == 0) & (merged["blip_correct"] == 1)
    ]
    both_wrong = merged[
        (merged["clip_correct"] == 0) & (merged["blip_correct"] == 0)
    ]

    style_slice = merged[
        (merged["style_split"] == "less_curated") &
        (merged["clip_correct"] == 0) &
        (merged["blip_correct"] == 0)
    ]

    print(f"\nTask: {task}")
    print("CLIP wrong / BLIP right:", len(clip_wrong_blip_right))
    print("Both wrong:", len(both_wrong))
    print("Less-curated both wrong:", len(style_slice))

    if len(clip_wrong_blip_right) > 0:
        plot_case_grid(
            clip_wrong_blip_right.sample(n=min(8, len(clip_wrong_blip_right)), random_state=42),
            task,
            "clip_wrong_blip_right",
            n_show=8
        )

    if len(both_wrong) > 0:
        plot_case_grid(
            both_wrong.sample(n=min(8, len(both_wrong)), random_state=42),
            task,
            "both_wrong",
            n_show=8
        )

    if len(style_slice) > 0:
        plot_case_grid(
            style_slice.sample(n=min(8, len(style_slice)), random_state=42),
            task,
            "less_curated_both_wrong",
            n_show=8
        )

In [ ]:
import os

print("Has local images:", os.path.exists("/content/deepfashion_data/img_highres"))
print("Has pattern csv:", os.path.exists("/content/drive/MyDrive/dsresearch/outputs/deepfashion_pattern_with_style.csv"))
print("Has material csv:", os.path.exists("/content/drive/MyDrive/dsresearch/outputs/deepfashion_material_with_style.csv"))
print("Has sleeve clip preds:", os.path.exists("/content/drive/MyDrive/dsresearch/outputs/deepfashion_sleeve_clip_predictions_exp1_full.csv"))
print("Has pattern clip preds:", os.path.exists("/content/drive/MyDrive/dsresearch/outputs/pattern_clip_predictions_full.csv"))
print("Has material clip preds:", os.path.exists("/content/drive/MyDrive/dsresearch/outputs/material_clip_predictions_full.csv"))

BLIP 通用评估代码

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from PIL import Image, ImageFile
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

import torch
from transformers import BlipProcessor, BlipForImageTextRetrieval

ImageFile.LOAD_TRUNCATED_IMAGES = True
Image.MAX_IMAGE_PIXELS = None

OUTPUT_DIR = "/content/drive/MyDrive/dsresearch/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

def run_blip_task(task_name, csv_path, class_names, prompts):
    print(f"\n===== Running BLIP on {task_name} =====")
    df_test = pd.read_csv(csv_path).reset_index(drop=True)

    print("Rows:", len(df_test))
    print("\nLabel distribution:")
    print(df_test["label"].value_counts())
    print("\nStyle split distribution:")
    print(df_test["style_split"].value_counts())

    print("Loading BLIP model...")
    processor = BlipProcessor.from_pretrained("Salesforce/blip-itm-base-coco")
    model = BlipForImageTextRetrieval.from_pretrained("Salesforce/blip-itm-base-coco").to(device)
    model.eval()
    print("BLIP model loaded.")

    y_true, y_pred, rows = [], [], []

    for _, row in tqdm(df_test.iterrows(), total=len(df_test), desc=f"BLIP-{task_name}"):
        try:
            image = Image.open(row["image_path"]).convert("RGB")

            scores = []
            for prompt in prompts:
                inputs = processor(
                    images=image,
                    text=prompt,
                    return_tensors="pt",
                    padding=True,
                    truncation=True
                )
                inputs = {k: v.to(device) for k, v in inputs.items()}

                with torch.no_grad():
                    outputs = model(**inputs, use_itm_head=True)
                    score = outputs.itm_score[:, 1].item()
                scores.append(score)

            pred_idx = int(np.argmax(scores))
            pred = class_names[pred_idx]
            gt = row["label"]

            y_true.append(gt)
            y_pred.append(pred)

            rows.append({
                "image_path": row["image_path"],
                "style_split": row["style_split"],
                "ground_truth": gt,
                "prediction": pred,
                "correct": int(gt == pred),
                "model": "BLIP",
                "task": task_name,
            })

        except Exception as e:
            print("Skipping due to error:", row["image_path"], e)

    pred_df = pd.DataFrame(rows)

    pred_path = os.path.join(OUTPUT_DIR, f"{task_name}_blip_predictions_full.csv")
    pred_df.to_csv(pred_path, index=False)

    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    cm = confusion_matrix(y_true, y_pred, labels=class_names)
    clf_report = classification_report(
        y_true, y_pred,
        labels=class_names,
        target_names=class_names
    )

    print("\n===== Overall Results =====")
    print("Accuracy:", acc)
    print("Macro F1:", macro_f1)
    print("Confusion Matrix:\n", cm)
    print(clf_report)

    plt.figure(figsize=(6, 5))
    plt.imshow(cm, interpolation="nearest")
    plt.title(f"BLIP Confusion Matrix: {task_name}")
    plt.colorbar()
    tick_marks = np.arange(len(class_names))
    plt.xticks(tick_marks, class_names, rotation=45)
    plt.yticks(tick_marks, class_names)

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, str(cm[i, j]), ha="center", va="center")

    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()

    cm_png = os.path.join(OUTPUT_DIR, f"{task_name}_blip_confusion_matrix_full.png")
    cm_pdf = os.path.join(OUTPUT_DIR, f"{task_name}_blip_confusion_matrix_full.pdf")
    plt.savefig(cm_png, dpi=300, bbox_inches="tight")
    plt.savefig(cm_pdf, bbox_inches="tight")
    plt.show()

    summary_rows = [{
        "model": "BLIP",
        "task": task_name,
        "subset": "overall",
        "n": len(pred_df),
        "accuracy": acc,
        "macro_f1": macro_f1,
    }]

    print("\n===== By Style Split =====")
    for split_name in ["catalog_like", "less_curated"]:
        sub = pred_df[pred_df["style_split"] == split_name]
        if len(sub) == 0:
            continue

        sub_acc = accuracy_score(sub["ground_truth"], sub["prediction"])
        sub_macro_f1 = f1_score(sub["ground_truth"], sub["prediction"], average="macro")

        print(f"\nBLIP | {split_name}")
        print("n =", len(sub))
        print("accuracy =", sub_acc)
        print("macro_f1 =", sub_macro_f1)

        summary_rows.append({
            "model": "BLIP",
            "task": task_name,
            "subset": split_name,
            "n": len(sub),
            "accuracy": sub_acc,
            "macro_f1": sub_macro_f1,
        })

    summary_df = pd.DataFrame(summary_rows)
    summary_csv = os.path.join(OUTPUT_DIR, f"{task_name}_blip_summary_metrics_full.csv")
    summary_df.to_csv(summary_csv, index=False)

    summary_txt = os.path.join(OUTPUT_DIR, f"{task_name}_blip_summary_full.txt")
    with open(summary_txt, "w", encoding="utf-8") as f:
        f.write(f"BLIP Results | {task_name}\n\n")
        f.write(f"Accuracy: {acc}\n")
        f.write(f"Macro F1: {macro_f1}\n\n")
        f.write("Confusion Matrix:\n")
        f.write(str(cm))
        f.write("\n\nClassification Report:\n")
        f.write(clf_report)
        f.write("\n\nStyle Split Summary:\n")
        f.write(summary_df.to_string(index=False))

    print("\nSaved predictions:", pred_path)
    print("Saved summary csv:", summary_csv)
    print("Saved summary txt:", summary_txt)
    print("Saved confusion matrix:", cm_png)

    return pred_df, summary_df

In [ ]:
pattern_classes = ["floral", "graphic", "striped", "solid"]
pattern_prompts = [
    "a floral garment",
    "a graphic garment",
    "a striped garment",
    "a solid garment",
]

material_classes = ["denim", "chiffon", "cotton", "leather", "knit"]
material_prompts = [
    "a denim garment",
    "a chiffon garment",
    "a cotton garment",
    "a leather garment",
    "a knit garment",
]

pattern_blip_pred, pattern_blip_summary = run_blip_task(
    task_name="pattern",
    csv_path="/content/drive/MyDrive/dsresearch/outputs/deepfashion_pattern_with_style.csv",
    class_names=pattern_classes,
    prompts=pattern_prompts
)

material_blip_pred, material_blip_summary = run_blip_task(
    task_name="material",
    csv_path="/content/drive/MyDrive/dsresearch/outputs/deepfashion_material_with_style.csv",
    class_names=material_classes,
    prompts=material_prompts
)

. 统一汇总 CLIP vs BLIP × sleeve/pattern/material

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    confusion_matrix
)

OUTPUT_DIR = "/content/drive/MyDrive/dsresearch/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

FILE_MAP = {
    ("CLIP", "sleeve"): "/content/drive/MyDrive/dsresearch/outputs/deepfashion_sleeve_clip_predictions_exp1_full.csv",
    ("CLIP", "pattern"): "/content/drive/MyDrive/dsresearch/outputs/pattern_clip_predictions_full.csv",
    ("CLIP", "material"): "/content/drive/MyDrive/dsresearch/outputs/material_clip_predictions_full.csv",

    ("BLIP", "sleeve"): "/content/drive/MyDrive/dsresearch/outputs/deepfashion_blip_predictions_full.csv",
    ("BLIP", "pattern"): "/content/drive/MyDrive/dsresearch/outputs/pattern_blip_predictions_full.csv",
    ("BLIP", "material"): "/content/drive/MyDrive/dsresearch/outputs/material_blip_predictions_full.csv",
}

TASK_CLASSES = {
    "sleeve": ["long sleeve", "short sleeve", "sleeveless"],
    "pattern": ["floral", "graphic", "striped", "solid"],
    "material": ["denim", "chiffon", "cotton", "leather", "knit"],
}

TASK_ORDER = ["sleeve", "pattern", "material"]
MODEL_ORDER = ["CLIP", "BLIP"]

data = {}
for key, path in FILE_MAP.items():
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing file: {path}")
    data[key] = pd.read_csv(path)
    print("Loaded", key, "rows =", len(data[key]))

overall_rows = []
style_rows = []
class_rows = []
gap_rows = []
improvement_rows = []
conf_mats = {}

for model in MODEL_ORDER:
    for task in TASK_ORDER:
        df = data[(model, task)]
        classes = TASK_CLASSES[task]

        y_true = df["ground_truth"]
        y_pred = df["prediction"]

        acc = accuracy_score(y_true, y_pred)
        macro_f1 = f1_score(y_true, y_pred, average="macro")
        cm = confusion_matrix(y_true, y_pred, labels=classes)
        conf_mats[(model, task)] = cm

        overall_rows.append({
            "model": model,
            "task": task,
            "subset": "overall",
            "n": len(df),
            "accuracy": acc,
            "macro_f1": macro_f1,
        })

        p, r, f1, support = precision_recall_fscore_support(
            y_true, y_pred, labels=classes, zero_division=0
        )
        for i, cls in enumerate(classes):
            class_rows.append({
                "model": model,
                "task": task,
                "class": cls,
                "precision": p[i],
                "recall": r[i],
                "f1": f1[i],
                "support": support[i],
            })

        for split_name in ["catalog_like", "less_curated"]:
            sub = df[df["style_split"] == split_name]
            style_rows.append({
                "model": model,
                "task": task,
                "subset": split_name,
                "n": len(sub),
                "accuracy": accuracy_score(sub["ground_truth"], sub["prediction"]),
                "macro_f1": f1_score(sub["ground_truth"], sub["prediction"], average="macro"),
            })

overall_df = pd.DataFrame(overall_rows)
style_df = pd.DataFrame(style_rows)
class_df = pd.DataFrame(class_rows)

for model in MODEL_ORDER:
    for task in TASK_ORDER:
        sub = style_df[(style_df["model"] == model) & (style_df["task"] == task)].set_index("subset")
        gap_rows.append({
            "model": model,
            "task": task,
            "accuracy_gap_catalog_minus_less": sub.loc["catalog_like", "accuracy"] - sub.loc["less_curated", "accuracy"],
            "macro_f1_gap_catalog_minus_less": sub.loc["catalog_like", "macro_f1"] - sub.loc["less_curated", "macro_f1"],
        })

gap_df = pd.DataFrame(gap_rows)

for task in TASK_ORDER:
    clip_row = overall_df[(overall_df["model"] == "CLIP") & (overall_df["task"] == task)].iloc[0]
    blip_row = overall_df[(overall_df["model"] == "BLIP") & (overall_df["task"] == task)].iloc[0]

    for metric in ["accuracy", "macro_f1"]:
        improvement_rows.append({
            "task": task,
            "metric": metric,
            "clip": clip_row[metric],
            "blip": blip_row[metric],
            "absolute_improvement": blip_row[metric] - clip_row[metric],
            "relative_improvement_percent": 100 * (blip_row[metric] - clip_row[metric]) / clip_row[metric],
        })

improvement_df = pd.DataFrame(improvement_rows)

overall_df.to_csv(os.path.join(OUTPUT_DIR, "paper_overall_metrics.csv"), index=False)
style_df.to_csv(os.path.join(OUTPUT_DIR, "paper_style_metrics.csv"), index=False)
class_df.to_csv(os.path.join(OUTPUT_DIR, "paper_class_metrics.csv"), index=False)
gap_df.to_csv(os.path.join(OUTPUT_DIR, "paper_style_gap_metrics.csv"), index=False)
improvement_df.to_csv(os.path.join(OUTPUT_DIR, "paper_relative_improvement.csv"), index=False)

print("\n=== Overall Metrics ===")
print(overall_df)
print("\n=== Style Gap ===")
print(gap_df)
print("\n=== Relative Improvement ===")
print(improvement_df)

fig = plt.figure(figsize=(16, 11))

ax1 = plt.subplot(2, 2, 1)
heat_acc = np.zeros((len(TASK_ORDER), len(MODEL_ORDER)))
heat_f1 = np.zeros((len(TASK_ORDER), len(MODEL_ORDER)))
for i, task in enumerate(TASK_ORDER):
    for j, model in enumerate(MODEL_ORDER):
        row = overall_df[(overall_df["task"] == task) & (overall_df["model"] == model)].iloc[0]
        heat_acc[i, j] = row["accuracy"]
        heat_f1[i, j] = row["macro_f1"]

im = ax1.imshow(heat_acc, aspect="auto")
ax1.set_xticks(np.arange(len(MODEL_ORDER)))
ax1.set_xticklabels(MODEL_ORDER)
ax1.set_yticks(np.arange(len(TASK_ORDER)))
ax1.set_yticklabels(TASK_ORDER)
ax1.set_title("A. Overall Performance (Acc / Macro-F1)")
for i in range(len(TASK_ORDER)):
    for j in range(len(MODEL_ORDER)):
        ax1.text(j, i, f"{heat_acc[i,j]:.3f}\n{heat_f1[i,j]:.3f}", ha="center", va="center")
plt.colorbar(im, ax=ax1, fraction=0.046, pad=0.04)

ax2 = plt.subplot(2, 2, 2)
x_labels, catalog_vals, less_vals = [], [], []
for task in TASK_ORDER:
    for model in MODEL_ORDER:
        x_labels.append(f"{task}\n{model}")
        sub = style_df[(style_df["task"] == task) & (style_df["model"] == model)].set_index("subset")
        catalog_vals.append(sub.loc["catalog_like", "accuracy"])
        less_vals.append(sub.loc["less_curated", "accuracy"])

x = np.arange(len(x_labels))
ax2.scatter(x, catalog_vals, s=80, label="catalog_like")
ax2.scatter(x, less_vals, s=80, label="less_curated")
for i in range(len(x)):
    ax2.plot([x[i], x[i]], [catalog_vals[i], less_vals[i]])
ax2.set_xticks(x)
ax2.set_xticklabels(x_labels)
ax2.set_ylim(0, 1)
ax2.set_title("B. Style Robustness (Accuracy)")
ax2.legend()

ax3 = plt.subplot(2, 2, 3)
row_labels, f1_rows = [], []
for task in TASK_ORDER:
    for cls in TASK_CLASSES[task]:
        row_labels.append(f"{task}:{cls}")
        vals = []
        for model in MODEL_ORDER:
            row = class_df[(class_df["task"] == task) & (class_df["class"] == cls) & (class_df["model"] == model)].iloc[0]
            vals.append(row["f1"])
        f1_rows.append(vals)
f1_mat = np.array(f1_rows)
im2 = ax3.imshow(f1_mat, aspect="auto")
ax3.set_xticks(np.arange(len(MODEL_ORDER)))
ax3.set_xticklabels(MODEL_ORDER)
ax3.set_yticks(np.arange(len(row_labels)))
ax3.set_yticklabels(row_labels)
ax3.set_title("C. Class-wise F1 Heatmap")
for i in range(f1_mat.shape[0]):
    for j in range(f1_mat.shape[1]):
        ax3.text(j, i, f"{f1_mat[i,j]:.2f}", ha="center", va="center")
plt.colorbar(im2, ax=ax3, fraction=0.046, pad=0.04)

ax4 = plt.subplot(2, 2, 4)
acc_imps, f1_imps = [], []
for task in TASK_ORDER:
    acc_imps.append(improvement_df[(improvement_df["task"] == task) & (improvement_df["metric"] == "accuracy")]["absolute_improvement"].iloc[0])
    f1_imps.append(improvement_df[(improvement_df["task"] == task) & (improvement_df["metric"] == "macro_f1")]["absolute_improvement"].iloc[0])

base_x = np.arange(len(TASK_ORDER))
w = 0.35
ax4.bar(base_x - w/2, acc_imps, width=w, label="Accuracy gain")
ax4.bar(base_x + w/2, f1_imps, width=w, label="Macro-F1 gain")
ax4.axhline(0, linestyle="--")
ax4.set_xticks(base_x)
ax4.set_xticklabels(TASK_ORDER)
ax4.set_title("D. BLIP Improvement over CLIP")
ax4.legend()

plt.tight_layout()
main_png = os.path.join(OUTPUT_DIR, "paper_main_multitask_comparison.png")
main_pdf = os.path.join(OUTPUT_DIR, "paper_main_multitask_comparison.pdf")
plt.savefig(main_png, dpi=300, bbox_inches="tight")
plt.savefig(main_pdf, bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(3, 2, figsize=(14, 18))
for i, task in enumerate(TASK_ORDER):
    classes = TASK_CLASSES[task]
    for j, model in enumerate(MODEL_ORDER):
        ax = axes[i, j]
        cm = conf_mats[(model, task)]
        ax.imshow(cm, interpolation="nearest")
        ax.set_title(f"{task} | {model}")
        ax.set_xticks(np.arange(len(classes)))
        ax.set_yticks(np.arange(len(classes)))
        ax.set_xticklabels(classes, rotation=45)
        ax.set_yticklabels(classes)
        ax.set_xlabel("Predicted")
        ax.set_ylabel("True")
        for r in range(cm.shape[0]):
            for c in range(cm.shape[1]):
                ax.text(c, r, str(cm[r, c]), ha="center", va="center")
plt.tight_layout()
cm_png = os.path.join(OUTPUT_DIR, "paper_confusion_matrices_3tasks_2models.png")
cm_pdf = os.path.join(OUTPUT_DIR, "paper_confusion_matrices_3tasks_2models.pdf")
plt.savefig(cm_png, dpi=300, bbox_inches="tight")
plt.savefig(cm_pdf, bbox_inches="tight")
plt.show()

print("\nSaved main outputs to Drive:")
print(main_png)
print(main_pdf)
print(cm_png)
print(cm_pdf)

prompt perturbation sanity check

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from PIL import Image, ImageFile
import torch
from transformers import CLIPModel, CLIPProcessor

ImageFile.LOAD_TRUNCATED_IMAGES = True
Image.MAX_IMAGE_PIXELS = None

OUTPUT_DIR = "/content/drive/MyDrive/dsresearch/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"

def run_clip_prompt_perturbation(task_name, csv_path, class_names, prompt_groups, sample_n=2000):
    print(f"\n===== Prompt Perturbation | {task_name} =====")
    df = pd.read_csv(csv_path).reset_index(drop=True)
    df = df.sample(n=min(sample_n, len(df)), random_state=42).reset_index(drop=True)

    model_id = "openai/clip-vit-base-patch32"
    model = CLIPModel.from_pretrained(model_id).to(device)
    processor = CLIPProcessor.from_pretrained(model_id)
    model.eval()

    group_text_features = {}
    for group_name, prompts in prompt_groups.items():
        text_inputs = processor(text=prompts, return_tensors="pt", padding=True, truncation=True)
        text_inputs = {k: v.to(device) for k, v in text_inputs.items()}
        with torch.no_grad():
            text_outputs = model.text_model(
                input_ids=text_inputs["input_ids"],
                attention_mask=text_inputs["attention_mask"]
            )
            text_features = text_outputs.pooler_output
            text_features = model.text_projection(text_features)
            text_features = text_features / text_features.norm(p=2, dim=-1, keepdim=True)
        group_text_features[group_name] = text_features

    rows = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"PromptPerturb-{task_name}"):
        try:
            image = Image.open(row["image_path"]).convert("RGB")
            image_inputs = processor(images=image, return_tensors="pt")
            image_inputs = {k: v.to(device) for k, v in image_inputs.items()}

            with torch.no_grad():
                image_outputs = model.vision_model(pixel_values=image_inputs["pixel_values"])
                image_features = image_outputs.pooler_output
                image_features = model.visual_projection(image_features)
                image_features = image_features / image_features.norm(p=2, dim=-1, keepdim=True)

            gt = row["label"]
            gt_idx = class_names.index(gt)

            rec = {
                "task": task_name,
                "image_path": row["image_path"],
                "style_split": row["style_split"],
                "ground_truth": gt,
            }

            for group_name, text_features in group_text_features.items():
                logits = (image_features @ text_features.T).cpu().numpy().flatten()
                correct_score = float(logits[gt_idx])
                wrong_scores = [float(logits[i]) for i in range(len(class_names)) if i != gt_idx]
                max_wrong_score = max(wrong_scores)
                margin = correct_score - max_wrong_score

                rec[f"{group_name}_correct_score"] = correct_score
                rec[f"{group_name}_max_wrong_score"] = max_wrong_score
                rec[f"{group_name}_margin"] = margin

            rows.append(rec)
        except Exception as e:
            print("Skipping due to error:", row["image_path"], e)

    margin_df = pd.DataFrame(rows)
    raw_csv = os.path.join(OUTPUT_DIR, f"{task_name}_clip_prompt_perturbation_raw.csv")
    margin_df.to_csv(raw_csv, index=False)

    summary_rows = []
    for group_name in prompt_groups.keys():
        for subset_name, subset_df in [("overall", margin_df)] + [
            (s, margin_df[margin_df["style_split"] == s]) for s in ["catalog_like", "less_curated"]
        ]:
            if len(subset_df) == 0:
                continue

            summary_rows.append({
                "task": task_name,
                "prompt_group": group_name,
                "subset": subset_name,
                "n": len(subset_df),
                "mean_margin": subset_df[f"{group_name}_margin"].mean(),
                "median_margin": subset_df[f"{group_name}_margin"].median(),
                "std_margin": subset_df[f"{group_name}_margin"].std(),
            })

    summary_df = pd.DataFrame(summary_rows)
    summary_csv = os.path.join(OUTPUT_DIR, f"{task_name}_clip_prompt_perturbation_summary.csv")
    summary_df.to_csv(summary_csv, index=False)

    plt.figure(figsize=(8, 5))
    for group_name in prompt_groups.keys():
        plt.hist(margin_df[f"{group_name}_margin"], bins=40, alpha=0.5, label=group_name)
    plt.title(f"CLIP Prompt Margin Distribution | {task_name}")
    plt.xlabel("correct_score - max_wrong_score")
    plt.ylabel("count")
    plt.legend()
    plt.tight_layout()
    hist_png = os.path.join(OUTPUT_DIR, f"{task_name}_clip_prompt_margin_hist.png")
    hist_pdf = os.path.join(OUTPUT_DIR, f"{task_name}_clip_prompt_margin_hist.pdf")
    plt.savefig(hist_png, dpi=300, bbox_inches="tight")
    plt.savefig(hist_pdf, bbox_inches="tight")
    plt.show()

    print(summary_df)
    print("Saved:", raw_csv)
    print("Saved:", summary_csv)

    return margin_df, summary_df

In [ ]:
sleeve_prompt_groups = {
    "correct_style": [
        "a long-sleeve top",
        "a short-sleeve top",
        "a sleeveless top",
    ],
    "near_miss": [
        "a top with sleeves",
        "a top with medium sleeves",
        "a top without visible sleeves",
    ],
    "wrong_style": [
        "a striped garment",
        "a floral garment",
        "a denim garment",
    ],
}

pattern_prompt_groups = {
    "correct_style": [
        "a floral garment",
        "a graphic garment",
        "a striped garment",
        "a solid garment",
    ],
    "near_miss": [
        "a garment with flower-like pattern",
        "a garment with printed design",
        "a garment with line pattern",
        "a garment with no obvious pattern",
    ],
    "wrong_style": [
        "a denim garment",
        "a sleeveless top",
        "a leather garment",
        "a long-sleeve top",
    ],
}

material_prompt_groups = {
    "correct_style": [
        "a denim garment",
        "a chiffon garment",
        "a cotton garment",
        "a leather garment",
        "a knit garment",
    ],
    "near_miss": [
        "a garment made of thick woven fabric",
        "a garment made of light flowing fabric",
        "a garment made of soft natural fabric",
        "a garment made of smooth glossy material",
        "a garment made of textured woven material",
    ],
    "wrong_style": [
        "a floral garment",
        "a striped garment",
        "a sleeveless top",
        "a solid garment",
        "a graphic garment",
    ],
}

sleeve_margin_df, sleeve_margin_summary = run_clip_prompt_perturbation(
    "sleeve",
    "/content/deepfashion_sleeve_with_style.csv",
    ["long sleeve", "short sleeve", "sleeveless"],
    sleeve_prompt_groups,
    sample_n=2000
)

pattern_margin_df, pattern_margin_summary = run_clip_prompt_perturbation(
    "pattern",
    "/content/drive/MyDrive/dsresearch/outputs/deepfashion_pattern_with_style.csv",
    ["floral", "graphic", "striped", "solid"],
    pattern_prompt_groups,
    sample_n=2000
)

material_margin_df, material_margin_summary = run_clip_prompt_perturbation(
    "material",
    "/content/drive/MyDrive/dsresearch/outputs/deepfashion_material_with_style.csv",
    ["denim", "chiffon", "cotton", "leather", "knit"],
    material_prompt_groups,
    sample_n=2000
)

统一读取结果并构建总表

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support, confusion_matrix

OUTPUT_DIR = "/content/drive/MyDrive/dsresearch/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

FILE_MAP = {
    ("CLIP", "sleeve"): "/content/drive/MyDrive/dsresearch/outputs/deepfashion_sleeve_clip_predictions_exp1_full.csv",
    ("CLIP", "pattern"): "/content/drive/MyDrive/dsresearch/outputs/pattern_clip_predictions_full.csv",
    ("CLIP", "material"): "/content/drive/MyDrive/dsresearch/outputs/material_clip_predictions_full.csv",

    ("BLIP", "sleeve"): "/content/drive/MyDrive/dsresearch/outputs/deepfashion_blip_predictions_full.csv",
    ("BLIP", "pattern"): "/content/drive/MyDrive/dsresearch/outputs/pattern_blip_predictions_full.csv",
    ("BLIP", "material"): "/content/drive/MyDrive/dsresearch/outputs/material_blip_predictions_full.csv",
}

TASK_CLASSES = {
    "sleeve": ["long sleeve", "short sleeve", "sleeveless"],
    "pattern": ["floral", "graphic", "striped", "solid"],
    "material": ["denim", "chiffon", "cotton", "leather", "knit"],
}

TASK_ORDER = ["sleeve", "pattern", "material"]
MODEL_ORDER = ["CLIP", "BLIP"]

data = {}
for key, path in FILE_MAP.items():
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing file: {path}")
    data[key] = pd.read_csv(path)
    print("Loaded", key, "rows =", len(data[key]))

overall_rows = []
style_rows = []
class_rows = []
cm_store = {}

for model in MODEL_ORDER:
    for task in TASK_ORDER:
        df = data[(model, task)]
        classes = TASK_CLASSES[task]

        y_true = df["ground_truth"]
        y_pred = df["prediction"]

        acc = accuracy_score(y_true, y_pred)
        macro_f1 = f1_score(y_true, y_pred, average="macro")
        cm = confusion_matrix(y_true, y_pred, labels=classes)
        cm_store[(model, task)] = cm

        overall_rows.append({
            "model": model,
            "task": task,
            "subset": "overall",
            "n": len(df),
            "accuracy": acc,
            "macro_f1": macro_f1,
        })

        p, r, f1, support = precision_recall_fscore_support(
            y_true, y_pred, labels=classes, zero_division=0
        )

        for i, cls in enumerate(classes):
            class_rows.append({
                "model": model,
                "task": task,
                "class": cls,
                "precision": p[i],
                "recall": r[i],
                "f1": f1[i],
                "support": support[i],
            })

        for split_name in ["catalog_like", "less_curated"]:
            sub = df[df["style_split"] == split_name]
            style_rows.append({
                "model": model,
                "task": task,
                "subset": split_name,
                "n": len(sub),
                "accuracy": accuracy_score(sub["ground_truth"], sub["prediction"]),
                "macro_f1": f1_score(sub["ground_truth"], sub["prediction"], average="macro"),
            })

overall_df = pd.DataFrame(overall_rows)
style_df = pd.DataFrame(style_rows)
class_df = pd.DataFrame(class_rows)

overall_df.to_csv(os.path.join(OUTPUT_DIR, "paper_overall_metrics.csv"), index=False)
style_df.to_csv(os.path.join(OUTPUT_DIR, "paper_style_metrics.csv"), index=False)
class_df.to_csv(os.path.join(OUTPUT_DIR, "paper_class_metrics.csv"), index=False)

print("\nOverall")
print(overall_df)
print("\nStyle")
print(style_df)
print("\nClass")
print(class_df.head())

Bootstrap 95% CI + BLIP-CLIP 差值 CI

In [ ]:
import numpy as np
import pandas as pd
import os
from sklearn.metrics import accuracy_score, f1_score

OUTPUT_DIR = "/content/drive/MyDrive/dsresearch/outputs"

def bootstrap_ci(y_true, y_pred, metric_fn, n_boot=1000, seed=42):
    rng = np.random.default_rng(seed)
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    n = len(y_true)

    vals = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        vals.append(metric_fn(y_true[idx], y_pred[idx]))
    vals = np.array(vals)

    return vals.mean(), np.percentile(vals, 2.5), np.percentile(vals, 97.5)

def bootstrap_diff_ci(df_a, df_b, metric_fn, n_boot=1000, seed=42):
    rng = np.random.default_rng(seed)

    merged = df_a[["image_path", "ground_truth", "prediction"]].rename(columns={"prediction": "pred_a"}).merge(
        df_b[["image_path", "ground_truth", "prediction"]].rename(columns={"prediction": "pred_b"}),
        on=["image_path", "ground_truth"],
        how="inner"
    )

    y_true = merged["ground_truth"].to_numpy()
    pred_a = merged["pred_a"].to_numpy()
    pred_b = merged["pred_b"].to_numpy()
    n = len(merged)

    vals = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        score_a = metric_fn(y_true[idx], pred_a[idx])
        score_b = metric_fn(y_true[idx], pred_b[idx])
        vals.append(score_b - score_a)

    vals = np.array(vals)
    return vals.mean(), np.percentile(vals, 2.5), np.percentile(vals, 97.5)

ci_rows = []
diff_rows = []

for task in TASK_ORDER:
    clip_df = data[("CLIP", task)]
    blip_df = data[("BLIP", task)]

    for model, df in [("CLIP", clip_df), ("BLIP", blip_df)]:
        y_true = df["ground_truth"].to_numpy()
        y_pred = df["prediction"].to_numpy()

        acc_mean, acc_lo, acc_hi = bootstrap_ci(y_true, y_pred, accuracy_score)
        f1_mean, f1_lo, f1_hi = bootstrap_ci(
            y_true, y_pred,
            lambda yt, yp: f1_score(yt, yp, average="macro")
        )

        ci_rows.append({
            "task": task,
            "model": model,
            "n": len(df),
            "accuracy_mean": acc_mean,
            "accuracy_ci_lower": acc_lo,
            "accuracy_ci_upper": acc_hi,
            "macro_f1_mean": f1_mean,
            "macro_f1_ci_lower": f1_lo,
            "macro_f1_ci_upper": f1_hi,
        })

    d_acc_mean, d_acc_lo, d_acc_hi = bootstrap_diff_ci(
        clip_df, blip_df, accuracy_score
    )
    d_f1_mean, d_f1_lo, d_f1_hi = bootstrap_diff_ci(
        clip_df, blip_df,
        lambda yt, yp: f1_score(yt, yp, average="macro")
    )

    diff_rows.append({
        "task": task,
        "metric": "accuracy",
        "blip_minus_clip_mean": d_acc_mean,
        "ci_lower": d_acc_lo,
        "ci_upper": d_acc_hi,
    })
    diff_rows.append({
        "task": task,
        "metric": "macro_f1",
        "blip_minus_clip_mean": d_f1_mean,
        "ci_lower": d_f1_lo,
        "ci_upper": d_f1_hi,
    })

ci_df = pd.DataFrame(ci_rows)
diff_df = pd.DataFrame(diff_rows)

ci_df.to_csv(os.path.join(OUTPUT_DIR, "paper_bootstrap_ci_metrics.csv"), index=False)
diff_df.to_csv(os.path.join(OUTPUT_DIR, "paper_bootstrap_difference_metrics.csv"), index=False)

print(ci_df)
print(diff_df)

McNemar 显著性检验

In [ ]:
import os
import pandas as pd
from statsmodels.stats.contingency_tables import mcnemar

OUTPUT_DIR = "/content/drive/MyDrive/dsresearch/outputs"

mcnemar_rows = []

for task in TASK_ORDER:
    clip_df = data[("CLIP", task)]
    blip_df = data[("BLIP", task)]

    merged = clip_df[["image_path", "ground_truth", "correct"]].rename(columns={"correct": "clip_correct"}).merge(
        blip_df[["image_path", "ground_truth", "correct"]].rename(columns={"correct": "blip_correct"}),
        on=["image_path", "ground_truth"],
        how="inner"
    )

    a = ((merged["clip_correct"] == 1) & (merged["blip_correct"] == 1)).sum()
    b = ((merged["clip_correct"] == 1) & (merged["blip_correct"] == 0)).sum()
    c = ((merged["clip_correct"] == 0) & (merged["blip_correct"] == 1)).sum()
    d = ((merged["clip_correct"] == 0) & (merged["blip_correct"] == 0)).sum()

    table = [[a, b], [c, d]]
    result = mcnemar(table, exact=False, correction=True)

    mcnemar_rows.append({
        "task": task,
        "both_correct": a,
        "clip_only_correct": b,
        "blip_only_correct": c,
        "both_wrong": d,
        "statistic": result.statistic,
        "p_value": result.pvalue,
    })

mcnemar_df = pd.DataFrame(mcnemar_rows)
mcnemar_df.to_csv(os.path.join(OUTPUT_DIR, "paper_mcnemar_results.csv"), index=False)
print(mcnemar_df)

Normalized confusion matrices

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

OUTPUT_DIR = "/content/drive/MyDrive/dsresearch/outputs"

def normalize_confusion(cm):
    row_sums = cm.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1
    return cm / row_sums

fig, axes = plt.subplots(3, 2, figsize=(14, 18))

for i, task in enumerate(TASK_ORDER):
    classes = TASK_CLASSES[task]
    for j, model in enumerate(MODEL_ORDER):
        ax = axes[i, j]
        cm = cm_store[(model, task)]
        ncm = normalize_confusion(cm)

        im = ax.imshow(ncm, vmin=0, vmax=1, interpolation="nearest")
        ax.set_title(f"{task} | {model}")
        ax.set_xticks(np.arange(len(classes)))
        ax.set_yticks(np.arange(len(classes)))
        ax.set_xticklabels(classes, rotation=45)
        ax.set_yticklabels(classes)
        ax.set_xlabel("Predicted")
        ax.set_ylabel("True")

        for r in range(ncm.shape[0]):
            for c in range(ncm.shape[1]):
                ax.text(c, r, f"{ncm[r, c]:.2f}", ha="center", va="center")

plt.tight_layout()
png_path = os.path.join(OUTPUT_DIR, "paper_normalized_confusion_matrices.png")
pdf_path = os.path.join(OUTPUT_DIR, "paper_normalized_confusion_matrices.pdf")
plt.savefig(png_path, dpi=300, bbox_inches="tight")
plt.savefig(pdf_path, bbox_inches="tight")
plt.show()

print(png_path)
print(pdf_path)

Per-class gain heatmap

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

OUTPUT_DIR = "/content/drive/MyDrive/dsresearch/outputs"

row_labels = []
gain_rows = []

for task in TASK_ORDER:
    for cls in TASK_CLASSES[task]:
        clip_row = class_df[
            (class_df["task"] == task) &
            (class_df["class"] == cls) &
            (class_df["model"] == "CLIP")
        ].iloc[0]

        blip_row = class_df[
            (class_df["task"] == task) &
            (class_df["class"] == cls) &
            (class_df["model"] == "BLIP")
        ].iloc[0]

        row_labels.append(f"{task}:{cls}")
        gain_rows.append([
            blip_row["precision"] - clip_row["precision"],
            blip_row["recall"] - clip_row["recall"],
            blip_row["f1"] - clip_row["f1"],
        ])

gain_mat = np.array(gain_rows)

plt.figure(figsize=(8, 10))
im = plt.imshow(gain_mat, aspect="auto")
plt.xticks(np.arange(3), ["Precision gain", "Recall gain", "F1 gain"])
plt.yticks(np.arange(len(row_labels)), row_labels)
plt.title("BLIP minus CLIP: Per-class Metric Gains")

for i in range(gain_mat.shape[0]):
    for j in range(gain_mat.shape[1]):
        plt.text(j, i, f"{gain_mat[i, j]:.2f}", ha="center", va="center")

plt.colorbar(im)
plt.tight_layout()

png_path = os.path.join(OUTPUT_DIR, "paper_per_class_metric_gains_blip_minus_clip.png")
pdf_path = os.path.join(OUTPUT_DIR, "paper_per_class_metric_gains_blip_minus_clip.pdf")
plt.savefig(png_path, dpi=300, bbox_inches="tight")
plt.savefig(pdf_path, bbox_inches="tight")
plt.show()

print(png_path)
print(pdf_path)

Style-gap 分析图

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

OUTPUT_DIR = "/content/drive/MyDrive/dsresearch/outputs"

gap_rows = []
for model in MODEL_ORDER:
    for task in TASK_ORDER:
        sub = style_df[(style_df["model"] == model) & (style_df["task"] == task)].set_index("subset")
        gap_rows.append({
            "model": model,
            "task": task,
            "accuracy_gap_catalog_minus_less": sub.loc["catalog_like", "accuracy"] - sub.loc["less_curated", "accuracy"],
            "macro_f1_gap_catalog_minus_less": sub.loc["catalog_like", "macro_f1"] - sub.loc["less_curated", "macro_f1"],
        })

gap_df = pd.DataFrame(gap_rows)
gap_df.to_csv(os.path.join(OUTPUT_DIR, "paper_style_gap_metrics.csv"), index=False)
print(gap_df)

plt.figure(figsize=(10, 5))
x_labels, acc_gap_vals, f1_gap_vals = [], [], []

for task in TASK_ORDER:
    for model in MODEL_ORDER:
        row = gap_df[(gap_df["task"] == task) & (gap_df["model"] == model)].iloc[0]
        x_labels.append(f"{task}\n{model}")
        acc_gap_vals.append(row["accuracy_gap_catalog_minus_less"])
        f1_gap_vals.append(row["macro_f1_gap_catalog_minus_less"])

x = np.arange(len(x_labels))
w = 0.35

plt.bar(x - w/2, acc_gap_vals, width=w, label="Accuracy gap")
plt.bar(x + w/2, f1_gap_vals, width=w, label="Macro-F1 gap")
plt.axhline(0, linestyle="--")
plt.xticks(x, x_labels)
plt.title("Style Gap: catalog_like minus less_curated")
plt.legend()
plt.tight_layout()

png_path = os.path.join(OUTPUT_DIR, "paper_style_gap_comparison.png")
pdf_path = os.path.join(OUTPUT_DIR, "paper_style_gap_comparison.pdf")
plt.savefig(png_path, dpi=300, bbox_inches="tight")
plt.savefig(pdf_path, bbox_inches="tight")
plt.show()

print(png_path)
print(pdf_path)

Qualitative error-case visualization

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

OUTPUT_DIR = "/content/drive/MyDrive/dsresearch/outputs"

def merge_preds(task):
    clip_df = data[("CLIP", task)].copy()
    blip_df = data[("BLIP", task)].copy()

    clip_df = clip_df.rename(columns={"prediction": "clip_pred", "correct": "clip_correct"})
    blip_df = blip_df.rename(columns={"prediction": "blip_pred", "correct": "blip_correct"})

    merged = clip_df[["image_path", "style_split", "ground_truth", "clip_pred", "clip_correct"]].merge(
        blip_df[["image_path", "style_split", "ground_truth", "blip_pred", "blip_correct"]],
        on=["image_path", "style_split", "ground_truth"],
        how="inner"
    )
    return merged

def plot_case_grid(df_case, task, case_name, n_show=8):
    if len(df_case) == 0:
        print(f"No samples for {task} | {case_name}")
        return

    df_case = df_case.head(n_show).copy()
    ncols = 4
    nrows = int(np.ceil(len(df_case) / ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=(16, 4 * nrows))
    axes = np.array(axes).reshape(-1)

    for ax in axes[len(df_case):]:
        ax.axis("off")

    for i, (_, row) in enumerate(df_case.iterrows()):
        ax = axes[i]
        try:
            img = Image.open(row["image_path"]).convert("RGB")
            ax.imshow(img)
            ax.set_title(
                f"GT: {row['ground_truth']}\n"
                f"CLIP: {row['clip_pred']} ({row['clip_correct']})\n"
                f"BLIP: {row['blip_pred']} ({row['blip_correct']})\n"
                f"{row['style_split']}",
                fontsize=9
            )
            ax.axis("off")
        except Exception as e:
            ax.text(0.5, 0.5, f"Load error\n{e}", ha="center", va="center")
            ax.axis("off")

    plt.tight_layout()
    png_path = os.path.join(OUTPUT_DIR, f"qual_{task}_{case_name}.png")
    pdf_path = os.path.join(OUTPUT_DIR, f"qual_{task}_{case_name}.pdf")
    plt.savefig(png_path, dpi=300, bbox_inches="tight")
    plt.savefig(pdf_path, bbox_inches="tight")
    plt.show()

    print(png_path)
    print(pdf_path)

for task in TASK_ORDER:
    merged = merge_preds(task)

    case1 = merged[(merged["clip_correct"] == 0) & (merged["blip_correct"] == 1)]
    case2 = merged[(merged["clip_correct"] == 0) & (merged["blip_correct"] == 0)]
    case3 = merged[(merged["style_split"] == "less_curated") & (merged["clip_correct"] == 0) & (merged["blip_correct"] == 0)]

    print(f"\nTask: {task}")
    print("CLIP wrong, BLIP right:", len(case1))
    print("Both wrong:", len(case2))
    print("Less-curated both wrong:", len(case3))

    if len(case1) > 0:
        plot_case_grid(case1.sample(n=min(8, len(case1)), random_state=42), task, "clip_wrong_blip_right")
    if len(case2) > 0:
        plot_case_grid(case2.sample(n=min(8, len(case2)), random_state=42), task, "both_wrong")
    if len(case3) > 0:
        plot_case_grid(case3.sample(n=min(8, len(case3)), random_state=42), task, "less_curated_both_wrong")

Prompt ensembling for CLIP


CLIP 的结果是不是太依赖单一句 prompt wording？

	每个类别不用一句 prompt
	每个类别用 3 到 5 句 prompt
	把同一类别的 text embedding 取平均再做分类


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from PIL import Image, ImageFile
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

import torch
from transformers import CLIPModel, CLIPProcessor

ImageFile.LOAD_TRUNCATED_IMAGES = True
Image.MAX_IMAGE_PIXELS = None

OUTPUT_DIR = "/content/drive/MyDrive/dsresearch/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

def run_clip_prompt_ensemble(task_name, csv_path, class_names, prompt_ensemble_dict):
    """
    prompt_ensemble_dict:
      {
        "class1": [prompt1, prompt2, ...],
        "class2": [prompt1, prompt2, ...],
      }
    keys must match class_names exactly
    """

    print(f"\n===== Running CLIP Prompt Ensemble on {task_name} =====")
    df_test = pd.read_csv(csv_path).reset_index(drop=True)

    print("Rows:", len(df_test))
    print(df_test["label"].value_counts())
    print(df_test["style_split"].value_counts())

    model_id = "openai/clip-vit-base-patch32"
    model = CLIPModel.from_pretrained(model_id).to(device)
    processor = CLIPProcessor.from_pretrained(model_id)
    model.eval()

    class_text_features = []

    for cls in class_names:
        prompts = prompt_ensemble_dict[cls]

        text_inputs = processor(
            text=prompts,
            return_tensors="pt",
            padding=True,
            truncation=True
        )
        text_inputs = {k: v.to(device) for k, v in text_inputs.items()}

        with torch.no_grad():
            text_outputs = model.text_model(
                input_ids=text_inputs["input_ids"],
                attention_mask=text_inputs["attention_mask"]
            )
            text_features = text_outputs.pooler_output
            text_features = model.text_projection(text_features)
            text_features = text_features / text_features.norm(p=2, dim=-1, keepdim=True)

        mean_feature = text_features.mean(dim=0, keepdim=True)
        mean_feature = mean_feature / mean_feature.norm(p=2, dim=-1, keepdim=True)
        class_text_features.append(mean_feature)

    text_feature_matrix = torch.cat(class_text_features, dim=0) 

    y_true, y_pred, rows = [], [], []

    for _, row in tqdm(df_test.iterrows(), total=len(df_test), desc=f"CLIP-Ensemble-{task_name}"):
        try:
            image = Image.open(row["image_path"]).convert("RGB")
            image_inputs = processor(images=image, return_tensors="pt")
            image_inputs = {k: v.to(device) for k, v in image_inputs.items()}

            with torch.no_grad():
                image_outputs = model.vision_model(pixel_values=image_inputs["pixel_values"])
                image_features = image_outputs.pooler_output
                image_features = model.visual_projection(image_features)
                image_features = image_features / image_features.norm(p=2, dim=-1, keepdim=True)

                logits = image_features @ text_feature_matrix.T
                pred_idx = int(logits.argmax(dim=1).cpu().item())

            pred = class_names[pred_idx]
            gt = row["label"]

            y_true.append(gt)
            y_pred.append(pred)

            rows.append({
                "image_path": row["image_path"],
                "style_split": row["style_split"],
                "ground_truth": gt,
                "prediction": pred,
                "correct": int(gt == pred),
                "model": "CLIP_ENSEMBLE",
                "task": task_name,
            })

        except Exception as e:
            print("Skipping due to error:", row["image_path"], e)

    pred_df = pd.DataFrame(rows)

    pred_csv = os.path.join(OUTPUT_DIR, f"{task_name}_clip_prompt_ensemble_predictions.csv")
    pred_df.to_csv(pred_csv, index=False)

    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    cm = confusion_matrix(y_true, y_pred, labels=class_names)
    clf_report = classification_report(y_true, y_pred, labels=class_names, target_names=class_names)

    print("\n===== Overall Results =====")
    print("Accuracy:", acc)
    print("Macro F1:", macro_f1)
    print("Confusion Matrix:\n", cm)
    print(clf_report)

    summary_rows = [{
        "model": "CLIP_ENSEMBLE",
        "task": task_name,
        "subset": "overall",
        "n": len(pred_df),
        "accuracy": acc,
        "macro_f1": macro_f1,
    }]

    for split_name in ["catalog_like", "less_curated"]:
        sub = pred_df[pred_df["style_split"] == split_name]
        if len(sub) == 0:
            continue

        sub_acc = accuracy_score(sub["ground_truth"], sub["prediction"])
        sub_macro_f1 = f1_score(sub["ground_truth"], sub["prediction"], average="macro")

        print(f"\n{split_name}")
        print("n =", len(sub))
        print("accuracy =", sub_acc)
        print("macro_f1 =", sub_macro_f1)

        summary_rows.append({
            "model": "CLIP_ENSEMBLE",
            "task": task_name,
            "subset": split_name,
            "n": len(sub),
            "accuracy": sub_acc,
            "macro_f1": sub_macro_f1,
        })

    summary_df = pd.DataFrame(summary_rows)
    summary_csv = os.path.join(OUTPUT_DIR, f"{task_name}_clip_prompt_ensemble_summary.csv")
    summary_df.to_csv(summary_csv, index=False)

    cm_png = os.path.join(OUTPUT_DIR, f"{task_name}_clip_prompt_ensemble_confusion_matrix.png")
    plt.figure(figsize=(6, 5))
    plt.imshow(cm, interpolation="nearest")
    plt.title(f"CLIP Prompt Ensemble Confusion Matrix: {task_name}")
    plt.colorbar()
    tick_marks = np.arange(len(class_names))
    plt.xticks(tick_marks, class_names, rotation=45)
    plt.yticks(tick_marks, class_names)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    plt.savefig(cm_png, dpi=300, bbox_inches="tight")
    plt.show()

    print("\nSaved:")
    print(pred_csv)
    print(summary_csv)
    print(cm_png)

    return pred_df, summary_df

Prompt ensemble 配置并运行
只跑 sleeve + pattern

In [ ]:
sleeve_prompt_ensemble = {
    "long sleeve": [
        "a long-sleeve top",
        "a top with long sleeves",
        "a garment with long sleeves",
        "a fashion product photo of a long-sleeve top",
    ],
    "short sleeve": [
        "a short-sleeve top",
        "a top with short sleeves",
        "a garment with short sleeves",
        "a fashion product photo of a short-sleeve top",
    ],
    "sleeveless": [
        "a sleeveless top",
        "a top without sleeves",
        "a sleeveless garment",
        "a fashion product photo of a sleeveless top",
    ],
}

pattern_prompt_ensemble = {
    "floral": [
        "a floral garment",
        "a garment with floral pattern",
        "a flower-patterned garment",
        "a fashion product photo of a floral garment",
    ],
    "graphic": [
        "a graphic garment",
        "a garment with graphic print",
        "a printed graphic garment",
        "a fashion product photo of a graphic garment",
    ],
    "striped": [
        "a striped garment",
        "a garment with stripes",
        "a line-patterned garment",
        "a fashion product photo of a striped garment",
    ],
    "solid": [
        "a solid garment",
        "a garment with no obvious pattern",
        "a plain solid garment",
        "a fashion product photo of a solid garment",
    ],
}

sleeve_ens_pred, sleeve_ens_summary = run_clip_prompt_ensemble(
    task_name="sleeve",
    csv_path="/content/deepfashion_sleeve_with_style.csv",
    class_names=["long sleeve", "short sleeve", "sleeveless"],
    prompt_ensemble_dict=sleeve_prompt_ensemble
)

pattern_ens_pred, pattern_ens_summary = run_clip_prompt_ensemble(
    task_name="pattern",
    csv_path="/content/drive/MyDrive/dsresearch/outputs/deepfashion_pattern_with_style.csv",
    class_names=["floral", "graphic", "striped", "solid"],
    prompt_ensemble_dict=pattern_prompt_ensemble
)

BLIP prompt perturbation


BLIP 更强是不是也体现在 prompt 语言稳健上？

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from PIL import Image, ImageFile

import torch
from transformers import BlipProcessor, BlipForImageTextRetrieval

ImageFile.LOAD_TRUNCATED_IMAGES = True
Image.MAX_IMAGE_PIXELS = None

OUTPUT_DIR = "/content/drive/MyDrive/dsresearch/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"

def run_blip_prompt_perturbation(task_name, csv_path, class_names, prompt_groups, sample_n=1000):
    print(f"\n===== BLIP Prompt Perturbation | {task_name} =====")
    df = pd.read_csv(csv_path).reset_index(drop=True)
    df = df.sample(n=min(sample_n, len(df)), random_state=42).reset_index(drop=True)

    print("Rows used:", len(df))

    processor = BlipProcessor.from_pretrained("Salesforce/blip-itm-base-coco")
    model = BlipForImageTextRetrieval.from_pretrained("Salesforce/blip-itm-base-coco").to(device)
    model.eval()

    rows = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"BLIP-PromptPerturb-{task_name}"):
        try:
            image = Image.open(row["image_path"]).convert("RGB")
            gt = row["label"]
            gt_idx = class_names.index(gt)

            rec = {
                "task": task_name,
                "image_path": row["image_path"],
                "style_split": row["style_split"],
                "ground_truth": gt,
            }

            for group_name, prompts in prompt_groups.items():
                scores = []
                for prompt in prompts:
                    inputs = processor(
                        images=image,
                        text=prompt,
                        return_tensors="pt",
                        padding=True,
                        truncation=True
                    )
                    inputs = {k: v.to(device) for k, v in inputs.items()}

                    with torch.no_grad():
                        outputs = model(**inputs, use_itm_head=True)
                        score = outputs.itm_score[:, 1].item()
                    scores.append(score)

                correct_score = float(scores[gt_idx])
                wrong_scores = [float(scores[i]) for i in range(len(class_names)) if i != gt_idx]
                max_wrong_score = max(wrong_scores)
                margin = correct_score - max_wrong_score

                rec[f"{group_name}_correct_score"] = correct_score
                rec[f"{group_name}_max_wrong_score"] = max_wrong_score
                rec[f"{group_name}_margin"] = margin

            rows.append(rec)

        except Exception as e:
            print("Skipping due to error:", row["image_path"], e)

    margin_df = pd.DataFrame(rows)
    raw_csv = os.path.join(OUTPUT_DIR, f"{task_name}_blip_prompt_perturbation_raw.csv")
    margin_df.to_csv(raw_csv, index=False)

    summary_rows = []
    for group_name in prompt_groups.keys():
        for subset_name, subset_df in [("overall", margin_df)] + [
            (s, margin_df[margin_df["style_split"] == s]) for s in ["catalog_like", "less_curated"]
        ]:
            if len(subset_df) == 0:
                continue

            summary_rows.append({
                "task": task_name,
                "prompt_group": group_name,
                "subset": subset_name,
                "n": len(subset_df),
                "mean_margin": subset_df[f"{group_name}_margin"].mean(),
                "median_margin": subset_df[f"{group_name}_margin"].median(),
                "std_margin": subset_df[f"{group_name}_margin"].std(),
            })

    summary_df = pd.DataFrame(summary_rows)
    summary_csv = os.path.join(OUTPUT_DIR, f"{task_name}_blip_prompt_perturbation_summary.csv")
    summary_df.to_csv(summary_csv, index=False)

    plt.figure(figsize=(8, 5))
    for group_name in prompt_groups.keys():
        plt.hist(margin_df[f"{group_name}_margin"], bins=40, alpha=0.5, label=group_name)
    plt.title(f"BLIP Prompt Margin Distribution | {task_name}")
    plt.xlabel("correct_score - max_wrong_score")
    plt.ylabel("count")
    plt.legend()
    plt.tight_layout()

    png_path = os.path.join(OUTPUT_DIR, f"{task_name}_blip_prompt_margin_hist.png")
    pdf_path = os.path.join(OUTPUT_DIR, f"{task_name}_blip_prompt_margin_hist.pdf")
    plt.savefig(png_path, dpi=300, bbox_inches="tight")
    plt.savefig(pdf_path, bbox_inches="tight")
    plt.show()

    print(summary_df)
    print(raw_csv)
    print(summary_csv)

    return margin_df, summary_df

Class × style split 度统计

	哪个类在 less_curated 掉最多
	哪个类对 style shift 最敏感
	BLIP 是否缓解了某些特定类的 style sensitivity

In [ ]:
import os
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support

OUTPUT_DIR = "/content/drive/MyDrive/dsresearch/outputs"

def class_style_breakdown(df, model_name, task_name, class_names):
    rows = []
    for split_name in ["catalog_like", "less_curated"]:
        sub = df[df["style_split"] == split_name]
        y_true = sub["ground_truth"]
        y_pred = sub["prediction"]

        p, r, f1, support = precision_recall_fscore_support(
            y_true, y_pred, labels=class_names, zero_division=0
        )

        for i, cls in enumerate(class_names):
            rows.append({
                "model": model_name,
                "task": task_name,
                "style_split": split_name,
                "class": cls,
                "precision": p[i],
                "recall": r[i],
                "f1": f1[i],
                "support": support[i],
            })
    return pd.DataFrame(rows)

fine_rows = []

for task in TASK_ORDER:
    classes = TASK_CLASSES[task]
    fine_rows.append(class_style_breakdown(data[("CLIP", task)], "CLIP", task, classes))
    fine_rows.append(class_style_breakdown(data[("BLIP", task)], "BLIP", task, classes))

fine_df = pd.concat(fine_rows, ignore_index=True)
fine_csv = os.path.join(OUTPUT_DIR, "paper_class_style_breakdown.csv")
fine_df.to_csv(fine_csv, index=False)

print(fine_df.head(20))
print("Saved:", fine_csv)

Appendix figure layout

In [ ]:
import os
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

OUTPUT_DIR = "/content/drive/MyDrive/dsresearch/outputs"

figure_paths = [
    os.path.join(OUTPUT_DIR, "paper_main_multitask_comparison.png"),
    os.path.join(OUTPUT_DIR, "paper_confusion_matrices_3tasks_2models.png"),
    os.path.join(OUTPUT_DIR, "paper_normalized_confusion_matrices.png"),
    os.path.join(OUTPUT_DIR, "paper_per_class_metric_gains_blip_minus_clip.png"),
    os.path.join(OUTPUT_DIR, "paper_style_gap_comparison.png"),
]

existing = [p for p in figure_paths if os.path.exists(p)]
print("Existing figures:", existing)

n = len(existing)
ncols = 2
nrows = (n + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(16, 5 * nrows))
axes = axes.flatten()

for ax in axes[n:]:
    ax.axis("off")

for i, path in enumerate(existing):
    img = mpimg.imread(path)
    axes[i].imshow(img)
    axes[i].set_title(os.path.basename(path))
    axes[i].axis("off")

plt.tight_layout()
png_path = os.path.join(OUTPUT_DIR, "appendix_figure_overview.png")
pdf_path = os.path.join(OUTPUT_DIR, "appendix_figure_overview.pdf")
plt.savefig(png_path, dpi=300, bbox_inches="tight")
plt.savefig(pdf_path, bbox_inches="tight")
plt.show()

print(png_path)
print(pdf_path)

统一配置

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from PIL import Image, ImageFile

import torch
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from transformers import CLIPModel, CLIPProcessor, BlipProcessor, BlipForImageTextRetrieval

ImageFile.LOAD_TRUNCATED_IMAGES = True
Image.MAX_IMAGE_PIXELS = None

OUTPUT_DIR = "/content/drive/MyDrive/dsresearch/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

TASK_INFO = {
    "sleeve": {
        "csv": "/content/deepfashion_sleeve_with_style.csv",
        "classes": ["long sleeve", "short sleeve", "sleeveless"],
    },
    "pattern": {
        "csv": "/content/drive/MyDrive/dsresearch/outputs/deepfashion_pattern_with_style.csv",
        "classes": ["floral", "graphic", "striped", "solid"],
    },
    "material": {
        "csv": "/content/drive/MyDrive/dsresearch/outputs/deepfashion_material_with_style.csv",
        "classes": ["denim", "chiffon", "cotton", "leather", "knit"],
    },
}

全新Prompt

Prompt Families
	descriptive_single
	studio_single
	lifestyle_single
	attribute_only_single
	technical_single

Prompt Ensembles
每类一组多 prompt，后面平均


In [ ]:
SLEEVE_PROMPT_BANK = {
    "descriptive_single": {
        "long sleeve": "upper-body clothing whose sleeves extend to the wrists",
        "short sleeve": "upper-body clothing whose sleeves stop above the elbow",
        "sleeveless": "upper-body clothing with the arms fully uncovered",
    },
    "studio_single": {
        "long sleeve": "studio product shot of a top with full-length sleeves",
        "short sleeve": "studio product shot of a top with abbreviated sleeves",
        "sleeveless": "studio product shot of a top with no sleeves at all",
    },
    "lifestyle_single": {
        "long sleeve": "casual outfit featuring a top with sleeves down to the wrists",
        "short sleeve": "casual outfit featuring a top with sleeves ending high on the arm",
        "sleeveless": "casual outfit featuring a top that leaves the shoulders and arms bare",
    },
    "attribute_only_single": {
        "long sleeve": "full-length sleeves",
        "short sleeve": "brief sleeves",
        "sleeveless": "no sleeves",
    },
    "technical_single": {
        "long sleeve": "garment category defined by continuous sleeve coverage to wrist level",
        "short sleeve": "garment category defined by partial sleeve coverage above elbow level",
        "sleeveless": "garment category defined by complete absence of sleeves",
    },
    "ensemble": {
        "long sleeve": [
            "upper-body clothing whose sleeves extend to the wrists",
            "top with full arm coverage",
            "shirt with sleeves reaching the wrist area",
            "studio merchandise image of a fully sleeved upper garment",
        ],
        "short sleeve": [
            "upper-body clothing whose sleeves stop above the elbow",
            "top with abbreviated arm coverage",
            "shirt with sleeves ending around the upper arm",
            "studio merchandise image of a briefly sleeved upper garment",
        ],
        "sleeveless": [
            "upper-body clothing with the arms fully uncovered",
            "top with no sleeve attachment at all",
            "shirt leaving both shoulders and arms exposed",
            "studio merchandise image of a sleeveless upper garment",
        ],
    }
}

Pattern Prompt Bank

In [ ]:
PATTERN_PROMPT_BANK = {
    "descriptive_single": {
        "floral": "clothing surface covered with flower-like motifs",
        "graphic": "clothing surface dominated by printed visual artwork or symbols",
        "striped": "clothing surface organized by repeated parallel bands",
        "solid": "clothing surface with no distinct repeated motif",
    },
    "studio_single": {
        "floral": "studio product image of apparel with botanical print",
        "graphic": "studio product image of apparel with bold printed artwork",
        "striped": "studio product image of apparel with repeated line bands",
        "solid": "studio product image of apparel with plain uninterrupted surface",
    },
    "lifestyle_single": {
        "floral": "street-style clothing featuring a flower-inspired print",
        "graphic": "street-style clothing featuring a strong printed visual design",
        "striped": "street-style clothing featuring repeated stripe-like bands",
        "solid": "street-style clothing featuring a plain unpatterned surface",
    },
    "attribute_only_single": {
        "floral": "botanical print",
        "graphic": "printed artwork",
        "striped": "parallel bands",
        "solid": "plain surface",
    },
    "technical_single": {
        "floral": "textile appearance characterized by repeated botanical motifs",
        "graphic": "textile appearance characterized by high-level printed symbols or artwork",
        "striped": "textile appearance characterized by repeated linear orientation",
        "solid": "textile appearance characterized by visually uniform fill without motif repetition",
    },
    "ensemble": {
        "floral": [
            "clothing surface covered with flower-like motifs",
            "apparel decorated by botanical print elements",
            "fabric showing repeated blossom-style shapes",
            "studio merchandise image of a botanically patterned item",
        ],
        "graphic": [
            "clothing surface dominated by printed visual artwork or symbols",
            "apparel with prominent printed design elements",
            "fabric carrying large graphic motifs",
            "studio merchandise image of a visually printed statement item",
        ],
        "striped": [
            "clothing surface organized by repeated parallel bands",
            "apparel with repeated stripe-like lines",
            "fabric showing clear linear band repetition",
            "studio merchandise image of an item with repeated stripe structure",
        ],
        "solid": [
            "clothing surface with no distinct repeated motif",
            "apparel with visually plain uninterrupted fill",
            "fabric showing uniform color with no obvious pattern",
            "studio merchandise image of a plain unpatterned item",
        ],
    }
}

Material Prompt Bank

In [ ]:
MATERIAL_PROMPT_BANK = {
    "descriptive_single": {
        "denim": "apparel made from heavy blue twill-like cloth",
        "chiffon": "apparel made from sheer airy lightweight fabric",
        "cotton": "apparel made from soft matte natural fiber fabric",
        "leather": "apparel made from smooth dense hide-like material",
        "knit": "apparel made from looped textured yarn structure",
    },
    "studio_single": {
        "denim": "studio product image of clothing made from sturdy twill-like fabric",
        "chiffon": "studio product image of clothing made from sheer flowing material",
        "cotton": "studio product image of clothing made from soft natural fabric",
        "leather": "studio product image of clothing made from smooth hide-like material",
        "knit": "studio product image of clothing made from textured looped yarn fabric",
    },
    "lifestyle_single": {
        "denim": "casual outfit featuring rugged twill-like fabric",
        "chiffon": "casual outfit featuring light sheer draping fabric",
        "cotton": "casual outfit featuring soft breathable natural fabric",
        "leather": "casual outfit featuring glossy dense hide-like material",
        "knit": "casual outfit featuring textured sweater-like yarn construction",
    },
    "attribute_only_single": {
        "denim": "heavy twill cloth",
        "chiffon": "sheer draping fabric",
        "cotton": "soft natural fabric",
        "leather": "dense hide-like material",
        "knit": "looped yarn texture",
    },
    "technical_single": {
        "denim": "textile class defined by robust twill-based weave and structured surface",
        "chiffon": "textile class defined by translucency, drape, and low fabric density",
        "cotton": "textile class defined by matte natural-fiber surface with soft hand feel",
        "leather": "material class defined by dense smooth hide-derived surface",
        "knit": "textile class defined by looped yarn construction rather than flat weave",
    },
    "ensemble": {
        "denim": [
            "apparel made from heavy blue twill-like cloth",
            "garment with rugged jean-like fabric texture",
            "clothing constructed from sturdy twill material",
            "studio merchandise image of a twill-based structured fabric item",
        ],
        "chiffon": [
            "apparel made from sheer airy lightweight fabric",
            "garment with translucent flowing cloth",
            "clothing constructed from delicate draping material",
            "studio merchandise image of a light semi-transparent fabric item",
        ],
        "cotton": [
            "apparel made from soft matte natural fiber fabric",
            "garment with breathable natural-fabric surface",
            "clothing constructed from soft everyday fiber cloth",
            "studio merchandise image of a matte soft-fabric item",
        ],
        "leather": [
            "apparel made from smooth dense hide-like material",
            "garment with firm glossy hide-textured surface",
            "clothing constructed from dense leather-like material",
            "studio merchandise image of a polished hide-like item",
        ],
        "knit": [
            "apparel made from looped textured yarn structure",
            "garment with sweater-like interlocking yarn texture",
            "clothing constructed from knitted textile structure",
            "studio merchandise image of a loop-textured yarn item",
        ],
    }
}

通用 CLIP Prompt Family Sweep

比较：
	descriptive_single
	studio_single
	lifestyle_single
	attribute_only_single
	technical_single

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from PIL import Image, ImageFile

import torch
from sklearn.metrics import accuracy_score, f1_score
from transformers import CLIPModel, CLIPProcessor

ImageFile.LOAD_TRUNCATED_IMAGES = True
Image.MAX_IMAGE_PIXELS = None

OUTPUT_DIR = "/content/drive/MyDrive/dsresearch/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

TASK_INFO = {
    "sleeve": {
        "csv": "/content/deepfashion_sleeve_with_style.csv",
        "classes": ["long sleeve", "short sleeve", "sleeveless"],
    },
    "pattern": {
        "csv": "/content/drive/MyDrive/dsresearch/outputs/deepfashion_pattern_with_style.csv",
        "classes": ["floral", "graphic", "striped", "solid"],
    },
    "material": {
        "csv": "/content/drive/MyDrive/dsresearch/outputs/deepfashion_material_with_style.csv",
        "classes": ["denim", "chiffon", "cotton", "leather", "knit"],
    },
}

print("TASK_INFO loaded.")
print(TASK_INFO)

In [ ]:
def run_clip_prompt_family_sweep(task_name, prompt_bank):
    info = TASK_INFO[task_name]
    csv_path = info["csv"]
    class_names = info["classes"]

    df_test = pd.read_csv(csv_path).reset_index(drop=True)
    print(f"\n===== CLIP Prompt Family Sweep | {task_name} =====")
    print("Rows:", len(df_test))

    model_id = "openai/clip-vit-base-patch32"
    model = CLIPModel.from_pretrained(model_id).to(device)
    processor = CLIPProcessor.from_pretrained(model_id)
    model.eval()

    family_rows = []

    for family_name, prompt_spec in prompt_bank.items():
        if family_name == "ensemble":
            continue

        prompts = [prompt_spec[c] for c in class_names]
        print("\nRunning family:", family_name)
        for p in prompts:
            print("-", p)

        text_inputs = processor(
            text=prompts,
            return_tensors="pt",
            padding=True,
            truncation=True
        )
        text_inputs = {k: v.to(device) for k, v in text_inputs.items()}

        with torch.no_grad():
            text_outputs = model.text_model(
                input_ids=text_inputs["input_ids"],
                attention_mask=text_inputs["attention_mask"]
            )
            text_features = text_outputs.pooler_output
            text_features = model.text_projection(text_features)
            text_features = text_features / text_features.norm(p=2, dim=-1, keepdim=True)

        y_true, y_pred, rows = [], [], []

        for _, row in tqdm(df_test.iterrows(), total=len(df_test), desc=f"{task_name}-{family_name}"):
            try:
                image = Image.open(row["image_path"]).convert("RGB")
                image_inputs = processor(images=image, return_tensors="pt")
                image_inputs = {k: v.to(device) for k, v in image_inputs.items()}

                with torch.no_grad():
                    image_outputs = model.vision_model(pixel_values=image_inputs["pixel_values"])
                    image_features = image_outputs.pooler_output
                    image_features = model.visual_projection(image_features)
                    image_features = image_features / image_features.norm(p=2, dim=-1, keepdim=True)

                    logits = image_features @ text_features.T
                    pred_idx = int(logits.argmax(dim=1).cpu().item())

                pred = class_names[pred_idx]
                gt = row["label"]
                y_true.append(gt)
                y_pred.append(pred)

                rows.append({
                    "image_path": row["image_path"],
                    "style_split": row["style_split"],
                    "ground_truth": gt,
                    "prediction": pred,
                    "correct": int(gt == pred),
                    "model": "CLIP",
                    "task": task_name,
                    "prompt_family": family_name,
                })
            except Exception as e:
                print("Skipping due to error:", row["image_path"], e)

        pred_df = pd.DataFrame(rows)
        pred_path = os.path.join(OUTPUT_DIR, f"{task_name}_clip_{family_name}_predictions.csv")
        pred_df.to_csv(pred_path, index=False)

        acc = accuracy_score(y_true, y_pred)
        macro_f1 = f1_score(y_true, y_pred, average="macro")

        family_rows.append({
            "task": task_name,
            "model": "CLIP",
            "prompt_family": family_name,
            "subset": "overall",
            "n": len(pred_df),
            "accuracy": acc,
            "macro_f1": macro_f1,
        })

        for split_name in ["catalog_like", "less_curated"]:
            sub = pred_df[pred_df["style_split"] == split_name]
            family_rows.append({
                "task": task_name,
                "model": "CLIP",
                "prompt_family": family_name,
                "subset": split_name,
                "n": len(sub),
                "accuracy": accuracy_score(sub["ground_truth"], sub["prediction"]),
                "macro_f1": f1_score(sub["ground_truth"], sub["prediction"], average="macro"),
            })

    summary_df = pd.DataFrame(family_rows)
    summary_csv = os.path.join(OUTPUT_DIR, f"{task_name}_clip_prompt_family_sweep_summary.csv")
    summary_df.to_csv(summary_csv, index=False)

    print(summary_df)

    overall_df = summary_df[summary_df["subset"] == "overall"].copy()
    plt.figure(figsize=(9, 5))
    x = np.arange(len(overall_df))
    plt.bar(x - 0.15, overall_df["accuracy"], width=0.3, label="Accuracy")
    plt.bar(x + 0.15, overall_df["macro_f1"], width=0.3, label="Macro F1")
    plt.xticks(x, overall_df["prompt_family"], rotation=30)
    plt.ylim(0, 1)
    plt.title(f"CLIP Prompt Family Sweep | {task_name}")
    plt.legend()
    plt.tight_layout()

    png_path = os.path.join(OUTPUT_DIR, f"{task_name}_clip_prompt_family_sweep.png")
    pdf_path = os.path.join(OUTPUT_DIR, f"{task_name}_clip_prompt_family_sweep.pdf")
    plt.savefig(png_path, dpi=300, bbox_inches="tight")
    plt.savefig(pdf_path, bbox_inches="tight")
    plt.show()

    return summary_df

In [ ]:
TASK_INFO = {
    "sleeve": {
        "csv": "/content/drive/MyDrive/dsresearch/outputs/deepfashion_sleeve_with_style.csv",
        "classes": ["long sleeve", "short sleeve", "sleeveless"],
    },
    "pattern": {
        "csv": "/content/drive/MyDrive/dsresearch/outputs/deepfashion_pattern_with_style.csv",
        "classes": ["floral", "graphic", "striped", "solid"],
    },
    "material": {
        "csv": "/content/drive/MyDrive/dsresearch/outputs/deepfashion_material_with_style.csv",
        "classes": ["denim", "chiffon", "cotton", "leather", "knit"],
    },
}
print(TASK_INFO)

In [ ]:
sleeve_clip_family_summary = run_clip_prompt_family_sweep("sleeve", SLEEVE_PROMPT_BANK)
pattern_clip_family_summary = run_clip_prompt_family_sweep("pattern", PATTERN_PROMPT_BANK)
material_clip_family_summary = run_clip_prompt_family_sweep("material", MATERIAL_PROMPT_BANK)

CLIP Prompt Ensemble Sweep

专门跑ensemble，和single prompt结果区分开。

In [ ]:
def run_clip_prompt_ensemble_single(task_name, prompt_bank):
    info = TASK_INFO[task_name]
    csv_path = info["csv"]
    class_names = info["classes"]

    df_test = pd.read_csv(csv_path).reset_index(drop=True)
    print(f"\n===== CLIP Prompt Ensemble | {task_name} =====")
    print("Rows:", len(df_test))

    model_id = "openai/clip-vit-base-patch32"
    model = CLIPModel.from_pretrained(model_id).to(device)
    processor = CLIPProcessor.from_pretrained(model_id)
    model.eval()

    ensemble_prompts = prompt_bank["ensemble"]

    class_text_features = []
    for cls in class_names:
        prompts = ensemble_prompts[cls]

        text_inputs = processor(
            text=prompts,
            return_tensors="pt",
            padding=True,
            truncation=True
        )
        text_inputs = {k: v.to(device) for k, v in text_inputs.items()}

        with torch.no_grad():
            text_outputs = model.text_model(
                input_ids=text_inputs["input_ids"],
                attention_mask=text_inputs["attention_mask"]
            )
            text_features = text_outputs.pooler_output
            text_features = model.text_projection(text_features)
            text_features = text_features / text_features.norm(p=2, dim=-1, keepdim=True)

        mean_feature = text_features.mean(dim=0, keepdim=True)
        mean_feature = mean_feature / mean_feature.norm(p=2, dim=-1, keepdim=True)
        class_text_features.append(mean_feature)

    text_feature_matrix = torch.cat(class_text_features, dim=0)

    y_true, y_pred, rows = [], [], []

    for _, row in tqdm(df_test.iterrows(), total=len(df_test), desc=f"{task_name}-ensemble"):
        try:
            image = Image.open(row["image_path"]).convert("RGB")
            image_inputs = processor(images=image, return_tensors="pt")
            image_inputs = {k: v.to(device) for k, v in image_inputs.items()}

            with torch.no_grad():
                image_outputs = model.vision_model(pixel_values=image_inputs["pixel_values"])
                image_features = image_outputs.pooler_output
                image_features = model.visual_projection(image_features)
                image_features = image_features / image_features.norm(p=2, dim=-1, keepdim=True)

                logits = image_features @ text_feature_matrix.T
                pred_idx = int(logits.argmax(dim=1).cpu().item())

            pred = class_names[pred_idx]
            gt = row["label"]

            y_true.append(gt)
            y_pred.append(pred)
            rows.append({
                "image_path": row["image_path"],
                "style_split": row["style_split"],
                "ground_truth": gt,
                "prediction": pred,
                "correct": int(gt == pred),
                "model": "CLIP_ENSEMBLE",
                "task": task_name,
            })
        except Exception as e:
            print("Skipping due to error:", row["image_path"], e)

    pred_df = pd.DataFrame(rows)
    pred_csv = os.path.join(OUTPUT_DIR, f"{task_name}_clip_ensemble_predictions.csv")
    pred_df.to_csv(pred_csv, index=False)

    rows_out = [{
        "task": task_name,
        "model": "CLIP_ENSEMBLE",
        "subset": "overall",
        "n": len(pred_df),
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
    }]
    for split_name in ["catalog_like", "less_curated"]:
        sub = pred_df[pred_df["style_split"] == split_name]
        rows_out.append({
            "task": task_name,
            "model": "CLIP_ENSEMBLE",
            "subset": split_name,
            "n": len(sub),
            "accuracy": accuracy_score(sub["ground_truth"], sub["prediction"]),
            "macro_f1": f1_score(sub["ground_truth"], sub["prediction"], average="macro"),
        })

    summary_df = pd.DataFrame(rows_out)
    summary_csv = os.path.join(OUTPUT_DIR, f"{task_name}_clip_ensemble_summary.csv")
    summary_df.to_csv(summary_csv, index=False)

    print(summary_df)
    return pred_df, summary_df

In [ ]:
lsleeve_clip_ens_pred, sleeve_clip_ens_summary = run_clip_prompt_ensemble_single("sleeve", SLEEVE_PROMPT_BANK)
pattern_clip_ens_pred, pattern_clip_ens_summary = run_clip_prompt_ensemble_single("pattern", PATTERN_PROMPT_BANK)
material_clip_ens_pred, material_clip_ens_summary = run_clip_prompt_ensemble_single("material", MATERIAL_PROMPT_BANK)

BLIP Prompt Family Sweep对比。

In [ ]:
def run_blip_prompt_family_sweep(task_name, prompt_bank, families=None):
    info = TASK_INFO[task_name]
    csv_path = info["csv"]
    class_names = info["classes"]

    df_test = pd.read_csv(csv_path).reset_index(drop=True)
    print(f"\n===== BLIP Prompt Family Sweep | {task_name} =====")
    print("Rows:", len(df_test))

    processor = BlipProcessor.from_pretrained("Salesforce/blip-itm-base-coco")
    model = BlipForImageTextRetrieval.from_pretrained("Salesforce/blip-itm-base-coco").to(device)
    model.eval()

    all_families = [k for k in prompt_bank.keys() if k != "ensemble"]
    if families is None:
        families = all_families

    family_rows = []

    for family_name in families:
        prompt_spec = prompt_bank[family_name]
        prompts = [prompt_spec[c] for c in class_names]

        print("\nRunning BLIP family:", family_name)
        y_true, y_pred, rows = [], [], []

        for _, row in tqdm(df_test.iterrows(), total=len(df_test), desc=f"{task_name}-{family_name}"):
            try:
                image = Image.open(row["image_path"]).convert("RGB")

                scores = []
                for prompt in prompts:
                    inputs = processor(
                        images=image,
                        text=prompt,
                        return_tensors="pt",
                        padding=True,
                        truncation=True
                    )
                    inputs = {k: v.to(device) for k, v in inputs.items()}

                    with torch.no_grad():
                        outputs = model(**inputs, use_itm_head=True)
                        score = outputs.itm_score[:, 1].item()
                    scores.append(score)

                pred_idx = int(np.argmax(scores))
                pred = class_names[pred_idx]
                gt = row["label"]

                y_true.append(gt)
                y_pred.append(pred)

                rows.append({
                    "image_path": row["image_path"],
                    "style_split": row["style_split"],
                    "ground_truth": gt,
                    "prediction": pred,
                    "correct": int(gt == pred),
                    "model": "BLIP",
                    "task": task_name,
                    "prompt_family": family_name,
                })
            except Exception as e:
                print("Skipping due to error:", row["image_path"], e)

        pred_df = pd.DataFrame(rows)
        pred_csv = os.path.join(OUTPUT_DIR, f"{task_name}_blip_{family_name}_predictions.csv")
        pred_df.to_csv(pred_csv, index=False)

        family_rows.append({
            "task": task_name,
            "model": "BLIP",
            "prompt_family": family_name,
            "subset": "overall",
            "n": len(pred_df),
            "accuracy": accuracy_score(y_true, y_pred),
            "macro_f1": f1_score(y_true, y_pred, average="macro"),
        })

        for split_name in ["catalog_like", "less_curated"]:
            sub = pred_df[pred_df["style_split"] == split_name]
            family_rows.append({
                "task": task_name,
                "model": "BLIP",
                "prompt_family": family_name,
                "subset": split_name,
                "n": len(sub),
                "accuracy": accuracy_score(sub["ground_truth"], sub["prediction"]),
                "macro_f1": f1_score(sub["ground_truth"], sub["prediction"], average="macro"),
            })

    summary_df = pd.DataFrame(family_rows)
    summary_csv = os.path.join(OUTPUT_DIR, f"{task_name}_blip_prompt_family_sweep_summary.csv")
    summary_df.to_csv(summary_csv, index=False)

    print(summary_df)
    return summary_df

In [ ]:
sleeve_blip_family_summary = run_blip_prompt_family_sweep(
    "sleeve",
    SLEEVE_PROMPT_BANK,
    families=["descriptive_single", "studio_single", "lifestyle_single", "technical_single"]
)

pattern_blip_family_summary = run_blip_prompt_family_sweep(
    "pattern",
    PATTERN_PROMPT_BANK,
)

Class  Style Split统计
哪个类在less_curated掉最多
BLIP是否缓解了某一类的style sensitivity

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

def class_style_breakdown(df, model_name, task_name, class_names):
    rows = []
    for split_name in ["catalog_like", "less_curated"]:
        sub = df[df["style_split"] == split_name]
        y_true = sub["ground_truth"]
        y_pred = sub["prediction"]

        p, r, f1, support = precision_recall_fscore_support(
            y_true, y_pred, labels=class_names, zero_division=0
        )

        for i, cls in enumerate(class_names):
            rows.append({
                "model": model_name,
                "task": task_name,
                "style_split": split_name,
                "class": cls,
                "precision": p[i],
                "recall": r[i],
                "f1": f1[i],
                "support": support[i],
            })
    return pd.DataFrame(rows)

fine_rows = []

BASE_FILE_MAP = {
    ("CLIP", "sleeve"): "/content/drive/MyDrive/dsresearch/outputs/deepfashion_sleeve_clip_predictions_exp1_full.csv",
    ("CLIP", "pattern"): "/content/drive/MyDrive/dsresearch/outputs/pattern_clip_predictions_full.csv",
    ("CLIP", "material"): "/content/drive/MyDrive/dsresearch/outputs/material_clip_predictions_full.csv",

    ("BLIP", "sleeve"): "/content/drive/MyDrive/dsresearch/outputs/deepfashion_blip_predictions_full.csv",
    ("BLIP", "pattern"): "/content/drive/MyDrive/dsresearch/outputs/pattern_blip_predictions_full.csv",
    ("BLIP", "material"): "/content/drive/MyDrive/dsresearch/outputs/material_blip_predictions_full.csv",
}

for task in ["sleeve", "pattern", "material"]:
    classes = TASK_INFO[task]["classes"]
    for model in ["CLIP", "BLIP"]:
        df_pred = pd.read_csv(BASE_FILE_MAP[(model, task)])
        fine_rows.append(class_style_breakdown(df_pred, model, task, classes))

fine_df = pd.concat(fine_rows, ignore_index=True)
fine_csv = os.path.join(OUTPUT_DIR, "paper_class_style_breakdown.csv")
fine_df.to_csv(fine_csv, index=False)

print(fine_df.head(20))
print("Saved:", fine_csv)

Appendix figure overview

整理已经生成图。

In [ ]:
import matplotlib.image as mpimg

figure_paths = [
    os.path.join(OUTPUT_DIR, "paper_main_multitask_comparison.png"),
    os.path.join(OUTPUT_DIR, "paper_confusion_matrices_3tasks_2models.png"),
    os.path.join(OUTPUT_DIR, "paper_normalized_confusion_matrices.png"),
    os.path.join(OUTPUT_DIR, "paper_per_class_metric_gains_blip_minus_clip.png"),
    os.path.join(OUTPUT_DIR, "paper_style_gap_comparison.png"),
]

existing = [p for p in figure_paths if os.path.exists(p)]
print("Existing figures:", existing)

n = len(existing)
ncols = 2
nrows = (n + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(16, 5 * nrows))
axes = np.array(axes).reshape(-1)

for ax in axes[n:]:
    ax.axis("off")

for i, path in enumerate(existing):
    img = mpimg.imread(path)
    axes[i].imshow(img)
    axes[i].set_title(os.path.basename(path))
    axes[i].axis("off")

plt.tight_layout()
png_path = os.path.join(OUTPUT_DIR, "appendix_figure_overview.png")
pdf_path = os.path.join(OUTPUT_DIR, "appendix_figure_overview.pdf")
plt.savefig(png_path, dpi=300, bbox_inches="tight")
plt.savefig(pdf_path, bbox_inches="tight")
plt.show()

print(png_path)
print(pdf_path)

 CLIP
	1.	Prompt ensemble
	2.	Image preprocess ablation
	3.	Simple post-hoc calibration
  基础配置

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from PIL import Image, ImageFile, ImageOps
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import torch
from transformers import CLIPModel, CLIPProcessor

ImageFile.LOAD_TRUNCATED_IMAGES = True
Image.MAX_IMAGE_PIXELS = None

OUTPUT_DIR = "/content/drive/MyDrive/dsresearch/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

TASK_INFO = {
    "sleeve": {
        "csv": "/content/drive/MyDrive/dsresearch/outputs/deepfashion_sleeve_with_style.csv",
        "classes": ["long sleeve", "short sleeve", "sleeveless"],
    },
    "pattern": {
        "csv": "/content/drive/MyDrive/dsresearch/outputs/deepfashion_pattern_with_style.csv",
        "classes": ["floral", "graphic", "striped", "solid"],
    },
    "material": {
        "csv": "/content/drive/MyDrive/dsresearch/outputs/deepfashion_material_with_style.csv",
        "classes": ["denim", "chiffon", "cotton", "leather", "knit"],
    },
}

新的 prompt ensemble

In [ ]:
PROMPT_ENSEMBLES_V2 = {
    "sleeve": {
        "long sleeve": [
            "apparel with sleeves covering the full arm length",
            "top whose sleeves extend down to the wrist area",
            "upper garment with complete arm coverage",
            "retail clothing item with full-length sleeves",
            "fashion item showing continuous sleeve coverage along the arm",
        ],
        "short sleeve": [
            "apparel with sleeves ending high on the arm",
            "top whose sleeves stop above the elbow",
            "upper garment with partial arm coverage only",
            "retail clothing item with abbreviated sleeves",
            "fashion item showing short sleeve coverage near the shoulder",
        ],
        "sleeveless": [
            "apparel without sleeves attached",
            "top leaving the arms uncovered",
            "upper garment with open armholes and no sleeve fabric",
            "retail clothing item with no sleeve coverage",
            "fashion item showing bare shoulder-to-arm outline",
        ],
    },

    "pattern": {
        "floral": [
            "apparel featuring repeated flower-inspired motifs",
            "garment covered with blossom-like decorative shapes",
            "clothing with botanical visual print",
            "retail fashion item with floral surface design",
            "fabric appearance dominated by flower motifs",
        ],
        "graphic": [
            "apparel featuring bold printed visual artwork",
            "garment covered with prominent graphic print elements",
            "clothing with symbol-like or illustration-like surface print",
            "retail fashion item with strong graphic design",
            "fabric appearance dominated by printed artwork",
        ],
        "striped": [
            "apparel featuring repeated parallel stripe bands",
            "garment covered with linear repeated bands",
            "clothing with clear stripe-like visual structure",
            "retail fashion item with repeated line pattern",
            "fabric appearance dominated by parallel band repetition",
        ],
        "solid": [
            "apparel with visually plain uninterrupted surface",
            "garment without repeated decorative motif",
            "clothing with uniform fill and no obvious pattern",
            "retail fashion item with plain surface appearance",
            "fabric appearance dominated by single uninterrupted visual field",
        ],
    },

    "material": {
        "denim": [
            "apparel made from rugged twill-like jean fabric",
            "garment with structured denim-style surface texture",
            "clothing constructed from heavy indigo-like twill cloth",
            "retail fashion item with jean-like material appearance",
            "fabric appearance consistent with sturdy denim textile",
        ],
        "chiffon": [
            "apparel made from light sheer draping fabric",
            "garment with airy translucent flowing textile",
            "clothing constructed from delicate semi-transparent cloth",
            "retail fashion item with chiffon-like material appearance",
            "fabric appearance consistent with lightweight sheer drape",
        ],
        "cotton": [
            "apparel made from soft matte natural-fiber cloth",
            "garment with breathable everyday fabric appearance",
            "clothing constructed from matte soft cotton-like textile",
            "retail fashion item with natural casual fabric appearance",
            "fabric appearance consistent with soft cotton surface",
        ],
        "leather": [
            "apparel made from dense smooth hide-like material",
            "garment with firm polished leather-like surface",
            "clothing constructed from glossy heavy hide-textured material",
            "retail fashion item with leather surface appearance",
            "fabric appearance consistent with dense smooth leather",
        ],
        "knit": [
            "apparel made from textured looped yarn construction",
            "garment with sweater-like interlocking knit texture",
            "clothing constructed from visible knitted textile structure",
            "retail fashion item with yarn-loop surface appearance",
            "fabric appearance consistent with textured knit material",
        ],
    },
}

图像预处理函数三种设置：
	•	original
	•	center_crop_square
	•	pad_to_square

In [ ]:
def preprocess_image(image, mode="original"):
    if mode == "original":
        return image

    if mode == "center_crop_square":
        w, h = image.size
        side = min(w, h)
        left = (w - side) // 2
        top = (h - side) // 2
        return image.crop((left, top, left + side, top + side))

    if mode == "pad_to_square":
        w, h = image.size
        side = max(w, h)
        new_img = Image.new("RGB", (side, side), (255, 255, 255))
        left = (side - w) // 2
        top = (side - h) // 2
        new_img.paste(image, (left, top))
        return new_img

    raise ValueError(f"Unknown preprocess mode: {mode}")

CLIP + Prompt Ensemble + Preprocess 的通用评估函数

In [ ]:
def run_clip_improvement_experiment(task_name, preprocess_mode="original"):
    info = TASK_INFO[task_name]
    csv_path = info["csv"]
    class_names = info["classes"]
    prompt_ensemble = PROMPT_ENSEMBLES_V2[task_name]

    df_test = pd.read_csv(csv_path).reset_index(drop=True)
    print(f"\n===== CLIP Improvement Experiment | {task_name} | {preprocess_mode} =====")
    print("Rows:", len(df_test))

    model_id = "openai/clip-vit-base-patch32"
    model = CLIPModel.from_pretrained(model_id).to(device)
    processor = CLIPProcessor.from_pretrained(model_id)
    model.eval()

    class_text_features = []
    for cls in class_names:
        prompts = prompt_ensemble[cls]

        text_inputs = processor(
            text=prompts,
            return_tensors="pt",
            padding=True,
            truncation=True
        )
        text_inputs = {k: v.to(device) for k, v in text_inputs.items()}

        with torch.no_grad():
            text_outputs = model.text_model(
                input_ids=text_inputs["input_ids"],
                attention_mask=text_inputs["attention_mask"]
            )
            text_features = text_outputs.pooler_output
            text_features = model.text_projection(text_features)
            text_features = text_features / text_features.norm(p=2, dim=-1, keepdim=True)

        mean_feature = text_features.mean(dim=0, keepdim=True)
        mean_feature = mean_feature / mean_feature.norm(p=2, dim=-1, keepdim=True)
        class_text_features.append(mean_feature)

    text_feature_matrix = torch.cat(class_text_features, dim=0)

    y_true, y_pred, rows = [], [], []

    for _, row in tqdm(df_test.iterrows(), total=len(df_test), desc=f"{task_name}-{preprocess_mode}"):
        try:
            image = Image.open(row["image_path"]).convert("RGB")
            image = preprocess_image(image, mode=preprocess_mode)

            image_inputs = processor(images=image, return_tensors="pt")
            image_inputs = {k: v.to(device) for k, v in image_inputs.items()}

            with torch.no_grad():
                image_outputs = model.vision_model(pixel_values=image_inputs["pixel_values"])
                image_features = image_outputs.pooler_output
                image_features = model.visual_projection(image_features)
                image_features = image_features / image_features.norm(p=2, dim=-1, keepdim=True)

                logits = image_features @ text_feature_matrix.T
                pred_idx = int(logits.argmax(dim=1).cpu().item())

            pred = class_names[pred_idx]
            gt = row["label"]

            y_true.append(gt)
            y_pred.append(pred)

            rows.append({
                "image_path": row["image_path"],
                "style_split": row["style_split"],
                "ground_truth": gt,
                "prediction": pred,
                "correct": int(gt == pred),
                "model": "CLIP_IMPROVED",
                "task": task_name,
                "preprocess_mode": preprocess_mode,
            })

        except Exception as e:
            print("Skipping due to error:", row["image_path"], e)

    pred_df = pd.DataFrame(rows)
    pred_csv = os.path.join(OUTPUT_DIR, f"{task_name}_clip_improved_{preprocess_mode}_predictions.csv")
    pred_df.to_csv(pred_csv, index=False)

    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    cm = confusion_matrix(y_true, y_pred, labels=class_names)

    summary_rows = [{
        "task": task_name,
        "model": "CLIP_IMPROVED",
        "preprocess_mode": preprocess_mode,
        "subset": "overall",
        "n": len(pred_df),
        "accuracy": acc,
        "macro_f1": macro_f1,
    }]

    for split_name in ["catalog_like", "less_curated"]:
        sub = pred_df[pred_df["style_split"] == split_name]
        summary_rows.append({
            "task": task_name,
            "model": "CLIP_IMPROVED",
            "preprocess_mode": preprocess_mode,
            "subset": split_name,
            "n": len(sub),
            "accuracy": accuracy_score(sub["ground_truth"], sub["prediction"]),
            "macro_f1": f1_score(sub["ground_truth"], sub["prediction"], average="macro"),
        })

    summary_df = pd.DataFrame(summary_rows)
    summary_csv = os.path.join(OUTPUT_DIR, f"{task_name}_clip_improved_{preprocess_mode}_summary.csv")
    summary_df.to_csv(summary_csv, index=False)

    print(summary_df)
    return pred_df, summary_df

运行 preprocess ablation
原图 包含背景、模特、姿态、构图
裁剪中心 去掉背景
保留原图 + 填充是否受resize distortion影响，是否依赖比例信息
 hypothesis

Visionlanguage models rely on contextual and layout cues rather than true attribute understanding.
We perform a preprocessing ablation to test whether performance differences stem from background bias rather than true visual understanding.

In [ ]:
TASK_INFO = {
    "sleeve": {
        "csv": "/content/drive/MyDrive/dsresearch/outputs/deepfashion_sleeve_with_style.csv",
        "classes": ['long sleeve', 'short sleeve', 'sleeveless']
    },
    "pattern": {
        "csv": "/content/drive/MyDrive/dsresearch/outputs/deepfashion_pattern_with_style.csv",
        "classes": ['floral', 'graphic', 'solid', 'striped']
    },
    "material": {
        "csv": "/content/drive/MyDrive/dsresearch/outputs/deepfashion_material_with_style.csv",
        "classes": ['chiffon', 'cotton', 'denim', 'knit', 'leather']
    }
}

In [ ]:
sleeve_original_pred, sleeve_original_summary = run_clip_improvement_experiment("sleeve", "original")
sleeve_crop_pred, sleeve_crop_summary = run_clip_improvement_experiment("sleeve", "center_crop_square")
sleeve_pad_pred, sleeve_pad_summary = run_clip_improvement_experiment("sleeve", "pad_to_square")

pattern_original_pred, pattern_original_summary = run_clip_improvement_experiment("pattern", "original")
pattern_crop_pred, pattern_crop_summary = run_clip_improvement_experiment("pattern", "center_crop_square")
pattern_pad_pred, pattern_pad_summary = run_clip_improvement_experiment("pattern", "pad_to_square")

In [ ]:

OUTPUT_DIR = "/content/drive/MyDrive/dsresearch/outputs"

TASK_INFO = {
    "sleeve": {
        "csv": f"{OUTPUT_DIR}/deepfashion_sleeve_with_style.csv",
        "classes": ["long sleeve", "short sleeve", "sleeveless"]
    },
    "pattern": {
        "csv": f"{OUTPUT_DIR}/deepfashion_pattern_with_style.csv",
        "classes": ["floral", "graphic", "solid", "striped"]
    },
    "material": {
        "csv": f"{OUTPUT_DIR}/deepfashion_material_with_style.csv",
        "classes": ["chiffon", "cotton", "denim", "knit", "leather"]
    }
}

简易calibration，做post-hoc bias correction
用已有预测结果估计每个类被过度or不足预测的程度
对logits加class bias

In [ ]:
def compute_log_prior_adjustment(pred_df, class_names):
    true_dist = pred_df["ground_truth"].value_counts(normalize=True).reindex(class_names).fillna(1e-8)
    pred_dist = pred_df["prediction"].value_counts(normalize=True).reindex(class_names).fillna(1e-8)

    adjustment = np.log(true_dist.values + 1e-8) - np.log(pred_dist.values + 1e-8)
    return adjustment

更完整的 calibrated inference 函数

In [ ]:
def run_clip_improvement_with_calibration(task_name, preprocess_mode="original"):
    info = TASK_INFO[task_name]
    csv_path = info["csv"]
    class_names = info["classes"]
    prompt_ensemble = PROMPT_ENSEMBLES_V2[task_name]

    df_test = pd.read_csv(csv_path).reset_index(drop=True)

    model_id = "openai/clip-vit-base-patch32"
    model = CLIPModel.from_pretrained(model_id).to(device)
    processor = CLIPProcessor.from_pretrained(model_id)
    model.eval()

    class_text_features = []
    for cls in class_names:
        prompts = prompt_ensemble[cls]
        text_inputs = processor(text=prompts, return_tensors="pt", padding=True, truncation=True)
        text_inputs = {k: v.to(device) for k, v in text_inputs.items()}

        with torch.no_grad():
            text_outputs = model.text_model(
                input_ids=text_inputs["input_ids"],
                attention_mask=text_inputs["attention_mask"]
            )
            text_features = text_outputs.pooler_output
            text_features = model.text_projection(text_features)
            text_features = text_features / text_features.norm(p=2, dim=-1, keepdim=True)

        mean_feature = text_features.mean(dim=0, keepdim=True)
        mean_feature = mean_feature / mean_feature.norm(p=2, dim=-1, keepdim=True)
        class_text_features.append(mean_feature)

    text_feature_matrix = torch.cat(class_text_features, dim=0)

    raw_rows = []
    for _, row in tqdm(df_test.iterrows(), total=len(df_test), desc=f"{task_name}-raw-pass"):
        try:
            image = Image.open(row["image_path"]).convert("RGB")
            image = preprocess_image(image, mode=preprocess_mode)

            image_inputs = processor(images=image, return_tensors="pt")
            image_inputs = {k: v.to(device) for k, v in image_inputs.items()}

            with torch.no_grad():
                image_outputs = model.vision_model(pixel_values=image_inputs["pixel_values"])
                image_features = image_outputs.pooler_output
                image_features = model.visual_projection(image_features)
                image_features = image_features / image_features.norm(p=2, dim=-1, keepdim=True)
                logits = (image_features @ text_feature_matrix.T).cpu().numpy().flatten()

            pred_idx = int(np.argmax(logits))
            raw_rows.append({
                "image_path": row["image_path"],
                "style_split": row["style_split"],
                "ground_truth": row["label"],
                "raw_prediction": class_names[pred_idx],
                **{f"logit_{i}": logits[i] for i in range(len(class_names))}
            })
        except Exception as e:
            print("Skipping due to error:", row["image_path"], e)

    raw_df = pd.DataFrame(raw_rows)
    adjustment = compute_log_prior_adjustment(
        raw_df.rename(columns={"raw_prediction": "prediction"})[["ground_truth", "prediction"]],
        class_names
    )

    final_rows = []
    y_true, y_pred = [], []

    for _, row in raw_df.iterrows():
        logits = np.array([row[f"logit_{i}"] for i in range(len(class_names))], dtype=float)
        cal_logits = logits + adjustment
        pred_idx = int(np.argmax(cal_logits))
        pred = class_names[pred_idx]
        gt = row["ground_truth"]

        y_true.append(gt)
        y_pred.append(pred)

        final_rows.append({
            "image_path": row["image_path"],
            "style_split": row["style_split"],
            "ground_truth": gt,
            "prediction": pred,
            "correct": int(gt == pred),
            "model": "CLIP_IMPROVED_CALIBRATED",
            "task": task_name,
            "preprocess_mode": preprocess_mode,
        })

    pred_df = pd.DataFrame(final_rows)
    pred_csv = os.path.join(OUTPUT_DIR, f"{task_name}_clip_improved_calibrated_{preprocess_mode}_predictions.csv")
    pred_df.to_csv(pred_csv, index=False)

    rows_out = [{
        "task": task_name,
        "model": "CLIP_IMPROVED_CALIBRATED",
        "preprocess_mode": preprocess_mode,
        "subset": "overall",
        "n": len(pred_df),
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
    }]
    for split_name in ["catalog_like", "less_curated"]:
        sub = pred_df[pred_df["style_split"] == split_name]
        rows_out.append({
            "task": task_name,
            "model": "CLIP_IMPROVED_CALIBRATED",
            "preprocess_mode": preprocess_mode,
            "subset": split_name,
            "n": len(sub),
            "accuracy": accuracy_score(sub["ground_truth"], sub["prediction"]),
            "macro_f1": f1_score(sub["ground_truth"], sub["prediction"], average="macro"),
        })

    summary_df = pd.DataFrame(rows_out)
    summary_csv = os.path.join(OUTPUT_DIR, f"{task_name}_clip_improved_calibrated_{preprocess_mode}_summary.csv")
    summary_df.to_csv(summary_csv, index=False)

    print("Adjustment:", dict(zip(class_names, adjustment)))
    print(summary_df)
    return pred_df, summary_df

In [ ]:

sleeve_calib_pred, sleeve_calib_summary = run_clip_improvement_with_calibration(
    "sleeve",
    preprocess_mode="original"
)

pattern_calib_pred, pattern_calib_summary = run_clip_improvement_with_calibration(
    "pattern",
    preprocess_mode="original"
)

material_calib_pred, material_calib_summary = run_clip_improvement_with_calibration(
    "material",
    preprocess_mode="original"
)

In [ ]:

def compare_before_after(task):
    if task == "sleeve":
        base_path = f"{OUTPUT_DIR}/deepfashion_sleeve_clip_predictions_exp1_full.csv"
    elif task == "pattern":
        base_path = f"{OUTPUT_DIR}/pattern_clip_predictions_full.csv"
    elif task == "material":
        base_path = f"{OUTPUT_DIR}/material_clip_predictions_full.csv"

    calib_path = f"{OUTPUT_DIR}/{task}_clip_improved_calibrated_original_predictions.csv"

    base = pd.read_csv(base_path)
    calib = pd.read_csv(calib_path)

    acc_base = (base["ground_truth"] == base["prediction"]).mean()
    acc_calib = (calib["ground_truth"] == calib["prediction"]).mean()

    print("\n====", task, "====")
    print("Before (CLIP):", round(acc_base, 4))
    print("After  (Calib):", round(acc_calib, 4))


for t in ["sleeve", "pattern", "material"]:
    compare_before_after(t)

CLIP attention-based visualization
We use attention rollout over the vision transformer layers as a qualitative visualization of model focus.


In [ ]:
import os
import math
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn.functional as F
from transformers import CLIPProcessor, CLIPModel

device = "cuda" if torch.cuda.is_available() else "cpu"

model_id = "openai/clip-vit-base-patch32"

clip_model = CLIPModel.from_pretrained(
    model_id,
    attn_implementation="eager"
).to(device)

clip_model.config.output_attentions = True
clip_model.vision_model.config.output_attentions = True

clip_processor = CLIPProcessor.from_pretrained(model_id)


def get_clip_text_feature(prompt: str):
    text_inputs = clip_processor(
        text=[prompt],
        return_tensors="pt",
        padding=True,
        truncation=True
    )
    text_inputs = {k: v.to(device) for k, v in text_inputs.items()}

    with torch.no_grad():
        text_outputs = clip_model.text_model(
            input_ids=text_inputs["input_ids"],
            attention_mask=text_inputs["attention_mask"]
        )
        text_feature = text_outputs.pooler_output
        text_feature = clip_model.text_projection(text_feature)
        text_feature = F.normalize(text_feature, p=2, dim=-1)

    return text_feature



def get_clip_attention_rollout(image_path, prompt):
    image = Image.open(image_path).convert("RGB")
    inputs = clip_processor(images=image, return_tensors="pt")
    pixel_values = inputs["pixel_values"].to(device)

    text_feature = get_clip_text_feature(prompt)

    with torch.no_grad():
        vision_outputs = clip_model.vision_model(
            pixel_values=pixel_values,
            output_attentions=True,
            return_dict=True
        )

        image_feature = vision_outputs.pooler_output
        image_feature = clip_model.visual_projection(image_feature)
        image_feature = F.normalize(image_feature, p=2, dim=-1)

        similarity = float((image_feature @ text_feature.T).item())

    attentions = vision_outputs.attentions

    if attentions is None:
        print("⚠️ Attention not available, returning blank map")
        h, w = image.size[1], image.size[0]
        return image, np.zeros((h, w)), similarity

    rollout = None

    for attn in attentions:
        attn_mean = attn.mean(dim=1)[0]
        seq_len = attn_mean.shape[0]

        identity = torch.eye(seq_len, device=attn_mean.device)
        attn_aug = attn_mean + identity
        attn_aug = attn_aug / attn_aug.sum(dim=-1, keepdim=True)

        rollout = attn_aug if rollout is None else attn_aug @ rollout

    cls_attn = rollout[0, 1:]
    num_patches = cls_attn.shape[0]
    grid_size = int(math.sqrt(num_patches))

    attn_map = cls_attn.reshape(grid_size, grid_size).cpu().numpy()

    attn_map = attn_map - attn_map.min()
    if attn_map.max() > 0:
        attn_map = attn_map / attn_map.max()

    import cv2
    h, w = image.size[1], image.size[0]
    attn_map_resized = cv2.resize(attn_map, (w, h))

    return image, attn_map_resized, similarity


def show_clip_attention(image_path, prompt, save_prefix=None):
    image, attn_map, sim = get_clip_attention_rollout(image_path, prompt)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    axes[0].imshow(image)
    axes[0].set_title("Original")
    axes[0].axis("off")

    axes[1].imshow(attn_map, cmap="jet")
    axes[1].set_title("Attention Rollout")
    axes[1].axis("off")

    axes[2].imshow(image)
    axes[2].imshow(attn_map, cmap="jet", alpha=0.45)
    axes[2].set_title(f"Overlay\nsim={sim:.3f}")
    axes[2].axis("off")

    plt.suptitle(prompt)
    plt.tight_layout()

    if save_prefix is not None:
        plt.savefig(f"{save_prefix}.png", dpi=300, bbox_inches="tight")
        plt.savefig(f"{save_prefix}.pdf", bbox_inches="tight")

    plt.show()

把图像分块遮住，看目标 prompt 的相似度掉多少，如果掉得越多，这个区域对该预测越重要

  To complement attention-based visualization, we also perform a simple occlusion sensitivity analysis. Regions whose masking causes the largest drop in text-image similarity are treated as visually important for the target prediction.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
import torch
import torch.nn.functional as F

def get_clip_similarity(image_pil, prompt):
    image_inputs = clip_processor(images=image_pil, return_tensors="pt")
    image_inputs = {k: v.to(device) for k, v in image_inputs.items()}

    text_feature = get_clip_text_feature(prompt)

    with torch.no_grad():
        vision_outputs = clip_model.vision_model(pixel_values=image_inputs["pixel_values"])
        image_feature = vision_outputs.pooler_output
        image_feature = clip_model.visual_projection(image_feature)
        image_feature = F.normalize(image_feature, p=2, dim=-1)

        sim = float((image_feature @ text_feature.T).item())

    return sim

def compute_occlusion_map(image_path, prompt, grid_size=8, occlusion_color=(255, 255, 255)):
    image = Image.open(image_path).convert("RGB")
    w, h = image.size

    base_score = get_clip_similarity(image, prompt)

    occ_map = np.zeros((grid_size, grid_size), dtype=np.float32)

    patch_w = w / grid_size
    patch_h = h / grid_size

    for i in range(grid_size):
        for j in range(grid_size):
            occluded = image.copy()
            draw = ImageDraw.Draw(occluded)

            left = int(j * patch_w)
            top = int(i * patch_h)
            right = int((j + 1) * patch_w)
            bottom = int((i + 1) * patch_h)

            draw.rectangle([left, top, right, bottom], fill=occlusion_color)

            score = get_clip_similarity(occluded, prompt)
            occ_map[i, j] = base_score - score  # drop in score

    occ_map = occ_map - occ_map.min()
    if occ_map.max() > 0:
        occ_map = occ_map / occ_map.max()

    return image, occ_map, base_score

def show_occlusion_map(image_path, prompt, grid_size=8, save_prefix=None):
    image, occ_map, base_score = compute_occlusion_map(image_path, prompt, grid_size=grid_size)

    image_np = np.array(image)
    h, w = image_np.shape[:2]

    import cv2
    occ_map_resized = cv2.resize(occ_map, (w, h), interpolation=cv2.INTER_CUBIC)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    axes[0].imshow(image)
    axes[0].set_title("Original")
    axes[0].axis("off")

    axes[1].imshow(occ_map, cmap="hot")
    axes[1].set_title("Occlusion Grid Map")
    axes[1].axis("off")

    axes[2].imshow(image)
    axes[2].imshow(occ_map_resized, cmap="hot", alpha=0.45)
    axes[2].set_title(f"Overlay\nbase sim={base_score:.3f}")
    axes[2].axis("off")

    plt.suptitle(prompt)
    plt.tight_layout()

    if save_prefix is not None:
        png_path = f"{save_prefix}.png"
        pdf_path = f"{save_prefix}.pdf"
        plt.savefig(png_path, dpi=300, bbox_inches="tight")
        plt.savefig(pdf_path, bbox_inches="tight")

    plt.show()

选三种样本

每个任务先挑：
	1.	catalog_like 且预测正确
	2.	less_curated 且预测错误
	3.	CLIP 错 BLIP 对

分别做：
	attention rollout
	occlusion map


In [ ]:
import pandas as pd
import random

def select_qualitative_samples(task):
    clip_path = f"/content/drive/MyDrive/dsresearch/outputs/deepfashion_{task}_clip_predictions_exp1_full.csv"
    blip_path = f"/content/drive/MyDrive/dsresearch/outputs/deepfashion_{task}_blip_predictions_full.csv"

    clip_df = pd.read_csv(clip_path)
    blip_df = pd.read_csv(blip_path)

  
    df = clip_df.merge(
        blip_df[["image_path", "prediction"]],
        on="image_path",
        suffixes=("_clip", "_blip")
    )

   
    case1 = df[
        (df["style_split"] == "catalog_like") &
        (df["ground_truth"] == df["prediction_clip"])
    ]
    case1_sample = case1.sample(1).iloc[0]

    case2 = df[
        (df["style_split"] == "less_curated") &
        (df["ground_truth"] != df["prediction_clip"])
    ]
    case2_sample = case2.sample(1).iloc[0]


    case3 = df[
        (df["prediction_clip"] != df["ground_truth"]) &
        (df["prediction_blip"] == df["ground_truth"])
    ]
    case3_sample = case3.sample(1).iloc[0]

    return {
        "catalog_correct": case1_sample,
        "less_curated_wrong": case2_sample,
        "clip_wrong_blip_right": case3_sample
    }

In [ ]:
def select_qualitative_samples(task):
    BASE = "/content/drive/MyDrive/dsresearch/outputs/"

    FILE_MAP = {
        "sleeve": {
            "clip": BASE + "deepfashion_sleeve_clip_predictions_exp1_full.csv",
            "blip": BASE + "deepfashion_blip_predictions_full.csv"
        },
        "pattern": {
            "clip": BASE + "pattern_clip_predictions_full.csv",
            "blip": BASE + "pattern_blip_predictions_full.csv"
        },
        "material": {
            "clip": BASE + "material_clip_predictions_full.csv",
            "blip": BASE + "material_blip_predictions_full.csv"
        }
    }

    clip_df = pd.read_csv(FILE_MAP[task]["clip"])
    blip_df = pd.read_csv(FILE_MAP[task]["blip"])

    df = clip_df.merge(
        blip_df[["image_path", "prediction"]],
        on="image_path",
        suffixes=("_clip", "_blip")
    )

    case1 = df[
        (df["style_split"] == "catalog_like") &
        (df["ground_truth"] == df["prediction_clip"])
    ].sample(1).iloc[0]

    case2 = df[
        (df["style_split"] == "less_curated") &
        (df["ground_truth"] != df["prediction_clip"])
    ].sample(1).iloc[0]

    case3 = df[
        (df["prediction_clip"] != df["ground_truth"]) &
        (df["prediction_blip"] == df["ground_truth"])
    ].sample(1).iloc[0]

    return {
        "catalog_correct": case1,
        "less_curated_wrong": case2,
        "clip_wrong_blip_right": case3
    }

In [ ]:
samples = select_qualitative_samples("sleeve")

for k, v in samples.items():
    print(k)
    print(v["image_path"], v["ground_truth"])

sleeve

In [ ]:
cases = [
    {
        "name": "catalog_correct",
        "image": "/content/deepfashion_data/img_highres/Bonjour_Au_Revoir_Tee/img_00000031.jpg",
        "label": "long sleeve"
    },
    {
        "name": "less_curated_wrong",
        "image": "/content/deepfashion_data/img_highres/Slub_Knit_Sweatshorts/img_00000015.jpg",
        "label": "sleeveless"
    },
    {
        "name": "clip_wrong_blip_right",
        "image": "/content/deepfashion_data/img_highres/Hooded_Cotton_Canvas_Anorak/img_00000058.jpg",
        "label": "long sleeve"
    }
]

SLEEVE_PROMPT_MAP = {
    "long sleeve": "a long sleeve top",
    "short sleeve": "a short sleeve top",
    "sleeveless": "a sleeveless top"
}

for case in cases:
    prompt = SLEEVE_PROMPT_MAP[case["label"]]

    save_prefix = os.path.join(
        OUTPUT_DIR,
        f"qual_sleeve_{case['name']}"
    )

    print("Running:", case["name"])

    show_clip_attention(
        image_path=case["image"],
        prompt=prompt,
        save_prefix=save_prefix + "_attention"
    )

    show_occlusion_map(
        image_path=case["image"],
        prompt=prompt,
        grid_size=8,
        save_prefix=save_prefix + "_occlusion"
    )

In [ ]:
def normalize_columns(df):
    if "label" in df.columns:
        df = df.rename(columns={"label": "ground_truth"})
    return df

In [ ]:
def select_best_cases(task):
    BASE = "/content/drive/MyDrive/dsresearch/outputs/"

    FILE_MAP = {
        "sleeve": {
            "clip": BASE + "deepfashion_sleeve_clip_predictions_exp1_full.csv",
            "blip": BASE + "deepfashion_blip_predictions_full.csv"
        },
        "pattern": {
            "clip": BASE + "pattern_clip_predictions_full.csv",
            "blip": BASE + "pattern_blip_predictions_full.csv"
        },
        "material": {
            "clip": BASE + "material_clip_predictions_full.csv",
            "blip": BASE + "material_blip_predictions_full.csv"
        }
    }

    clip_df = normalize_columns(pd.read_csv(FILE_MAP[task]["clip"]))
    blip_df = normalize_columns(pd.read_csv(FILE_MAP[task]["blip"]))

    df = clip_df.merge(
        blip_df[["image_path", "prediction"]],
        on="image_path",
        suffixes=("_clip", "_blip")
    )


    case1 = df[
        (df["style_split"] == "catalog_like") &
        (df["ground_truth"] == df["prediction_clip"])
    ].iloc[0]


    wrong = df[
        (df["style_split"] == "less_curated") &
        (df["ground_truth"] != df["prediction_clip"])
    ].copy()

    logit_cols = [c for c in wrong.columns if "logit" in c]

    if len(logit_cols) > 0:
        wrong["confidence"] = wrong[logit_cols].max(axis=1)
        case2 = wrong.sort_values("confidence", ascending=False).iloc[0]
    else:
        case2 = wrong.iloc[0]


    case3 = df[
        (df["prediction_clip"] != df["ground_truth"]) &
        (df["prediction_blip"] == df["ground_truth"])
    ].iloc[0]

    return {
        "catalog_correct": case1,
        "less_curated_wrong": case2,
        "clip_wrong_blip_right": case3
    }

In [ ]:
TASK_PROMPT_MAP = {
    "sleeve": {
        "long sleeve": "a long sleeve top",
        "short sleeve": "a short sleeve top",
        "sleeveless": "a sleeveless top"
    },
    "pattern": {
        "floral": "clothing with floral patterns",
        "graphic": "clothing with graphic prints",
        "striped": "clothing with striped patterns",
        "solid": "plain clothing with no visible pattern"
    },
    "material": {
        "chiffon": "lightweight sheer chiffon fabric",
        "cotton": "cotton fabric clothing",
        "denim": "denim fabric clothing",
        "knit": "knitted fabric clothing",
        "leather": "leather material clothing"
    }
}

for task in ["sleeve", "pattern", "material"]:

    print("\n====================")
    print("TASK:", task)
    print("====================")

    samples = select_best_cases(task)

    for name, row in samples.items():

        label = row["ground_truth"]
        prompt = TASK_PROMPT_MAP[task][label]

        save_prefix = os.path.join(
            OUTPUT_DIR,
            f"qual_{task}_{name}"
        )

        print(f"\nRunning: {task} | {name}")
        print("image:", row["image_path"])
        print("label:", label)

        show_clip_attention(
            image_path=row["image_path"],
            prompt=prompt,
            save_prefix=save_prefix + "_attention"
        )

        show_occlusion_map(
            image_path=row["image_path"],
            prompt=prompt,
            grid_size=8,
            save_prefix=save_prefix + "_occlusion"
        )


catalog_like 且预测正确
less_curated 且预测错误
CLIP 错、BLIP 对

每类随机抽若干张，生成原图，attention rollout，attention overlay，occlusion overlay


In [ ]:
import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFile

import torch
import torch.nn.functional as F
from transformers import CLIPProcessor, CLIPModel

ImageFile.LOAD_TRUNCATED_IMAGES = True
Image.MAX_IMAGE_PIXELS = None

OUTPUT_DIR = "/content/drive/MyDrive/dsresearch/outputs"
QUAL_DIR = os.path.join(OUTPUT_DIR, "qualitative_visual_grounding")
os.makedirs(QUAL_DIR, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

FILE_MAP = {
    ("CLIP", "sleeve"): "/content/drive/MyDrive/dsresearch/outputs/deepfashion_sleeve_clip_predictions_exp1_full.csv",
    ("CLIP", "pattern"): "/content/drive/MyDrive/dsresearch/outputs/pattern_clip_predictions_full.csv",
    ("CLIP", "material"): "/content/drive/MyDrive/dsresearch/outputs/material_clip_predictions_full.csv",

    ("BLIP", "sleeve"): "/content/drive/MyDrive/dsresearch/outputs/deepfashion_blip_predictions_full.csv",
    ("BLIP", "pattern"): "/content/drive/MyDrive/dsresearch/outputs/pattern_blip_predictions_full.csv",
    ("BLIP", "material"): "/content/drive/MyDrive/dsresearch/outputs/material_blip_predictions_full.csv",
}

TASK_CLASSES = {
    "sleeve": ["long sleeve", "short sleeve", "sleeveless"],
    "pattern": ["floral", "graphic", "striped", "solid"],
    "material": ["denim", "chiffon", "cotton", "leather", "knit"],
}

TASK_PROMPTS = {
    "sleeve": {
        "long sleeve": "upper-body clothing whose sleeves extend to the wrists",
        "short sleeve": "upper-body clothing whose sleeves stop above the elbow",
        "sleeveless": "upper-body clothing with the arms fully uncovered",
    },
    "pattern": {
        "floral": "clothing surface covered with flower-like motifs",
        "graphic": "clothing surface dominated by printed visual artwork or symbols",
        "striped": "clothing surface organized by repeated parallel bands",
        "solid": "clothing surface with no distinct repeated motif",
    },
    "material": {
        "denim": "apparel made from heavy blue twill-like cloth",
        "chiffon": "apparel made from sheer airy lightweight fabric",
        "cotton": "apparel made from soft matte natural fiber fabric",
        "leather": "apparel made from smooth dense hide-like material",
        "knit": "apparel made from looped textured yarn structure",
    },
}

print("QUAL_DIR:", QUAL_DIR)

In [ ]:
model_id = "openai/clip-vit-base-patch32"
clip_model = CLIPModel.from_pretrained(
    model_id,
    attn_implementation="eager"
).to(device)
clip_processor = CLIPProcessor.from_pretrained(model_id)
clip_model.eval()

print("CLIP loaded.")

文本特征函数

In [ ]:
def get_clip_text_feature(prompt: str):
    text_inputs = clip_processor(
        text=[prompt],
        return_tensors="pt",
        padding=True,
        truncation=True
    )
    text_inputs = {k: v.to(device) for k, v in text_inputs.items()}

    with torch.no_grad():
        text_outputs = clip_model.text_model(
            input_ids=text_inputs["input_ids"],
            attention_mask=text_inputs["attention_mask"]
        )
        text_feature = text_outputs.pooler_output
        text_feature = clip_model.text_projection(text_feature)
        text_feature = F.normalize(text_feature, p=2, dim=-1)

    return text_feature

Attention rollout

In [ ]:
def get_clip_attention_rollout(image_path, prompt):
    image = Image.open(image_path).convert("RGB")
    inputs = clip_processor(images=image, return_tensors="pt")
    pixel_values = inputs["pixel_values"].to(device)

    text_feature = get_clip_text_feature(prompt)

    with torch.no_grad():
        vision_outputs = clip_model.vision_model(
            pixel_values=pixel_values,
            output_attentions=True
        )

        image_feature = vision_outputs.pooler_output
        image_feature = clip_model.visual_projection(image_feature)
        image_feature = F.normalize(image_feature, p=2, dim=-1)
        similarity = float((image_feature @ text_feature.T).item())

    attentions = vision_outputs.attentions
    rollout = None

    for attn in attentions:
        attn_mean = attn.mean(dim=1)[0]  # [seq, seq]
        seq_len = attn_mean.shape[0]

        attn_aug = attn_mean + torch.eye(seq_len, device=attn_mean.device)
        attn_aug = attn_aug / attn_aug.sum(dim=-1, keepdim=True)

        rollout = attn_aug if rollout is None else attn_aug @ rollout

    cls_attn = rollout[0, 1:]
    num_patches = cls_attn.shape[0]
    grid_size = int(math.sqrt(num_patches))

    attn_map = cls_attn.reshape(grid_size, grid_size).detach().cpu().numpy()
    attn_map = attn_map - attn_map.min()
    if attn_map.max() > 0:
        attn_map = attn_map / attn_map.max()

    image_np = np.array(image)
    h, w = image_np.shape[:2]

    import cv2
    attn_map_resized = cv2.resize(attn_map, (w, h))

    return image, attn_map_resized, similarity

Occlusion sensitivity

In [ ]:
def get_clip_similarity(image_pil, prompt):
    image_inputs = clip_processor(images=image_pil, return_tensors="pt")
    image_inputs = {k: v.to(device) for k, v in image_inputs.items()}

    text_feature = get_clip_text_feature(prompt)

    with torch.no_grad():
        vision_outputs = clip_model.vision_model(pixel_values=image_inputs["pixel_values"])
        image_feature = vision_outputs.pooler_output
        image_feature = clip_model.visual_projection(image_feature)
        image_feature = F.normalize(image_feature, p=2, dim=-1)

        sim = float((image_feature @ text_feature.T).item())

    return sim

def compute_occlusion_map(image_path, prompt, grid_size=8, occlusion_color=(255, 255, 255)):
    image = Image.open(image_path).convert("RGB")
    w, h = image.size

    base_score = get_clip_similarity(image, prompt)
    occ_map = np.zeros((grid_size, grid_size), dtype=np.float32)

    patch_w = w / grid_size
    patch_h = h / grid_size

    for i in range(grid_size):
        for j in range(grid_size):
            occluded = image.copy()
            draw = ImageDraw.Draw(occluded)

            left = int(j * patch_w)
            top = int(i * patch_h)
            right = int((j + 1) * patch_w)
            bottom = int((i + 1) * patch_h)

            draw.rectangle([left, top, right, bottom], fill=occlusion_color)

            score = get_clip_similarity(occluded, prompt)
            occ_map[i, j] = base_score - score

    occ_map = occ_map - occ_map.min()
    if occ_map.max() > 0:
        occ_map = occ_map / occ_map.max()

    return image, occ_map, base_score

单张图的可视化保存函数

In [ ]:
def save_attention_and_occlusion_figure(image_path, prompt, save_prefix, title_lines=None, grid_size=8):
    image, attn_map, sim = get_clip_attention_rollout(image_path, prompt)
    image2, occ_map, base_score = compute_occlusion_map(image_path, prompt, grid_size=grid_size)

    image_np = np.array(image)
    h, w = image_np.shape[:2]

    import cv2
    occ_map_resized = cv2.resize(occ_map, (w, h), interpolation=cv2.INTER_CUBIC)

    fig, axes = plt.subplots(1, 4, figsize=(20, 5))

    axes[0].imshow(image)
    axes[0].set_title("Original")
    axes[0].axis("off")

    axes[1].imshow(attn_map, cmap="jet")
    axes[1].set_title("Attention Rollout")
    axes[1].axis("off")

    axes[2].imshow(image)
    axes[2].imshow(attn_map, cmap="jet", alpha=0.45)
    axes[2].set_title(f"Attention Overlay\nsim={sim:.3f}")
    axes[2].axis("off")

    axes[3].imshow(image)
    axes[3].imshow(occ_map_resized, cmap="hot", alpha=0.45)
    axes[3].set_title(f"Occlusion Overlay\nbase sim={base_score:.3f}")
    axes[3].axis("off")

    if title_lines is not None:
        plt.suptitle("\n".join(title_lines), fontsize=11)

    plt.tight_layout()

    png_path = f"{save_prefix}.png"
    pdf_path = f"{save_prefix}.pdf"
    plt.savefig(png_path, dpi=300, bbox_inches="tight")
    plt.savefig(pdf_path, bbox_inches="tight")
    plt.close(fig)

    return png_path, pdf_path

自动合并 CLIP / BLIP 预测并挑样本

In [ ]:
def load_merged_predictions(task):
    clip_df = pd.read_csv(FILE_MAP[("CLIP", task)]).copy()
    blip_df = pd.read_csv(FILE_MAP[("BLIP", task)]).copy()

    clip_df = clip_df.rename(columns={
        "prediction": "clip_pred",
        "correct": "clip_correct"
    })
    blip_df = blip_df.rename(columns={
        "prediction": "blip_pred",
        "correct": "blip_correct"
    })

    merged = clip_df[["image_path", "style_split", "ground_truth", "clip_pred", "clip_correct"]].merge(
        blip_df[["image_path", "style_split", "ground_truth", "blip_pred", "blip_correct"]],
        on=["image_path", "style_split", "ground_truth"],
        how="inner"
    )
    return merged

def select_representative_cases(task, n_each=5, seed=42):
    merged = load_merged_predictions(task)

    case_catalog_correct = merged[
        (merged["style_split"] == "catalog_like") &
        (merged["clip_correct"] == 1)
    ]

    case_less_wrong = merged[
        (merged["style_split"] == "less_curated") &
        (merged["clip_correct"] == 0)
    ]

    case_clip_wrong_blip_right = merged[
        (merged["clip_correct"] == 0) &
        (merged["blip_correct"] == 1)
    ]

    selected = {
        "catalog_like_correct": case_catalog_correct.sample(
            n=min(n_each, len(case_catalog_correct)),
            random_state=seed
        ) if len(case_catalog_correct) > 0 else case_catalog_correct,

        "less_curated_wrong": case_less_wrong.sample(
            n=min(n_each, len(case_less_wrong)),
            random_state=seed
        ) if len(case_less_wrong) > 0 else case_less_wrong,

        "clip_wrong_blip_right": case_clip_wrong_blip_right.sample(
            n=min(n_each, len(case_clip_wrong_blip_right)),
            random_state=seed
        ) if len(case_clip_wrong_blip_right) > 0 else case_clip_wrong_blip_right,
    }

    return selected

*批量生成所有* qualitative 图

In [ ]:
def batch_generate_visual_grounding(task, n_each=5, grid_size=8):
    print(f"\n===== Generating visual grounding figures | {task} =====")
    selected = select_representative_cases(task, n_each=n_each)

    task_dir = os.path.join(QUAL_DIR, task)
    os.makedirs(task_dir, exist_ok=True)

    records = []

    for case_name, df_case in selected.items():
        case_dir = os.path.join(task_dir, case_name)
        os.makedirs(case_dir, exist_ok=True)

        print(f"{case_name}: {len(df_case)} samples")

        for idx, (_, row) in enumerate(df_case.iterrows()):
            gt = row["ground_truth"]
            prompt = TASK_PROMPTS[task][gt]

            save_prefix = os.path.join(case_dir, f"{case_name}_{idx:02d}")

            title_lines = [
                f"Task: {task}",
                f"Case: {case_name}",
                f"GT: {gt}",
                f"CLIP: {row['clip_pred']} ({row['clip_correct']})",
                f"BLIP: {row['blip_pred']} ({row['blip_correct']})",
                f"Split: {row['style_split']}",
                f"Prompt: {prompt}",
            ]

            try:
                png_path, pdf_path = save_attention_and_occlusion_figure(
                    image_path=row["image_path"],
                    prompt=prompt,
                    save_prefix=save_prefix,
                    title_lines=title_lines,
                    grid_size=grid_size
                )

                records.append({
                    "task": task,
                    "case_name": case_name,
                    "image_path": row["image_path"],
                    "ground_truth": gt,
                    "clip_pred": row["clip_pred"],
                    "clip_correct": row["clip_correct"],
                    "blip_pred": row["blip_pred"],
                    "blip_correct": row["blip_correct"],
                    "style_split": row["style_split"],
                    "prompt": prompt,
                    "png_path": png_path,
                    "pdf_path": pdf_path,
                })
            except Exception as e:
                print("Failed on:", row["image_path"], e)

    out_df = pd.DataFrame(records)
    csv_path = os.path.join(task_dir, f"{task}_visual_grounding_index.csv")
    out_df.to_csv(csv_path, index=False)

    print("Saved index:", csv_path)
    return out_df

自动生成 overview 联系图

> Add blockquote



In [ ]:
def select_best_cases(task):
    BASE = "/content/drive/MyDrive/dsresearch/outputs/"

    FILE_MAP = {
        "sleeve": {
            "clip": BASE + "deepfashion_sleeve_clip_predictions_exp1_full.csv",
            "blip": BASE + "deepfashion_blip_predictions_full.csv"
        },
        "pattern": {
            "clip": BASE + "pattern_clip_predictions_full.csv",
            "blip": BASE + "pattern_blip_predictions_full.csv"
        },
        "material": {
            "clip": BASE + "material_clip_predictions_full.csv",
            "blip": BASE + "material_blip_predictions_full.csv"
        }
    }

    clip_df = pd.read_csv(FILE_MAP[task]["clip"])
    blip_df = pd.read_csv(FILE_MAP[task]["blip"])

    df = clip_df.merge(
        blip_df[["image_path", "prediction"]],
        on="image_path",
        suffixes=("_clip", "_blip")
    )

    case1 = df[
        (df["style_split"] == "catalog_like") &
        (df["ground_truth"] == df["prediction_clip"])
    ].sample(2)

    case2 = df[
        (df["style_split"] == "less_curated") &
        (df["ground_truth"] != df["prediction_clip"])
    ].sample(2)

    case3 = df[
        (df["prediction_clip"] != df["ground_truth"]) &
        (df["prediction_blip"] == df["ground_truth"])
    ].sample(2)

    return {
        "catalog_like_correct": case1,
        "less_curated_wrong": case2,
        "clip_wrong_blip_right": case3
    }

生成图 + 自动记录index


In [ ]:
QUAL_DIR = os.path.join(OUTPUT_DIR, "qualitative")
os.makedirs(QUAL_DIR, exist_ok=True)

def generate_visual_grounding(task):

    print(f"\n===== Generating visual grounding figures | {task} =====")

    task_dir = os.path.join(QUAL_DIR, task)
    os.makedirs(task_dir, exist_ok=True)

    samples = select_best_cases(task)

    records = []

    for case_name, df_case in samples.items():

        print(f"{case_name}: {len(df_case)} samples")

        for _, row in df_case.iterrows():

            try:
                label = row["ground_truth"]

                prompt = TASK_PROMPT_MAP[task][label]

                prefix = os.path.join(
                    task_dir,
                    f"{task}_{case_name}_{_}"
                )

                attn_path = prefix + "_attention.png"
                show_clip_attention(
                    row["image_path"],
                    prompt,
                    save_prefix=prefix + "_attention"
                )

                occ_path = prefix + "_occlusion.png"
                show_occlusion_map(
                    row["image_path"],
                    prompt,
                    save_prefix=prefix + "_occlusion"
                )

                records.append({
                    "task": task,
                    "case": case_name,
                    "image_path": row["image_path"],
                    "png_path": attn_path  
                })

            except Exception as e:
                print("Failed on:", row["image_path"], e)

    index_csv = os.path.join(task_dir, f"{task}_visual_grounding_index.csv")
    pd.DataFrame(records).to_csv(index_csv, index=False)

    print("Saved index:", index_csv)